# Step 1: Entry Legitimacy Assessment

Classifies each row from OCR-segmented ASCC catalog input as one of:

- **listing** -- a real catalog entry; proceed to field extraction
- **cross_reference** -- a real catalog row that redirects to another section; no extractable postmark data
- **non_entry** -- annotation, fragment, or page artifact mistakenly included; flag for review

Assessment is purely text-structure-based. Section-context columns (Default Shape, etc.) are ignored at this stage.

### Signals

Evaluated in order:

1. **Relationship indicator prefix** -- starts with Same, (L), (E), or asterisked variants. Immediate listing.
2. **Cross-reference pattern** -- contains "See [Place/Section]" without a semicolon-parenthetical. Cross-reference.
3. **Fragment detection** -- starts with unmatched closing paren or lowercase mid-sentence content. Non-entry.
4. **Trailing valuation** -- ends with a number (e.g. `1500.00`, `1,500`), `--`, or `---`. Nearly all legitimate entries have this.
5. **Core structural anatomy** -- has at least one of: semicolon-parenthetical, four-digit year (1700-1869), decade ref (e.g. `1850's`), or c-prefixed year.

Final classification:
- Signal 1 hit -> listing (auto)
- Signal 2 hit -> cross_reference
- Signal 3 hit -> non_entry
- Signal 4 AND Signal 5 -> listing
- Signal 4 only OR Signal 5 only -> listing (low confidence, flagged for review)
- Neither Signal 4 nor Signal 5 -> non_entry

## 0. Setup

In [ ]:
import pandas as pd
import re

INPUT_CSV = './wip/in/GOOD-WV_ASCC_CTLG.csv'  # <-- Change per state

df = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df)} rows from {INPUT_CSV}')
print(f'Columns: {list(df.columns)}')

Loaded 137 rows from ./wip/in/CatDat/GOOD-DC_ASCC_CTLG.csv
Columns: ['Listing', 'Page', 'Images Above', 'Default Shape']


## 1. Preprocessing

Strip dot leaders before signal detection. These are OCR artifacts from the catalog's
visual formatting (dots connecting the entry text to the price column) and carry no
semantic content.

In [2]:
def strip_dot_leaders(text):
    """Remove runs of 2+ dots (dot leaders) and collapse resulting whitespace."""
    t = re.sub(r'\.{2,}', ' ', str(text))
    return re.sub(r'  +', ' ', t).strip()

df['clean_text'] = df['Listing'].apply(strip_dot_leaders)

# Quick check
print('Sample cleaned entries:')
for t in df['clean_text'].head(5):
    print(f'  {t}')

Sample cleaned entries:
  W City(Washington City)(E)(Feb.27,1797;Ms;Black) . . . 2000.00
  (L)(April 19,1798) . . 1750.00
  WASH.CITY(period high)(1799-1801;26.5;FREE[SL-22x5],PAID;Red,Black-brown) 50.00
  Same(slotted circle)(1801;26.5;FREE;Brown,Purple) 60.00
  WASHN.CITY("N"high)(1801-05;29;FREE;Gray-brown,Red-brown) . . . 45.00


## 2. Signal Detection

### Signal 1: Relationship Indicator Prefix

Entry starts with Same, (L), (E), or their asterisked/plus-prefixed variants.
Leading `*` on its own (marking first-of-town) is NOT a relationship indicator --
it only counts when combined with `(L)` or `(E)`.

In [3]:
RELATIONSHIP_PATTERN = re.compile(
    r'^\+?'
    r'(?:'
    r'Same'
    r'|\*?\(L\)\*?'
    r'|\*?\(E\)\*?'
    r')',
    re.IGNORECASE
)

df['s1_relationship'] = df['clean_text'].apply(
    lambda t: bool(RELATIONSHIP_PATTERN.match(t))
)

print(f'Signal 1 hits: {df["s1_relationship"].sum()}')
print()
print('Examples:')
for t in df.loc[df['s1_relationship'], 'clean_text'].head(8):
    print(f'  {t[:100]}')

Signal 1 hits: 74

Examples:
  (L)(April 19,1798) . . 1750.00
  Same(slotted circle)(1801;26.5;FREE;Brown,Purple) 60.00
  Same(1802-04;28;FREE,PAID;Red-brown) 35.00
  Same(1804;26;Red-brown) 35.00
  Same(1804-05;28.5;Red-brown) 35.00
  Same(1809-16;26.5;FREE;Red-brown,Purple) 30.00
  Same(1813-14;25;FREE;Red-brown,Black-brown) . . . . . 30.00
  Same(1816;26;Red-Brown) . 30.00


### Signal 2: Cross-Reference

Contains "See" followed by a section/place name, AND lacks a semicolon-parenthetical
(which would indicate it is a real listing that happens to also mention a cross-reference).

In [4]:
def detect_cross_reference(text):
    """True if the entry is a pure cross-reference (See X, no semicolon-parenthetical)."""
    has_see = bool(re.search(r'\bSee\b', text))
    has_semicolon_paren = bool(re.search(r'\([^)]*;[^)]*\)', text))
    return has_see and not has_semicolon_paren

df['s2_cross_ref'] = df['clean_text'].apply(detect_cross_reference)

print(f'Signal 2 hits: {df["s2_cross_ref"].sum()}')
print()
for t in df.loc[df['s2_cross_ref'], 'clean_text']:
    print(f'  {t[:120]}')

Signal 2 hits: 7

  Same(with STEAM) See Inland Waterways Listing
  Same(STEAM[SL-33x4.5]) See Inland Waterways
  Same(STEAMBOAT)See Inland Waterways listing
  ALEX., ALEXA. AND ALEXAN. before 1791 - See Virginia Section
  Same(with SHIP[SL-19x5])See Inland Waterways
  Same(STEAM)See Inland Waterways listing . .
  Same(with both above,SHIP[18x5])See Ocean Mail


### Signal 3: Fragment Detection

Two sub-checks:
- Starts with an unmatched closing paren fragment (`)` appears before any `(`)
- Starts with lowercase mid-sentence content (segmentation split mid-entry)

In [5]:
def detect_fragment(text):
    """True if the entry looks like a segmentation fragment."""
    t = text.strip()
    if not t:
        return True
    # Unmatched closing paren: ')' appears before any '('
    first_open = t.find('(')
    first_close = t.find(')')
    if first_close != -1 and (first_open == -1 or first_close < first_open):
        return True
    # Starts with lowercase (mid-sentence fragment)
    # But exclude known patterns: 'c1850' (c-date), 'la.' (abbreviation)
    if t[0].islower() and not re.match(r'^c\d{4}', t):
        return True
    return False

df['s3_fragment'] = df['clean_text'].apply(detect_fragment)

print(f'Signal 3 hits: {df["s3_fragment"].sum()}')
print()
for t in df.loc[df['s3_fragment'], 'clean_text']:
    print(f'  {repr(t[:120])}')

Signal 3 hits: 0



### Signal 4: Trailing Valuation

Entry ends with a recognizable value token: a decimal number (e.g. `3500.00`),
a comma-formatted integer (e.g. `1,500`), a plain integer, `--`, or `---`.

In [6]:
TRAILING_VALUE_PATTERN = re.compile(
    r'(?:'
    r'\d[\d,]*(?:\.\d+)?'  # number: 3500.00 or 1,500 or 50
    r'|---?'                # dashes: -- or ---
    r')\s*$'
)

df['s4_trailing_value'] = df['clean_text'].apply(
    lambda t: bool(TRAILING_VALUE_PATTERN.search(t))
)

print(f'Signal 4 hits: {df["s4_trailing_value"].sum()} / {len(df)}')
print()
misses = df[~df['s4_trailing_value']]
if len(misses):
    print('Entries WITHOUT trailing value:')
    for t in misses['clean_text']:
        print(f'  {repr(t[:120])}')
else:
    print('All entries have a trailing value.')

Signal 4 hits: 128 / 137

Entries WITHOUT trailing value:
  'Same(with STEAM) See Inland Waterways Listing'
  'Same(STEAM[SL-33x4.5]) See Inland Waterways'
  'Same(STEAMBOAT)See Inland Waterways listing'
  'ALEX., ALEXA. AND ALEXAN. before 1791 - See Virginia Section'
  '*Alex.Alexa or Alexand(1792-95,1827;Ms;Black) .'
  'Same(with SHIP[SL-19x5])See Inland Waterways'
  'Same(STEAM)See Inland Waterways listing . .'
  'Same(with both above,SHIP[18x5])See Ocean Mail'
  '*George or Geo Town(1792-1800;Ms;Black) .'


### Signal 5: Core Structural Anatomy

At least one of:
- Parenthetical with semicolons: `(stuff;stuff)`
- Four-digit year in range 1700-1869
- Decade reference: `1850's`, `1850s`
- c-prefixed year: `c1850`

In [7]:
def detect_structural_anatomy(text):
    """Returns a dict of which structural sub-signals are present."""
    result = {
        'semicolon_paren': bool(re.search(r'\([^)]*;[^)]*\)', text)),
        'four_digit_year': bool(re.search(r'\b1[78]\d{2}\b', text)),
        'decade_ref': bool(re.search(r"1[78]\d0['\'s]", text)),
        'c_year': bool(re.search(r'\bc1[78]\d{2}\b', text, re.IGNORECASE)),
    }
    result['any'] = any(result.values())
    return result

anatomy = df['clean_text'].apply(detect_structural_anatomy).apply(pd.Series)
df['s5_semicolon_paren'] = anatomy['semicolon_paren']
df['s5_four_digit_year'] = anatomy['four_digit_year']
df['s5_decade_ref'] = anatomy['decade_ref']
df['s5_c_year'] = anatomy['c_year']
df['s5_anatomy'] = anatomy['any']

print(f'Signal 5 hits: {df["s5_anatomy"].sum()} / {len(df)}')
print()
print('Sub-signal breakdown:')
print(f'  Semicolon-parenthetical: {df["s5_semicolon_paren"].sum()}')
print(f'  Four-digit year:         {df["s5_four_digit_year"].sum()}')
print(f'  Decade reference:        {df["s5_decade_ref"].sum()}')
print(f'  c-prefixed year:         {df["s5_c_year"].sum()}')
print()
misses = df[~df['s5_anatomy']]
if len(misses):
    print(f'Entries WITHOUT structural anatomy ({len(misses)}):')
    for t in misses['clean_text']:
        print(f'  {repr(t[:120])}')
else:
    print('All entries have structural anatomy.')

Signal 5 hits: 130 / 137

Sub-signal breakdown:
  Semicolon-parenthetical: 128
  Four-digit year:         129
  Decade reference:        1
  c-prefixed year:         0

Entries WITHOUT structural anatomy (7):
  'Same(with STEAM) See Inland Waterways Listing'
  'Same(STEAM[SL-33x4.5]) See Inland Waterways'
  'Same(STEAMBOAT)See Inland Waterways listing'
  'Same(Green) . 40.00'
  'Same(with SHIP[SL-19x5])See Inland Waterways'
  'Same(STEAM)See Inland Waterways listing . .'
  'Same(with both above,SHIP[18x5])See Ocean Mail'


## 3. Classification

In [8]:
def classify_entry(row):
    """Apply signals in priority order. Returns (classification, confidence, reason)."""

    # Signal 1: relationship indicator -> auto listing
    if row['s1_relationship']:
        return 'listing', 'high', 'relationship_indicator'

    # Signal 2: cross-reference
    if row['s2_cross_ref']:
        return 'cross_reference', 'high', 'see_pattern'

    # Signal 3: fragment
    if row['s3_fragment']:
        return 'non_entry', 'high', 'fragment'

    # Signals 4+5 conjunction
    has_value = row['s4_trailing_value']
    has_anatomy = row['s5_anatomy']

    if has_value and has_anatomy:
        return 'listing', 'high', 'value_and_anatomy'
    elif has_value and not has_anatomy:
        return 'listing', 'low', 'value_only'
    elif has_anatomy and not has_value:
        return 'listing', 'low', 'anatomy_only'
    else:
        return 'non_entry', 'high', 'no_signals'

classifications = df.apply(classify_entry, axis=1, result_type='expand')
df['classification'] = classifications[0]
df['confidence'] = classifications[1]
df['reason'] = classifications[2]

print('Classification results:')
print(df['classification'].value_counts())
print()
print('By reason:')
print(df.groupby(['classification', 'confidence', 'reason']).size().reset_index(name='count').to_string(index=False))

Classification results:
classification
listing            136
cross_reference      1
Name: count, dtype: int64

By reason:
 classification confidence                 reason  count
cross_reference       high            see_pattern      1
        listing       high relationship_indicator     74
        listing       high      value_and_anatomy     60
        listing        low           anatomy_only      2


## 4. Review: Low-Confidence and Non-Listings

These are the entries that need human eyes.

In [9]:
review = df[(df['confidence'] == 'low') | (df['classification'] != 'listing')].copy()
print(f'Entries requiring review: {len(review)} / {len(df)}')
print()

for _, row in review.iterrows():
    print(f'[{row["classification"]}] ({row["confidence"]}, {row["reason"]})')
    print(f'  {row["clean_text"][:140]}')
    print()

Entries requiring review: 3 / 137

[cross_reference] (high, see_pattern)
  ALEX., ALEXA. AND ALEXAN. before 1791 - See Virginia Section

[listing] (low, anatomy_only)
  *Alex.Alexa or Alexand(1792-95,1827;Ms;Black) .

[listing] (low, anatomy_only)
  *George or Geo Town(1792-1800;Ms;Black) .



## 5. Signal Summary

Full signal report for every entry.

In [10]:
report_cols = [
    'clean_text',
    's1_relationship',
    's2_cross_ref',
    's3_fragment',
    's4_trailing_value',
    's5_anatomy',
    'classification',
    'confidence',
    'reason',
]

# Truncate text for display
report = df[report_cols].copy()
report['clean_text'] = report['clean_text'].str[:80]

pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 82)
pd.set_option('display.width', 200)
report

,clean_text,s1_relationship,s2_cross_ref,s3_fragment,s4_trailing_value,s5_anatomy,classification,confidence,reason
0,"W City(Washington City)(E)(Feb.27,1797;Ms;Black) . . . 2000.00",False,False,False,True,True,listing,high,value_and_anatomy
1,"(L)(April 19,1798) . . 1750.00",True,False,False,True,True,listing,high,relationship_indicator
2,"WASH.CITY(period high)(1799-1801;26.5;FREE[SL-22x5],PAID;Red,Black-brown) 50.00",False,False,False,True,True,listing,high,value_and_anatomy
3,"Same(slotted circle)(1801;26.5;FREE;Brown,Purple) 60.00",True,False,False,True,True,listing,high,relationship_indicator
4,"WASHN.CITY(""N""high)(1801-05;29;FREE;Gray-brown,Red-brown) . . . 45.00",False,False,False,True,True,listing,high,value_and_anatomy
5,"Same(1802-04;28;FREE,PAID;Red-brown) 35.00",True,False,False,True,True,listing,high,relationship_indicator
6,Same(1804;26;Red-brown) 35.00,True,False,False,True,True,listing,high,relationship_indicator
7,Same(1804-05;28.5;Red-brown) 35.00,True,False,False,True,True,listing,high,relationship_indicator
8,"Same(1809-16;26.5;FREE;Red-brown,Purple) 30.00",True,False,False,True,True,listing,high,relationship_indicator
9,"Same(1813-14;25;FREE;Red-brown,Black-brown) . . . . . 30.00",True,False,False,True,True,listing,high,relationship_indicator


## 6. Observations

This section summarizes what the signal detection found on this particular input file,
to inform refinement of thresholds before running on other state catalogs.

In [11]:
total = len(df)
print(f'Total entries: {total}')
print()
print('Signal coverage:')
print(f'  S1 (relationship indicator): {df["s1_relationship"].sum()} ({df["s1_relationship"].sum()/total*100:.1f}%)')
print(f'  S2 (cross-reference):        {df["s2_cross_ref"].sum()} ({df["s2_cross_ref"].sum()/total*100:.1f}%)')
print(f'  S3 (fragment):               {df["s3_fragment"].sum()} ({df["s3_fragment"].sum()/total*100:.1f}%)')
print(f'  S4 (trailing value):         {df["s4_trailing_value"].sum()} ({df["s4_trailing_value"].sum()/total*100:.1f}%)')
print(f'  S5 (structural anatomy):     {df["s5_anatomy"].sum()} ({df["s5_anatomy"].sum()/total*100:.1f}%)')
print()
print('Classification summary:')
for cls in ['listing', 'cross_reference', 'non_entry']:
    subset = df[df['classification'] == cls]
    n = len(subset)
    low = (subset['confidence'] == 'low').sum()
    if n > 0:
        conf_note = f' ({low} low-confidence)' if low else ''
        print(f'  {cls}: {n} ({n/total*100:.1f}%){conf_note}')

Total entries: 137

Signal coverage:
  S1 (relationship indicator): 74 (54.0%)
  S2 (cross-reference):        7 (5.1%)
  S3 (fragment):               0 (0.0%)
  S4 (trailing value):         128 (93.4%)
  S5 (structural anatomy):     130 (94.9%)

Classification summary:
  listing: 136 (99.3%) (2 low-confidence)
  cross_reference: 1 (0.7%)


# Step 2: Structural Segmentation

Splits each listing-classified row into three zones that all downstream extraction
steps depend on: **head**, **paren_body**, and **tail**.

Every subsequent field extraction (town name, dates, size, rates, colors, valuation)
needs to know *which part of the string* to look in. The parenthetical must be isolated
before semicolon fields can be parsed; the valuation must be separated from the attribute
block. This is the chokepoint.

### Three entry forms

| Form | Rule | Example |
|------|------|---------|
| `semicolon_paren` | Last balanced paren group contains `;` | `*Kaskaskia(April 8,1800;Ms;Black) 3500.00` |
| `simple_paren` | No semicolon-paren, but S1 (relationship indicator) hit and entry has parens | `Same(Green) 50.00` |
| `no_paren` | Everything else; any parens are name annotations | `*Ursine 1851 --` |

Why the S1 gate on `simple_paren`: in the ASCC format, non-relationship entries with
parens that lack semicolons are overwhelmingly name annotations -- county disambiguation
like `Shippensville (Clarion Co.)`, abbreviated spellings like `Lewisburg(h)`, or catalog
flags like `(1)Sistersville`. Only relationship-indicator entries use a single-attribute
paren as their modification block (e.g. `Same(Green)`).

### Segmentation zones

- **seg_head** -- text before the attribute parenthetical (town name, relationship indicators, name annotations)
- **seg_paren** -- content inside the attribute parens, excluding the parens themselves (null for no_paren)
- **seg_tail** -- text after the closing paren / trailing valuation

## 2.1 Entry Form Classification

In [12]:
# Only segment rows that Step 1 classified as listings
listings = df[df['classification'] == 'listing'].copy()
print(f'Segmenting {len(listings)} listings')

def find_last_semicolon_paren(text):
    """Find the last balanced paren group that contains a semicolon.
    Returns (open_pos, close_pos) or None."""
    # Walk backward through all ')' positions
    i = len(text) - 1
    while i >= 0:
        if text[i] == ')':
            close_pos = i
            depth = 1
            j = i - 1
            while j >= 0 and depth > 0:
                if text[j] == ')':
                    depth += 1
                elif text[j] == '(':
                    depth -= 1
                j -= 1
            if depth == 0:
                open_pos = j + 1
                body = text[open_pos + 1:close_pos]
                if ';' in body:
                    return (open_pos, close_pos)
            # This paren group had no semicolon; keep scanning left
            i = (j + 1) - 1 if depth == 0 else i - 1
        else:
            i -= 1
    return None

def find_last_paren_group(text):
    """Find the last balanced paren group (any content).
    Returns (open_pos, close_pos) or None."""
    close_pos = text.rfind(')')
    if close_pos == -1:
        return None
    depth = 1
    j = close_pos - 1
    while j >= 0 and depth > 0:
        if text[j] == ')':
            depth += 1
        elif text[j] == '(':
            depth -= 1
        j -= 1
    if depth != 0:
        return None  # unmatched
    open_pos = j + 1
    return (open_pos, close_pos)

def classify_entry_form(row):
    """Determine structural form: semicolon_paren, simple_paren, or no_paren."""
    text = row['clean_text']

    # Form 1: last paren group with semicolons
    if find_last_semicolon_paren(text) is not None:
        return 'semicolon_paren'

    # Form 2: relationship indicator + has parens (single-attribute modification)
    if row['s1_relationship'] and '(' in text:
        return 'simple_paren'

    # Form 3: everything else
    return 'no_paren'

listings['entry_form'] = listings.apply(classify_entry_form, axis=1)

print()
print('Entry form distribution:')
for form, count in listings['entry_form'].value_counts().items():
    print(f'  {form}: {count} ({count/len(listings)*100:.1f}%)')

Segmenting 136 listings

Entry form distribution:
  semicolon_paren: 128 (94.1%)
  simple_paren: 8 (5.9%)


## 2.2 Segmentation

In [13]:
# Trailing value pattern: captures slash-separated valuations, decimals, integers, dashes
TRAILING_VALUE_RE = re.compile(
    r'(\d[\d,]*(?:\.\d+)?(?:/\d[\d,]*(?:\.\d+)?)*|---?)\s*$'
)

def segment_entry(row):
    """Split entry into head / paren_body / tail based on entry_form."""
    text = row['clean_text']
    form = row['entry_form']

    if form == 'semicolon_paren':
        bounds = find_last_semicolon_paren(text)
        if bounds is None:
            return pd.Series({'seg_head': text, 'seg_paren': None, 'seg_tail': None,
                              'seg_error': 'semicolon_paren but no match'})
        open_pos, close_pos = bounds
        head = text[:open_pos].strip()
        paren_body = text[open_pos + 1:close_pos]
        tail = text[close_pos + 1:].strip()
        return pd.Series({'seg_head': head, 'seg_paren': paren_body,
                          'seg_tail': tail, 'seg_error': None})

    elif form == 'simple_paren':
        bounds = find_last_paren_group(text)
        if bounds is None:
            return pd.Series({'seg_head': text, 'seg_paren': None, 'seg_tail': None,
                              'seg_error': 'simple_paren but no paren found'})
        open_pos, close_pos = bounds
        head = text[:open_pos].strip()
        paren_body = text[open_pos + 1:close_pos]
        tail = text[close_pos + 1:].strip()
        return pd.Series({'seg_head': head, 'seg_paren': paren_body,
                          'seg_tail': tail, 'seg_error': None})

    else:  # no_paren
        m = TRAILING_VALUE_RE.search(text)
        if m is None:
            return pd.Series({'seg_head': text, 'seg_paren': None, 'seg_tail': None,
                              'seg_error': 'no_paren but no trailing value'})
        tail = m.group(1)
        head = text[:m.start()].strip()
        return pd.Series({'seg_head': head, 'seg_paren': None,
                          'seg_tail': tail, 'seg_error': None})

segments = listings.apply(segment_entry, axis=1)
listings = pd.concat([listings, segments], axis=1)

# Report errors
errors = listings[listings['seg_error'].notna()]
print(f'Segmentation errors: {len(errors)} / {len(listings)}')
if len(errors):
    for _, row in errors.iterrows():
        print(f'  [{row["entry_form"]}] {row["seg_error"]}')
        print(f'    {row["clean_text"][:140]}')
    print()

print(f'Successful segmentations: {len(listings) - len(errors)}')

Segmentation errors: 0 / 136
Successful segmentations: 136


## 2.3 Validation

In [14]:
ok = listings[listings['seg_error'].isna()]

print('=== SEMICOLON_PAREN examples ===')
sp = ok[ok['entry_form'] == 'semicolon_paren']
print(f'Count: {len(sp)}')
for _, row in sp.head(8).iterrows():
    print(f'  raw:   {row["clean_text"][:100]}')
    print(f'  head:  {row["seg_head"]}')
    print(f'  paren: {row["seg_paren"]}')
    print(f'  tail:  {row["seg_tail"]}')
    print()

print('=== SIMPLE_PAREN examples ===')
simp = ok[ok['entry_form'] == 'simple_paren']
print(f'Count: {len(simp)}')
for _, row in simp.head(8).iterrows():
    print(f'  raw:   {row["clean_text"][:100]}')
    print(f'  head:  {row["seg_head"]}')
    print(f'  paren: {row["seg_paren"]}')
    print(f'  tail:  {row["seg_tail"]}')
    print()

print('=== NO_PAREN examples ===')
np_ = ok[ok['entry_form'] == 'no_paren']
print(f'Count: {len(np_)}')
for _, row in np_.head(8).iterrows():
    print(f'  raw:   {row["clean_text"][:100]}')
    print(f'  head:  {row["seg_head"]}')
    print(f'  tail:  {row["seg_tail"]}')
    print()

=== SEMICOLON_PAREN examples ===
Count: 128
  raw:   W City(Washington City)(E)(Feb.27,1797;Ms;Black) . . . 2000.00
  head:  W City(Washington City)(E)
  paren: Feb.27,1797;Ms;Black
  tail:  . . . 2000.00

  raw:   WASH.CITY(period high)(1799-1801;26.5;FREE[SL-22x5],PAID;Red,Black-brown) 50.00
  head:  WASH.CITY(period high)
  paren: 1799-1801;26.5;FREE[SL-22x5],PAID;Red,Black-brown
  tail:  50.00

  raw:   Same(slotted circle)(1801;26.5;FREE;Brown,Purple) 60.00
  head:  Same(slotted circle)
  paren: 1801;26.5;FREE;Brown,Purple
  tail:  60.00

  raw:   WASHN.CITY("N"high)(1801-05;29;FREE;Gray-brown,Red-brown) . . . 45.00
  head:  WASHN.CITY("N"high)
  paren: 1801-05;29;FREE;Gray-brown,Red-brown
  tail:  . . . 45.00

  raw:   Same(1802-04;28;FREE,PAID;Red-brown) 35.00
  head:  Same
  paren: 1802-04;28;FREE,PAID;Red-brown
  tail:  35.00

  raw:   Same(1804;26;Red-brown) 35.00
  head:  Same
  paren: 1804;26;Red-brown
  tail:  35.00

  raw:   Same(1804-05;28.5;Red-brown) 35.00
  head:  Sam

### Sanity checks

- Every listing should have a non-empty seg_tail (the valuation is always present -- verified by Signal 4).
- Every non-relationship listing should have a non-empty seg_head (the town name).
- seg_paren should be non-empty for semicolon_paren and simple_paren forms.

In [15]:
ok = listings[listings['seg_error'].isna()]

# Check 1: seg_tail should always be non-empty
empty_tail = ok[ok['seg_tail'].isna() | (ok['seg_tail'] == '')]
print(f'Empty seg_tail: {len(empty_tail)}')
if len(empty_tail):
    for _, row in empty_tail.iterrows():
        print(f'  [{row["entry_form"]}] {row["clean_text"][:120]}')
    print()

# Check 2: seg_head should be non-empty for non-relationship listings
non_rel = ok[~ok['s1_relationship']]
empty_head = non_rel[non_rel['seg_head'].isna() | (non_rel['seg_head'] == '')]
print(f'Empty seg_head (non-relationship): {len(empty_head)}')
if len(empty_head):
    for _, row in empty_head.iterrows():
        print(f'  [{row["entry_form"]}] {row["clean_text"][:120]}')
    print()

# Check 3: seg_paren should be non-empty for paren forms
paren_forms = ok[ok['entry_form'].isin(['semicolon_paren', 'simple_paren'])]
empty_paren = paren_forms[paren_forms['seg_paren'].isna() | (paren_forms['seg_paren'] == '')]
print(f'Empty seg_paren (paren forms): {len(empty_paren)}')
if len(empty_paren):
    for _, row in empty_paren.iterrows():
        print(f'  [{row["entry_form"]}] {row["clean_text"][:120]}')
    print()

# Check 4: show all simple_paren entries for manual verification
# (these are the highest-risk form -- make sure none are name annotations)
print(f'\n=== ALL simple_paren entries ({len(simp)}) ===')
for _, row in ok[ok['entry_form'] == 'simple_paren'].iterrows():
    print(f'  head={row["seg_head"]!r}  paren={row["seg_paren"]!r}  tail={row["seg_tail"]!r}')

Empty seg_tail: 0
Empty seg_head (non-relationship): 0
Empty seg_paren (paren forms): 0

=== ALL simple_paren entries (8) ===
  head='(L)'  paren='April 19,1798'  tail='. . 1750.00'
  head='Same'  paren='with STEAM'  tail='See Inland Waterways Listing'
  head='Same'  paren='STEAM[SL-33x4.5]'  tail='See Inland Waterways'
  head='Same'  paren='STEAMBOAT'  tail='See Inland Waterways listing'
  head='Same'  paren='Green'  tail='. 40.00'
  head='Same'  paren='with SHIP[SL-19x5]'  tail='See Inland Waterways'
  head='Same'  paren='STEAM'  tail='See Inland Waterways listing . .'
  head='Same'  paren='with both above,SHIP[18x5]'  tail='See Ocean Mail'


## 2.4 Step 2 Summary

In [16]:
total = len(listings)
ok_count = len(ok)
err_count = len(errors)

print(f'Step 2: Structural Segmentation')
print(f'  Input listings: {total}')
print(f'  Successful:     {ok_count} ({ok_count/total*100:.1f}%)')
print(f'  Errors:         {err_count} ({err_count/total*100:.1f}%)')
print()
print(f'Entry form breakdown:')
for form in ['semicolon_paren', 'simple_paren', 'no_paren']:
    n = (listings['entry_form'] == form).sum()
    print(f'  {form}: {n} ({n/total*100:.1f}%)')

Step 2: Structural Segmentation
  Input listings: 136
  Successful:     136 (100.0%)
  Errors:         0 (0.0%)

Entry form breakdown:
  semicolon_paren: 128 (94.1%)
  simple_paren: 8 (5.9%)
  no_paren: 0 (0.0%)


# Step 3: Paren Field Splitting and Tail Extraction

Two mechanical splits:

1. **Paren fields** -- split `seg_paren` on semicolons into positional tokens. No semantic
   interpretation yet; just count and index. Only applies to `semicolon_paren` and
   `simple_paren` forms.

2. **Tail decomposition** -- split `seg_tail` into `tail_annotation` (nullable) and
   `tail_valuation`. For paren forms, `seg_tail` may contain annotation text between
   the closing `)` and the trailing valuation (~3% of semicolon-paren entries have this:
   "Soldier's mail", "See Way Mail Listing", etc.). For `no_paren` entries, Step 2 already
   isolated the valuation, so `tail_annotation` is null.

### Valuation patterns

The trailing valuation token is one of:
- Plain integer: `125`
- Comma-formatted: `1,200`
- Decimal: `3500.00`
- Slash-separated tiers: `200/15`
- Range: `125-200`
- Dashes (unpriced): `--` or `---`

Slash-separated valuations represent date-period pricing tiers per the domain model
(position 1 = earliest period). These are split into individual `PostmarkValuation`
records in a later step.

## 3.1 Paren Field Splitting

In [17]:
def split_paren_fields(row):
    """Split seg_paren on semicolons into positional list."""
    paren = row['seg_paren']
    if paren is None or (isinstance(paren, float) and pd.isna(paren)):
        return []
    fields = [f.strip() for f in paren.split(';')]
    return fields

listings['paren_fields'] = listings.apply(split_paren_fields, axis=1)
listings['paren_field_count'] = listings['paren_fields'].apply(len)

print('Paren field count distribution:')
fc = listings['paren_field_count'].value_counts().sort_index()
for n, count in fc.items():
    print(f'  {n} fields: {count} ({count/len(listings)*100:.1f}%)')
print()
print(f'Entries with 0 fields (no_paren): {(listings["paren_field_count"] == 0).sum()}')

Paren field count distribution:
  1 fields: 8 (5.9%)
  2 fields: 1 (0.7%)
  3 fields: 74 (54.4%)
  4 fields: 51 (37.5%)
  5 fields: 2 (1.5%)

Entries with 0 fields (no_paren): 0


In [18]:
# Show examples at each field count
for n in sorted(listings['paren_field_count'].unique()):
    subset = listings[listings['paren_field_count'] == n]
    print(f'=== {n} FIELDS ({len(subset)} entries) ===')
    for _, row in subset.head(5).iterrows():
        print(f'  raw:    {row["clean_text"][:110]}')
        if n > 0:
            print(f'  fields: {row["paren_fields"]}')
        print()

=== 1 FIELDS (8 entries) ===
  raw:    (L)(April 19,1798) . . 1750.00
  fields: ['April 19,1798']

  raw:    Same(with STEAM) See Inland Waterways Listing
  fields: ['with STEAM']

  raw:    Same(STEAM[SL-33x4.5]) See Inland Waterways
  fields: ['STEAM[SL-33x4.5]']

  raw:    Same(STEAMBOAT)See Inland Waterways listing
  fields: ['STEAMBOAT']

  raw:    Same(Green) . 40.00
  fields: ['Green']

=== 2 FIELDS (1 entries) ===
  raw:    Same/G.P.O.(1854;Red) 75.00
  fields: ['1854', 'Red']

=== 3 FIELDS (74 entries) ===
  raw:    W City(Washington City)(E)(Feb.27,1797;Ms;Black) . . . 2000.00
  fields: ['Feb.27,1797', 'Ms', 'Black']

  raw:    Same(1804;26;Red-brown) 35.00
  fields: ['1804', '26', 'Red-brown']

  raw:    Same(1804-05;28.5;Red-brown) 35.00
  fields: ['1804-05', '28.5', 'Red-brown']

  raw:    Same(1816;26;Red-Brown) . 30.00
  fields: ['1816', '26', 'Red-Brown']

  raw:    Same(small date,D.C. spaced 1mm)(1837-39;30;Orange) 25.00
  fields: ['1837-39', '30', 'Orange']

=== 4 FI

## 3.2 Tail Decomposition

Split `seg_tail` into `tail_annotation` (text before the valuation, nullable) and
`tail_valuation` (the trailing value token). The valuation regex is anchored at end
of string and handles slash-separated tiers (`200/15`), ranges (`125-200`), decimals
(`3500.00`), and dashes (`--`/`---`).

In [19]:
# Extended trailing value pattern: captures slash tiers, ranges, decimals, dashes
TAIL_VALUE_RE = re.compile(
    r'('
    r'\d[\d,]*(?:\.\d+)?'        # first number: 125, 1,200, 3500.00
    r'(?:[-/]\d[\d,]*(?:\.\d+)?)*'  # optional slash tiers or range: /15, -200
    r'|---?'                      # dashes: -- or ---
    r')\s*$'
)

def decompose_tail(row):
    """Split seg_tail into annotation (nullable) and valuation."""
    tail = row['seg_tail']
    form = row['entry_form']

    if tail is None or (isinstance(tail, float) and pd.isna(tail)) or tail.strip() == '':
        return pd.Series({'tail_annotation': None, 'tail_valuation': None,
                          'tail_error': 'empty tail'})

    tail = tail.strip()

    # For no_paren, Step 2 already isolated the valuation
    if form == 'no_paren':
        return pd.Series({'tail_annotation': None, 'tail_valuation': tail,
                          'tail_error': None})

    # For paren forms, split on trailing value
    m = TAIL_VALUE_RE.search(tail)
    if m is None:
        return pd.Series({'tail_annotation': tail if tail else None,
                          'tail_valuation': None,
                          'tail_error': 'no valuation found in tail'})

    valuation = m.group(1)
    annotation = tail[:m.start()].strip()
    if annotation in ('', '.', '*'):
        annotation = None
    return pd.Series({
        'tail_annotation': annotation,
        'tail_valuation': valuation,
        'tail_error': None
    })

tail_parts = listings.apply(decompose_tail, axis=1)
listings = pd.concat([listings, tail_parts], axis=1)

# Report
errors = listings[listings['tail_error'].notna()]
print(f'Tail decomposition errors: {len(errors)} / {len(listings)}')
if len(errors):
    for _, row in errors.head(10).iterrows():
        print(f'  [{row["entry_form"]}] {row["tail_error"]}')
        print(f'    seg_tail={row["seg_tail"]!r}')
        print(f'    raw: {row["clean_text"][:120]}')
        print()

has_annotation = listings['tail_annotation'].notna() & (listings['tail_annotation'] != '')
print(f'Entries with tail annotation: {has_annotation.sum()} ({has_annotation.sum()/len(listings)*100:.1f}%)')

Tail decomposition errors: 8 / 136
  [simple_paren] no valuation found in tail
    seg_tail='See Inland Waterways Listing'
    raw: Same(with STEAM) See Inland Waterways Listing

  [simple_paren] no valuation found in tail
    seg_tail='See Inland Waterways'
    raw: Same(STEAM[SL-33x4.5]) See Inland Waterways

  [simple_paren] no valuation found in tail
    seg_tail='See Inland Waterways listing'
    raw: Same(STEAMBOAT)See Inland Waterways listing

  [semicolon_paren] no valuation found in tail
    seg_tail='.'
    raw: *Alex.Alexa or Alexand(1792-95,1827;Ms;Black) .

  [simple_paren] no valuation found in tail
    seg_tail='See Inland Waterways'
    raw: Same(with SHIP[SL-19x5])See Inland Waterways

  [simple_paren] no valuation found in tail
    seg_tail='See Inland Waterways listing . .'
    raw: Same(STEAM)See Inland Waterways listing . .

  [simple_paren] no valuation found in tail
    seg_tail='See Ocean Mail'
    raw: Same(with both above,SHIP[18x5])See Ocean Mail

  [semicolo

In [20]:
# Show all tail annotations for manual inspection
annotated = listings[listings['tail_annotation'].notna() & (listings['tail_annotation'] != '')]
if len(annotated):
    print(f'=== ALL TAIL ANNOTATIONS ({len(annotated)}) ===')
    for _, row in annotated.iterrows():
        print(f'  annotation={row["tail_annotation"]!r}  val={row["tail_valuation"]!r}')
        print(f'    raw: {row["clean_text"][:120]}')
        print()
else:
    print('No tail annotations found in this file.')

=== ALL TAIL ANNOTATIONS (49) ===
  annotation='. . .'  val='2000.00'
    raw: W City(Washington City)(E)(Feb.27,1797;Ms;Black) . . . 2000.00

  annotation='. .'  val='1750.00'
    raw: (L)(April 19,1798) . . 1750.00

  annotation='. . .'  val='45.00'
    raw: WASHN.CITY("N"high)(1801-05;29;FREE;Gray-brown,Red-brown) . . . 45.00

  annotation='. . . . .'  val='30.00'
    raw: Same(1813-14;25;FREE;Red-brown,Black-brown) . . . . . 30.00

  annotation='. . . . . . . . . . .'  val='25.00'
    raw: Same(Letters 5mm high)(1822-27;32;PAID,FREE;Red-orange,Green) . . . . . . . . . . . 25.00

  annotation='. .'  val='30.00'
    raw: Same(Letters 4.5mm high,no fancy marks)(1827;30;FREE;Red-orange) . . 30.00

  annotation='See Inland Waterways Listing'  val=None
    raw: Same(with STEAM) See Inland Waterways Listing

  annotation='See Inland Waterways'  val=None
    raw: Same(STEAM[SL-33x4.5]) See Inland Waterways

  annotation='See Inland Waterways listing'  val=None
    raw: Same(STEAMBOAT)See I

## 3.3 Valuation Tier Splitting

Split slash-separated valuations into positional tiers. `200/15` becomes
`['200', '15']` where position 1 = earliest date period.

In [21]:
def split_valuation_tiers(val_str):
    """Split a valuation string into positional tiers.
    Returns list of tier strings. Dashes become [None]. Ranges stay intact."""
    if val_str is None:
        return []
    val_str = val_str.strip()
    if val_str in ('--', '---'):
        return [None]  # unpriced
    # Split on / for tier separation
    tiers = val_str.split('/')
    return tiers

listings['valuation_tiers'] = listings['tail_valuation'].apply(split_valuation_tiers)
listings['valuation_tier_count'] = listings['valuation_tiers'].apply(len)

print('Valuation tier count distribution:')
tc = listings['valuation_tier_count'].value_counts().sort_index()
for n, count in tc.items():
    print(f'  {n} tiers: {count} ({count/len(listings)*100:.1f}%)')
print()

# Show multi-tier examples
multi = listings[listings['valuation_tier_count'] > 1]
if len(multi):
    print(f'=== MULTI-TIER VALUATIONS ({len(multi)}) ===')
    for _, row in multi.head(10).iterrows():
        print(f'  {row["tail_valuation"]} -> {row["valuation_tiers"]}')
        print(f'    raw: {row["clean_text"][:100]}')
        print()

# Show unpriced entries
unpriced = listings[listings['valuation_tiers'].apply(lambda t: len(t) == 1 and t[0] is None)]
if len(unpriced):
    print(f'=== UNPRICED ENTRIES ({len(unpriced)}) ===')
    for _, row in unpriced.head(10).iterrows():
        print(f'  {row["clean_text"][:100]}')

Valuation tier count distribution:
  0 tiers: 8 (5.9%)
  1 tiers: 128 (94.1%)



## 3.4 Step 3 Summary

In [22]:
total = len(listings)
seg_err = listings['seg_error'].notna().sum() if 'seg_error' in listings.columns else 0
tail_err = listings['tail_error'].notna().sum()
has_ann = (listings['tail_annotation'].notna() & (listings['tail_annotation'] != '')).sum()

print(f'Step 3: Paren Field Splitting and Tail Extraction')
print(f'  Input listings: {total}')
print()
print(f'  Paren field counts:')
for n, count in listings['paren_field_count'].value_counts().sort_index().items():
    print(f'    {n} fields: {count}')
print()
print(f'  Tail decomposition:')
print(f'    Errors:      {tail_err}')
print(f'    Annotations: {has_ann}')
print()
print(f'  Valuation tiers:')
for n, count in listings['valuation_tier_count'].value_counts().sort_index().items():
    label = 'unpriced' if n == 1 and listings[listings['valuation_tier_count'] == n]['valuation_tiers'].apply(lambda t: t[0] is None).all() else f'{n}-tier'
    print(f'    {n} tiers: {count}')

Step 3: Paren Field Splitting and Tail Extraction
  Input listings: 136

  Paren field counts:
    1 fields: 8
    2 fields: 1
    3 fields: 74
    4 fields: 51
    5 fields: 2

  Tail decomposition:
    Errors:      8
    Annotations: 49

  Valuation tiers:
    0 tiers: 8
    1 tiers: 128


# Step 4: Head Parsing

Extracts structured components from `seg_head`. The head contains everything
before the attribute parenthetical: relationship indicators, the first-of-town
marker, the town/postmark inscription text, and name-annotation parens.

### Outputs

- **head_first_of_town** (bool) — leading `*` was present (ASCC convention: first listing for this town)
- **head_rel_type** (nullable str) — relationship indicator token: `Same`, `(L)`, `(E)`, `(L)*`, `(E)*`, or null
- **head_name_body** (nullable str) — text remaining after stripping `*`/`+`/rel-indicator AND removing annotation parens. This is the raw town-marking inscription (e.g. `LINCOLN/ILL`, `Kaskaskia`). Null when the head is a pure relationship indicator with no residual text.
- **head_annotations** (list of str) — parenthesized annotation contents from the head, in order. Captures period markers `(E)`/`(L)`, canonical-name expansions `(Cahokia)`, place-type qualifiers `(Fort)`, county disambiguations `(Clarion Co.)`, and letter-variant appendages `(h)`.

Section-context columns (`Default Shape`, `Page`, `Images Above`, `Institutional Ownership`) are already carried on the `listings` DataFrame from the input CSV and are validated at the end of this step.

In [23]:
# --- Step 4.1: Head parsing ---

PAREN_GROUP_RE = re.compile(r'\(([^)]*)\)')

# Relationship indicator at head start (after * and + have been stripped).
# Matches: Same, (L), (E), (L)*, (E)*
# Does NOT consume a leading * -- that's already handled as first_of_town.
REL_INDICATOR_RE = re.compile(
    r'^(?:'
    r'Same'
    r'|\([LE]\)\*?'
    r')'
)

def parse_head(row):
    """Extract structured components from seg_head."""
    head = str(row['seg_head']) if pd.notna(row['seg_head']) else ''

    # 1. First-of-town marker (leading *)
    first_of_town = head.startswith('*')
    if first_of_town:
        head = head[1:]

    # 2. Plus prefix (rare; allowed by S1 regex but uncommon)
    plus_prefix = head.startswith('+')
    if plus_prefix:
        head = head[1:]

    # 3. Relationship indicator
    rel_type = None
    if row['s1_relationship']:
        m = REL_INDICATOR_RE.match(head)
        if m:
            rel_type = m.group(0)
            head = head[m.end():]

    # 4. Annotations: all (...) groups remaining in head
    annotations = PAREN_GROUP_RE.findall(head)

    # 5. Name body: head text with annotation parens removed, stripped
    name_body = PAREN_GROUP_RE.sub('', head).strip()
    name_body = name_body if name_body else None

    return pd.Series({
        'head_first_of_town': first_of_town,
        'head_rel_type': rel_type,
        'head_name_body': name_body,
        'head_annotations': annotations,
    })

head_parts = listings.apply(parse_head, axis=1)
listings = pd.concat([listings, head_parts], axis=1)

print(f'Step 4: Head parsing applied to {len(listings)} listings')
print(f'  First-of-town markers: {listings["head_first_of_town"].sum()}')
print(f'  Relationship indicators: {listings["head_rel_type"].notna().sum()}')
has_name = listings['head_name_body'].notna()
print(f'  Entries with name body: {has_name.sum()} ({has_name.sum()/len(listings)*100:.1f}%)')
has_ann = listings['head_annotations'].apply(lambda a: len(a) > 0)
print(f'  Entries with annotations: {has_ann.sum()} ({has_ann.sum()/len(listings)*100:.1f}%)')

Step 4: Head parsing applied to 136 listings
  First-of-town markers: 2
  Relationship indicators: 74
  Entries with name body: 70 (51.5%)
  Entries with annotations: 49 (36.0%)


In [24]:
# --- Step 4.2: Relationship indicator distribution ---

rel_counts = listings['head_rel_type'].value_counts(dropna=False)
print('Relationship indicator distribution:')
for val, count in rel_counts.items():
    label = repr(val) if val is not None else '(none -- independent entry)'
    print(f'  {label}: {count}')
print()

# Show a few examples per rel type
for rt in listings['head_rel_type'].dropna().unique():
    subset = listings[listings['head_rel_type'] == rt]
    print(f'=== rel_type={rt!r} ({len(subset)} entries) ===')
    for _, row in subset.head(4).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  ->  name_body={row["head_name_body"]!r}  ann={row["head_annotations"]}')
    print()

Relationship indicator distribution:
  'Same': 73
  (none -- independent entry): 62
  '(L)': 1

=== rel_type='(L)' (1 entries) ===
  seg_head='(L)'  ->  name_body=None  ann=[]

=== rel_type='Same' (73 entries) ===
  seg_head='Same(slotted circle)'  ->  name_body=None  ann=['slotted circle']
  seg_head='Same'  ->  name_body=None  ann=[]
  seg_head='Same'  ->  name_body=None  ann=[]
  seg_head='Same'  ->  name_body=None  ann=[]



In [25]:
# --- Step 4.3: Name body and annotation inspection ---

# Independent entries (no rel indicator) should always have a name body
independent = listings[listings['head_rel_type'].isna()]
missing_name = independent[independent['head_name_body'].isna()]
print(f'Independent entries missing name body: {len(missing_name)} / {len(independent)}')
if len(missing_name):
    for _, row in missing_name.head(5).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  clean_text={row["clean_text"][:100]}')
print()

# Annotation value distribution
all_annotations = [a for ann_list in listings['head_annotations'] for a in ann_list]
ann_counts = pd.Series(all_annotations).value_counts()
print(f'Annotation values ({len(all_annotations)} total across {has_ann.sum()} entries):')
for val, count in ann_counts.head(20).items():
    print(f'  {val!r}: {count}')
print()

# Entries with >1 annotation
multi_ann = listings[listings['head_annotations'].apply(len) > 1]
if len(multi_ann):
    print(f'=== MULTI-ANNOTATION HEADS ({len(multi_ann)}) ===')
    for _, row in multi_ann.head(10).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  ->  name={row["head_name_body"]!r}  ann={row["head_annotations"]}')
    print()

# Rel-indicator entries WITH a name body (e.g. Same/ILL)
rel_with_name = listings[listings['head_rel_type'].notna() & listings['head_name_body'].notna()]
if len(rel_with_name):
    print(f'=== REL-INDICATOR ENTRIES WITH NAME BODY ({len(rel_with_name)}) ===')
    for _, row in rel_with_name.head(15).iterrows():
        print(f'  seg_head={row["seg_head"]!r}  rel={row["head_rel_type"]!r}  name={row["head_name_body"]!r}  ann={row["head_annotations"]}')

Independent entries missing name body: 0 / 62

Annotation values (50 total across 49 entries):
  'hollow letters': 2
  '"N"high': 2
  'ornament at bottom': 2
  '3mm letters': 2
  'both end "A"s high': 2
  'diamond': 2
  'both end"A"s high,line below date': 1
  'FREE straight': 1
  'FREE serif': 1
  'FREE block letters': 1
  '"Y"high': 1
  'period and "A"in VA high': 1
  'both end"A"s high': 1
  'curved dash below date': 1
  'dot and two curved lines below date': 1
  'FREE curved above circle': 1
  '"N"&"K"high': 1
  '"T"high': 1
  '"N"&"A"high': 1
  '"N"&"A"high,dot and two curved dashes': 1

=== MULTI-ANNOTATION HEADS (1) ===
  seg_head='W City(Washington City)(E)'  ->  name='W City'  ann=['Washington City', 'E']

=== REL-INDICATOR ENTRIES WITH NAME BODY (8) ===
  seg_head='Same D.C./5'  rel='Same'  name='D.C./5'  ann=[]
  seg_head='Same/D.C.'  rel='Same'  name='/D.C.'  ann=[]
  seg_head='Same/DUE 6(attached below)'  rel='Same'  name='/DUE 6'  ann=['attached below']
  seg_head='Same/D

In [26]:
# --- Step 4.4: Section-context pass-through validation ---

context_cols = ['Default Shape', 'Page', 'Images Above', 'Institutional Ownership']
print('Section-context columns carried on listings DataFrame:')
for col in context_cols:
    if col in listings.columns:
        non_null = listings[col].notna().sum()
        nunique = listings[col].nunique()
        print(f'  {col}: {non_null} non-null, {nunique} distinct values')
        # Show value distribution for low-cardinality columns
        if nunique <= 15:
            for val, count in listings[col].value_counts().head(10).items():
                print(f'    {val!r}: {count}')
    else:
        print(f'  {col}: *** MISSING ***')
    print()

Section-context columns carried on listings DataFrame:
  Default Shape: 136 non-null, 1 distinct values
    'C - Circle': 136

  Page: 136 non-null, 4 distinct values
    45: 43
    44: 37
    46: 35
    47: 21

  Images Above: 136 non-null, 6 distinct values
    0.0: 107
    2.0: 16
    1.0: 8
    3.0: 3
    7.0: 1
    4.0: 1

  Institutional Ownership: *** MISSING ***



## 4.5 Step 4 Summary

Head parsing is purely mechanical decomposition of `seg_head`. It does not resolve
inherited town names (that requires parent-child linkage in Step 6) or classify
annotation semantics (period marker vs. canonical name vs. county -- also Step 6).

At this point each listing row carries:
- **Structural segments** (Step 2): `seg_head`, `seg_paren`, `seg_tail`
- **Paren fields** (Step 3.1): `paren_fields`, `paren_field_count`
- **Tail parts** (Step 3.2-3.3): `tail_annotation`, `tail_valuation`, `valuation_tiers`
- **Head parts** (Step 4): `head_first_of_town`, `head_rel_type`, `head_name_body`, `head_annotations`
- **Section context**: `Default Shape`, `Page`, `Images Above`, `Institutional Ownership`

In [27]:
# --- Step 4.5: Summary ---

total = len(listings)
print(f'Step 4: Head Parsing')
print(f'  Input listings: {total}')
print()
print(f'  First-of-town: {listings["head_first_of_town"].sum()} ({listings["head_first_of_town"].sum()/total*100:.1f}%)')
print()
print(f'  Relationship indicators:')
for val, count in listings['head_rel_type'].value_counts(dropna=True).items():
    print(f'    {val!r}: {count}')
none_count = listings['head_rel_type'].isna().sum()
print(f'    (independent): {none_count}')
print()
print(f'  Name body present: {listings["head_name_body"].notna().sum()}')
print(f'  Entries with annotations: {(listings["head_annotations"].apply(len) > 0).sum()}')

Step 4: Head Parsing
  Input listings: 136

  First-of-town: 2 (1.5%)

  Relationship indicators:
    'Same': 73
    '(L)': 1
    (independent): 62

  Name body present: 70
  Entries with annotations: 49


# Step 5: Paren Field-Type Classification

Classifies each semicolon-delimited token in `paren_fields` as one of six
content types, using intrinsic text signals only (no positional assumptions).

### Field types

| Type | Signal | Examples |
|------|--------|----------|
| `date` | Month name, 4-digit year 1500-1899, decade, c-prefix | `April 8,1800`, `1850's`, `c1840` |
| `ms` | Exact token `Ms` | `Ms` |
| `size` | Shape-code prefix, WxH dimension, bare 1-3 digit number, dateformat suffix | `SL-16.5x5,MDD`, `DC-25,YD`, `32` |
| `rate` | PAID/FREE/STEAM/DUE keyword, `[...]` bracket, P.M. notation, frank | `FREE`, `25[ms]`, `P.M.Free`, `PAID 6,40` |
| `color` | All comma-separated tokens are known ASCC color names | `Black`, `Red,Blue`, `Olive-Yellow` |
| `other` | None of the above | Residual; inspect for missed patterns |

### Priority order

1. `ms` (exact match -- fastest, no ambiguity)
2. `date` (month names and years are unique to date fields)
3. `rate` (brackets and rate keywords disambiguate from size before bare-number fallback)
4. `size` (shape codes, dimension patterns, dateformat suffixes)
5. `color` (known color vocabulary)
6. Bare 1-3 digit number without other signals -> `size` (ASCC convention: bare numbers in paren fields are mm dimensions; rate values use brackets or keywords)
7. `other`

In [28]:
# --- Step 5.1: Field-type detectors ---

MONTHS_PAT = (
    r'(?:\b(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|June?|July?'
    r'|Aug(?:ust)?|Sep(?:t(?:ember)?)?\\.?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)\b)'
)

DATE_FIELD_RE = re.compile(
    r'(?:' + MONTHS_PAT + r'|c?1[5-8]\d{2})',
    re.IGNORECASE
)

# Tokens that appear as comma-suffixes in the marking-description field:
# date-format codes (MDD, MD, YD, YMD, YMDD) plus NOR (No Outer Rim)
SIZE_SUFFIX_PAT = r'(?:YMDD|MDD|YMD|YD|MD|NOR)'

# Shape codes from ASCC header -- ordered longest-first to avoid prefix collisions
SHAPE_CODE_PAT = (
    r'(?:DLDC|DLDO|DLC|DLO|Octagon|Box|Arc|Pmk|SL|DC|DO|NOR|O|C)'
)

SIZE_FIELD_RE = re.compile(
    r'(?:'
    + SHAPE_CODE_PAT + r'[\-\s]?\d'
    + r'|\d+\.?\d*\s*x\s*\d'
    + r'|^-{1,2}(?:\s*,'
    + SIZE_SUFFIX_PAT + r')?$'
    + r'|\d+\.?\d*\s*,'
    + SIZE_SUFFIX_PAT
    + r'|^'
    + SIZE_SUFFIX_PAT + r'$'
    + r')',
    re.IGNORECASE
)

RATE_FIELD_RE = re.compile(
    r'(?:'
    r'\bPAID\b|\bFREE\b|\bSTEAM\b|\bDUE\b'
    r'|\bP\.?M\.?'
    r'|\bfrank\b'
    r'|\[[^\]]*\]'         # brackets: [ms], [C], [hdstp rate], [cogged circle]
    r'|\bwith\s+\d'        # "with 24" = with adhesive
    r')',
    re.IGNORECASE
)

KNOWN_COLORS = {
    'black', 'red', 'blue', 'green', 'brown', 'orange', 'purple',
    'magenta', 'yellow', 'olive', 'violet', 'carmine', 'vermilion',
    'pink', 'gray', 'grey', 'buff', 'salmon', 'rose', 'maroon',
    'crimson', 'indigo', 'lilac', 'scarlet', 'amber',
}

def is_color_token(tok):
    """Check a single token (possibly hyphen-compound) against color vocabulary."""
    parts = tok.strip().lower().split('-')
    return all(p in KNOWN_COLORS for p in parts if p)

def is_color_field(field):
    """True if all comma-separated tokens in the field are known colors."""
    tokens = [t.strip() for t in field.split(',') if t.strip()]
    return bool(tokens) and all(is_color_token(t) for t in tokens)

BARE_NUMBER_RE = re.compile(r'^\d{1,3}(?:\.\d+)?$')

def classify_paren_field(field_text):
    """Classify a single paren field by intrinsic content signals.
    Returns one of: date, ms, size, rate, color, other, empty."""
    f = field_text.strip()
    if not f:
        return 'empty'

    # 1. Manuscript (exact)
    if f == 'Ms':
        return 'ms'

    # 2. Date expression
    if DATE_FIELD_RE.search(f):
        return 'date'

    # 3. Rate/auxmark (checked before size -- brackets disambiguate)
    if RATE_FIELD_RE.search(f):
        return 'rate'

    # 4. Size/shape/dateformat composite
    if SIZE_FIELD_RE.search(f):
        return 'size'

    # 5. Color
    if is_color_field(f):
        return 'color'

    # 6. Bare small number -> size by ASCC convention
    if BARE_NUMBER_RE.match(f):
        return 'size'

    return 'other'

# Quick self-test
assert classify_paren_field('April 8,1800') == 'date'
assert classify_paren_field('1850-53') == 'date'
assert classify_paren_field("1850's") == 'date'
assert classify_paren_field('Ms') == 'ms'
assert classify_paren_field('SL-16.5x5,MDD') == 'size'
assert classify_paren_field('DC-25,YD') == 'size'
assert classify_paren_field('32') == 'size'
assert classify_paren_field('--,YD') == 'size'
assert classify_paren_field('FREE') == 'rate'
assert classify_paren_field('25[ms]') == 'rate'
assert classify_paren_field('P.M.Free') == 'rate'
assert classify_paren_field('Geo.Fisher P.M.frank') == 'rate'
assert classify_paren_field('Black') == 'color'
assert classify_paren_field('Red,Blue') == 'color'
assert classify_paren_field('Olive-Yellow') == 'color'
print('All self-tests passed')

All self-tests passed


In [29]:
# --- Step 5.2: Apply classifier to all paren fields ---

def classify_all_fields(paren_fields):
    """Classify each field in the list. Returns parallel list of type labels."""
    types = [classify_paren_field(f) for f in paren_fields]
    
    # Positional disambiguation: ASCC entries have at most one size field.
    # If a second 'size' appears and it's a bare number (no shape code, no
    # dateformat, no dimension separator), reclassify it as 'rate'.
    size_seen = False
    for i, (field, ftype) in enumerate(zip(paren_fields, types)):
        if ftype == 'size':
            if size_seen and BARE_NUMBER_RE.match(field.strip()):
                types[i] = 'rate'
            else:
                size_seen = True

    return types

listings['paren_field_types'] = listings['paren_fields'].apply(classify_all_fields)

# Flatten for aggregate stats
all_fields = []
for _, row in listings.iterrows():
    for pos, (field, ftype) in enumerate(zip(row['paren_fields'], row['paren_field_types'])):
        all_fields.append({
            'position': pos,
            'field_text': field,
            'field_type': ftype,
            'field_count': row['paren_field_count'],
        })

field_df = pd.DataFrame(all_fields)
total_fields = len(field_df)

print(f'Total paren fields classified: {total_fields}')
print()
print('Overall type distribution:')
for ftype, count in field_df['field_type'].value_counts().items():
    print(f'  {ftype}: {count} ({count/total_fields*100:.1f}%)')

Total paren fields classified: 446

Overall type distribution:
  date: 128 (28.7%)
  color: 126 (28.3%)
  size: 116 (26.0%)
  rate: 55 (12.3%)
  other: 13 (2.9%)
  ms: 8 (1.8%)


In [30]:
# --- Step 5.3: Type distribution by field position ---

print('Field type by position:')
print()
for pos in sorted(field_df['position'].unique()):
    subset = field_df[field_df['position'] == pos]
    print(f'Position {pos} ({len(subset)} fields):')
    for ftype, count in subset['field_type'].value_counts().items():
        print(f'  {ftype}: {count} ({count/len(subset)*100:.1f}%)')
    print()

Field type by position:

Position 0 (136 fields):
  date: 128 (94.1%)
  rate: 5 (3.7%)
  other: 2 (1.5%)
  color: 1 (0.7%)

Position 1 (128 fields):
  size: 109 (85.2%)
  ms: 8 (6.2%)
  other: 6 (4.7%)
  rate: 3 (2.3%)
  color: 2 (1.6%)

Position 2 (127 fields):
  color: 71 (55.9%)
  rate: 45 (35.4%)
  size: 7 (5.5%)
  other: 4 (3.1%)

Position 3 (53 fields):
  color: 50 (94.3%)
  rate: 2 (3.8%)
  other: 1 (1.9%)

Position 4 (2 fields):
  color: 2 (100.0%)



In [31]:
# --- Step 5.4: Entry type-signature patterns ---

# Build a type-signature string per entry: e.g. "date|size|rate|color"
listings['field_type_sig'] = listings['paren_field_types'].apply(
    lambda types: '|'.join(types) if types else '(none)'
)

sig_counts = listings['field_type_sig'].value_counts()
print(f'Distinct type signatures: {len(sig_counts)}')
print()
print('Type signature distribution:')
for sig, count in sig_counts.head(20).items():
    print(f'  {sig}: {count} ({count/len(listings)*100:.1f}%)')

# Flag any signature containing 'other'
has_other = listings[listings['field_type_sig'].str.contains('other')]
print(f'\nEntries with "other" fields: {has_other.sum() if isinstance(has_other, pd.Series) else len(has_other)}')

Distinct type signatures: 20

Type signature distribution:
  date|size|color: 59 (43.4%)
  date|size|rate|color: 39 (28.7%)
  date|ms|color: 7 (5.1%)
  rate: 5 (3.7%)
  date|size|size|color: 5 (3.7%)
  date|other|color: 3 (2.2%)
  date|other|rate|color: 2 (1.5%)
  date|rate|color: 2 (1.5%)
  date|size|other|color: 2 (1.5%)
  date|size|rate|rate|color: 2 (1.5%)
  other: 1 (0.7%)
  color: 1 (0.7%)
  date: 1 (0.7%)
  other|size|rate|color: 1 (0.7%)
  date|size|rate|other: 1 (0.7%)
  date|other|size|color: 1 (0.7%)
  date|color|other: 1 (0.7%)
  date|rate|other: 1 (0.7%)
  date|ms|size: 1 (0.7%)
  date|color: 1 (0.7%)

Entries with "other" fields: 13


In [32]:
# --- Step 5.5: Sample inspection by type ---

for ftype in ['date', 'ms', 'size', 'rate', 'color', 'other']:
    subset = field_df[field_df['field_type'] == ftype]
    if len(subset) == 0:
        continue
    print(f'=== {ftype.upper()} ({len(subset)} fields) ===')

    # Show unique values (up to 30)
    uniques = subset['field_text'].value_counts()
    shown = 0
    for val, count in uniques.items():
        print(f'  {val!r}: {count}')
        shown += 1
        if shown >= 30:
            remaining = len(uniques) - shown
            if remaining > 0:
                print(f'  ... and {remaining} more distinct values')
            break
    print()

=== DATE (128 fields) ===
  '1861': 6
  '1854': 4
  '1850': 3
  '1863': 3
  '1828': 3
  '1832': 2
  '1847-49': 2
  '1845-48': 2
  '1859-61': 2
  '1837': 2
  '1804-06': 2
  '1799-1802': 2
  '1826': 2
  '1852': 2
  '1816': 2
  '1853-56': 2
  "1860's-70's": 1
  '1821-25': 1
  '1815-16': 1
  '1866': 1
  '1806-09': 1
  '1869': 1
  '1808-09': 1
  'April 7,1791': 1
  '1792-95,1827': 1
  '1795-1800': 1
  '1791-92': 1
  '1806-07': 1
  '1793': 1
  '1810-14': 1
  ... and 73 more distinct values

=== MS (8 fields) ===
  'Ms': 8

=== SIZE (116 fields) ===
  '30': 14
  '32': 9
  '26': 8
  '31': 8
  '26.5': 5
  '29': 5
  '29.5': 5
  'MDD': 4
  '31.5': 4
  '32.5': 3
  '27': 3
  '27.5': 3
  '28': 3
  '--': 2
  '28.5': 2
  'SL-12x3': 2
  '24.5': 2
  '25': 2
  '33': 2
  '33,YMDD': 2
  '32.5,YMDD': 2
  'DC-25,YD': 1
  '30.5': 1
  'NOR': 1
  'DO-44x37': 1
  'O-46x38': 1
  'MD': 1
  'SL-12x2.5': 1
  'SL-19x3,MDD': 1
  'DC-25-13': 1
  ... and 17 more distinct values

=== RATE (55 fields) ===
  'FREE': 15
  '

In [33]:
# --- Step 5.6: Other/unclassified fields ---

other_fields = field_df[field_df['field_type'] == 'other']
if len(other_fields) == 0:
    print('No unclassified fields -- all paren tokens matched a known type.')
else:
    print(f'=== UNCLASSIFIED FIELDS ({len(other_fields)}) ===')
    print()
    for _, frow in other_fields.iterrows():
        # Find the parent listing
        match = listings[
            listings['paren_fields'].apply(lambda pf: frow['field_text'] in pf)
        ]
        if len(match):
            first = match.iloc[0]
            print(f'  field={frow["field_text"]!r}  pos={frow["position"]}')
            print(f'    fields={first["paren_fields"]}')
            print(f'    types={first["paren_field_types"]}')
            print(f'    raw: {first["clean_text"][:120]}')
            print()

=== UNCLASSIFIED FIELDS (13) ===

  field='Red-brown,Purplish'  pos=3
    fields=['1805-08', '26.5', 'FREE', 'Red-brown,Purplish']
    types=['date', 'size', 'rate', 'other']
    raw: WASHINGTON.CITY(ornament at bottom)(1805-08;26.5;FREE;Red-brown,Purplish) 40.00

  field='STEAMBOAT'  pos=0
    fields=['STEAMBOAT']
    types=['other']
    raw: Same(STEAMBOAT)See Inland Waterways listing

  field='29,30,31'  pos=1
    fields=['1845-48', '29,30,31', 'PAID,5,10', 'Red,Orange,Red-orange']
    types=['date', 'other', 'rate', 'color']
    raw: Same(3mm letters)(1845-48;29,30,31;PAID,5,10;Red,Orange,Red-orange) . . . . 20.00

  field='1 over town mark,possibly attached rate'  pos=1
    fields=['1846', '1 over town mark,possibly attached rate', 'Red']
    types=['date', 'other', 'color']
    raw: Same(1846;1 over town mark,possibly attached rate;Red) 100.00

  field='S,10'  pos=2
    fields=['1847-49', '31', 'S,10', 'Red-orange']
    types=['date', 'size', 'other', 'color']
    raw: Same(4mm l

In [34]:
# --- Step 5.7: Positional consistency check ---

# Verify: position 0 should overwhelmingly be 'date'
if len(field_df[field_df['position'] == 0]) > 0:
    pos0 = field_df[field_df['position'] == 0]
    non_date_pos0 = pos0[pos0['field_type'] != 'date']
    print(f'Position 0 non-date fields: {len(non_date_pos0)} / {len(pos0)}')
    if len(non_date_pos0):
        for _, frow in non_date_pos0.iterrows():
            match = listings[
                listings['paren_fields'].apply(lambda pf: len(pf) > 0 and pf[0] == frow['field_text'])
            ]
            if len(match):
                first = match.iloc[0]
                print(f'  {frow["field_type"]}: {frow["field_text"]!r}')
                print(f'    raw: {first["clean_text"][:120]}')
                print()

# Last position should overwhelmingly be 'color' (for multi-field entries)
multi = field_df[field_df['field_count'] >= 3].copy()
if len(multi):
    last_pos = multi.groupby('field_count')['position'].transform('max')
    last_fields = multi[multi['position'] == last_pos]
    non_color_last = last_fields[last_fields['field_type'] != 'color']
    print(f'Last-position non-color fields (3+ field entries): {len(non_color_last)} / {len(last_fields)}')
    if len(non_color_last):
        for _, frow in non_color_last.head(10).iterrows():
            print(f'  pos={frow["position"]} type={frow["field_type"]}: {frow["field_text"]!r}')

Position 0 non-date fields: 8 / 136
  rate: 'with STEAM'
    raw: Same(with STEAM) See Inland Waterways Listing

  rate: 'STEAM[SL-33x4.5]'
    raw: Same(STEAM[SL-33x4.5]) See Inland Waterways

  other: 'STEAMBOAT'
    raw: Same(STEAMBOAT)See Inland Waterways listing

  color: 'Green'
    raw: Same(Green) . 40.00

  other: '186-'
    raw: WASHINGTON CITY(186-;31;DUE/3[C];Black) 30.00

  rate: 'with SHIP[SL-19x5]'
    raw: Same(with SHIP[SL-19x5])See Inland Waterways

  rate: 'STEAM'
    raw: Same(STEAM)See Inland Waterways listing . .

  rate: 'with both above,SHIP[18x5]'
    raw: Same(with both above,SHIP[18x5])See Ocean Mail

Last-position non-color fields (3+ field entries): 4 / 127
  pos=3 type=other: 'Red-brown,Purplish'
  pos=2 type=other: 'Olive green'
  pos=2 type=other: 'Bright Green'
  pos=2 type=size: '--'


## 5.8 Step 5 Summary

Field-type classification is content-signal-based, with no reliance on field position.
Position is used only as a validation cross-check (field 0 should be date, last field
should be color). Entries whose fields don't follow the expected positional pattern are
not reclassified -- they're flagged for review in Step 6.

At this point each listing row carries:
- **Structural segments** (Step 2): `seg_head`, `seg_paren`, `seg_tail`
- **Paren fields** (Step 3.1): `paren_fields`, `paren_field_count`
- **Tail parts** (Step 3.2-3.3): `tail_annotation`, `tail_valuation`, `valuation_tiers`
- **Head parts** (Step 4): `head_first_of_town`, `head_rel_type`, `head_name_body`, `head_annotations`
- **Field types** (Step 5): `paren_field_types`, `field_type_sig`
- **Section context**: `Default Shape`, `Page`, `Images Above`, `Institutional Ownership`

In [35]:
# --- Step 5.8: Summary ---

total = len(listings)
has_fields = listings[listings['paren_field_count'] > 0]
total_fields = field_df.shape[0]

print(f'Step 5: Paren Field-Type Classification')
print(f'  Listings with paren fields: {len(has_fields)} / {total}')
print(f'  Total fields classified: {total_fields}')
print()
print(f'  Type counts:')
for ftype, count in field_df['field_type'].value_counts().items():
    print(f'    {ftype}: {count} ({count/total_fields*100:.1f}%)')
print()
print(f'  Top type signatures:')
for sig, count in sig_counts.head(5).items():
    print(f'    {sig}: {count}')
print()
other_count = (field_df['field_type'] == 'other').sum()
print(f'  Unclassified ("other"): {other_count} ({other_count/total_fields*100:.1f}%)')

Step 5: Paren Field-Type Classification
  Listings with paren fields: 136 / 136
  Total fields classified: 446

  Type counts:
    date: 128 (28.7%)
    color: 126 (28.3%)
    size: 116 (26.0%)
    rate: 55 (12.3%)
    other: 13 (2.9%)
    ms: 8 (1.8%)

  Top type signatures:
    date|size|color: 59
    date|size|rate|color: 39
    date|ms|color: 7
    rate: 5
    date|size|size|color: 5

  Unclassified ("other"): 13 (2.9%)


# Step 6: Field-Level Sub-Parsing
 
Each classified paren field from Step 5 is decomposed into atomic components.
This is purely mechanical text decomposition — no cross-record logic, no
domain-entity assembly.
 
### Sub-parsers
 
| Type | Outputs |
|------|---------|
| `date` | month, day, year_start, year_end, granularity (DAY/MONTH/YEAR/DECADE/RANGE), is_circa |
| `size` | shape_code, dim1, dim2, dateformat, is_irregular, qualifier |
| `rate` | list of rate tokens, each with: keyword, amount, bracket_desc, is_manuscript |
| `color` | list of individual color names |
| `ms` | (no sub-parsing needed — flag consumed at assembly time) |
| `other` | attempted reclassification; survivors flagged for review |

In [36]:
# --- Step 6.1: Date sub-parser ---

MONTH_MAP = {
    'jan': 1, 'january': 1,
    'feb': 2, 'february': 2,
    'mar': 3, 'march': 3,
    'apr': 4, 'april': 4,
    'may': 5,
    'jun': 6, 'june': 6,
    'jul': 7, 'july': 7,
    'aug': 8, 'august': 8,
    'sep': 9, 'sept': 9, 'september': 9,
    'oct': 10, 'october': 10,
    'nov': 11, 'november': 11,
    'dec': 12, 'december': 12,
}

# Full date: Month day,year  e.g. "April 8,1800", "Oct.22,1803", "Feb. 2, 1818"
FULL_DATE_RE = re.compile(
    r'(' + MONTHS_PAT + r')\.?\s*'
    r'(\d{1,2})\s*,\s*'
    r'(\d{4})',
    re.IGNORECASE
)

# Month + year (no day): "April 1800"
MONTH_YEAR_RE = re.compile(
    r'(' + MONTHS_PAT + r')\.?\s+'
    r'(\d{4})',
    re.IGNORECASE
)

# Year range: "1850-53", "1842-52", "1850-1853"
YEAR_RANGE_RE = re.compile(
    r'c?(\d{4})\s*[-]\s*(\d{2,4})'
)

# Decade: "1850's", "1860's"
DECADE_RE = re.compile(r"c?(\d{4})'s", re.IGNORECASE)

# Bare year: "1852", possibly with circa prefix "c1840"
BARE_YEAR_RE = re.compile(r'c?(\d{4})$')

# Circa prefix
CIRCA_RE = re.compile(r'^c\d', re.IGNORECASE)


def parse_date_field(text):
    """Decompose a date-classified paren field into structured components."""
    t = text.strip()
    is_circa = bool(CIRCA_RE.match(t))

    # 1. Full date: Month day, year
    m = FULL_DATE_RE.search(t)
    if m:
        month_str = m.group(1).lower().rstrip('.')
        month = MONTH_MAP.get(month_str)
        day = int(m.group(2))
        year = int(m.group(3))
        return {
            'date_month': month,
            'date_day': day,
            'date_year_start': year,
            'date_year_end': year,
            'date_granularity': 'DAY',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 2. Decade: 1850's
    m = DECADE_RE.search(t)
    if m:
        base = int(m.group(1))
        return {
            'date_month': None,
            'date_day': None,
            'date_year_start': base,
            'date_year_end': base + 9,
            'date_granularity': 'DECADE',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 3. Year range: 1850-53
    m = YEAR_RANGE_RE.search(t)
    if m:
        y1 = int(m.group(1))
        y2_str = m.group(2)
        if len(y2_str) == 2:
            y2 = int(str(y1)[:2] + y2_str)
        else:
            y2 = int(y2_str)
        return {
            'date_month': None,
            'date_day': None,
            'date_year_start': y1,
            'date_year_end': y2,
            'date_granularity': 'RANGE',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 4. Month + year (no day)
    m = MONTH_YEAR_RE.search(t)
    if m:
        month_str = m.group(1).lower().rstrip('.')
        month = MONTH_MAP.get(month_str)
        year = int(m.group(2))
        return {
            'date_month': month,
            'date_day': None,
            'date_year_start': year,
            'date_year_end': year,
            'date_granularity': 'MONTH',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    # 5. Bare year
    m = BARE_YEAR_RE.search(t.lstrip('c'))
    if m:
        year = int(m.group(1))
        return {
            'date_month': None,
            'date_day': None,
            'date_year_start': year,
            'date_year_end': year,
            'date_granularity': 'YEAR',
            'date_is_circa': is_circa,
            'date_raw': t,
            'date_error': None,
        }

    return {
        'date_month': None, 'date_day': None,
        'date_year_start': None, 'date_year_end': None,
        'date_granularity': None, 'date_is_circa': is_circa,
        'date_raw': t,
        'date_error': f'unparsed date: {t!r}',
    }


# Self-tests
assert parse_date_field('April 8,1800') == {
    'date_month': 4, 'date_day': 8, 'date_year_start': 1800,
    'date_year_end': 1800, 'date_granularity': 'DAY',
    'date_is_circa': False, 'date_raw': 'April 8,1800', 'date_error': None,
}
assert parse_date_field("1850's")['date_granularity'] == 'DECADE'
assert parse_date_field("1850's")['date_year_end'] == 1859
assert parse_date_field('1850-53')['date_year_end'] == 1853
assert parse_date_field('1852')['date_granularity'] == 'YEAR'
assert parse_date_field('c1840')['date_is_circa'] is True
assert parse_date_field('c1840')['date_year_start'] == 1840
assert parse_date_field('Oct.22,1803')['date_month'] == 10
print('Date sub-parser self-tests passed')

Date sub-parser self-tests passed


In [37]:
# --- Step 6.2: Size sub-parser ---

# Shape codes ordered longest-first
SHAPE_CODES = ['DLDC', 'DLDO', 'DLC', 'DLO', 'Octagon', 'Box', 'Arc',
               'Pmk', 'SL', 'DC', 'DO', 'NOR', 'O', 'C']
SHAPE_CODE_SET = {s.upper() for s in SHAPE_CODES}

SIZE_DATEFORMAT_CODES = {'YMDD', 'MDD', 'YMD', 'YD', 'MD'}

# Pattern: optional "irregular" prefix, shape code, optional dash, dimensions,
#          optional comma + dateformat/qualifier
SIZE_PARSE_RE = re.compile(
    r'^(irregular\s+)?'              # optional irregular prefix
    r'(' + SHAPE_CODE_PAT + r')?'    # optional shape code
    r'[\s\-]*'                       # separator
    r'('                             # dimension group
    r'  \d+\.?\d*\s*x\s*\d+\.?\d*'  # WxH
    r'  |\d+\.?\d*'                  # single diameter
    r'  |--?'                        # dash = unknown
    r')?'
    r'(?:\s*,\s*(.+))?'             # optional suffix (dateformat, qualifier)
    r'$',
    re.IGNORECASE | re.VERBOSE
)


def parse_size_field(text):
    """Decompose a size-classified paren field into components."""
    t = text.strip()

    # Catch bare dashes
    if t in ('-', '--'):
        return {
            'size_shape_code': None, 'size_dim1': None, 'size_dim2': None,
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t, 'size_error': None,
        }

    m = SIZE_PARSE_RE.match(t)
    if not m:
        return {
            'size_shape_code': None, 'size_dim1': None, 'size_dim2': None,
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t,
            'size_error': f'unparsed size: {t!r}',
        }

    irregular_prefix = m.group(1)
    shape_raw = m.group(2)
    dim_raw = m.group(3)
    suffix_raw = m.group(4)

    is_irregular = bool(irregular_prefix)
    shape_code = shape_raw.upper() if shape_raw else None

    # Dimensions
    dim1, dim2 = None, None
    if dim_raw and dim_raw not in ('-', '--'):
        if 'x' in dim_raw.lower():
            parts = re.split(r'\s*x\s*', dim_raw, flags=re.IGNORECASE)
            dim1 = float(parts[0]) if parts[0] else None
            dim2 = float(parts[1]) if len(parts) > 1 and parts[1] else None
        else:
            dim1 = float(dim_raw)

    # Suffix: dateformat code, NOR, or free-text qualifier
    dateformat = None
    qualifier = None
    if suffix_raw:
        # May contain multiple tokens: "YD", "MDD", "NOR", "YMDD below"
        suffix_upper = suffix_raw.strip().upper()
        # Check if it starts with a known dateformat code
        for code in sorted(SIZE_DATEFORMAT_CODES, key=len, reverse=True):
            if suffix_upper.startswith(code):
                dateformat = code
                remainder = suffix_raw.strip()[len(code):].strip()
                if remainder:
                    qualifier = remainder
                break
        else:
            if suffix_upper == 'NOR':
                qualifier = 'NOR'
            else:
                qualifier = suffix_raw.strip()

    return {
        'size_shape_code': shape_code,
        'size_dim1': dim1,
        'size_dim2': dim2,
        'size_dateformat': dateformat,
        'size_is_irregular': is_irregular,
        'size_qualifier': qualifier,
        'size_raw': t,
        'size_error': None,
    }


# Self-tests
r = parse_size_field('SL-16.5x5,MDD')
assert r['size_shape_code'] == 'SL'
assert r['size_dim1'] == 16.5
assert r['size_dim2'] == 5.0
assert r['size_dateformat'] == 'MDD'

r = parse_size_field('DC-25,YD')
assert r['size_shape_code'] == 'DC'
assert r['size_dim1'] == 25.0
assert r['size_dateformat'] == 'YD'

r = parse_size_field('32')
assert r['size_shape_code'] is None
assert r['size_dim1'] == 32.0
assert r['size_dateformat'] is None

r = parse_size_field('--,YD')
assert r['size_dim1'] is None
assert r['size_dateformat'] == 'YD'

r = parse_size_field('DO-30x24')
assert r['size_shape_code'] == 'DO'
assert r['size_dim1'] == 30.0
assert r['size_dim2'] == 24.0

r = parse_size_field('SL-45x4,YMDD below')
assert r['size_shape_code'] == 'SL'
assert r['size_dateformat'] == 'YMDD'
assert r['size_qualifier'] == 'below'

print('Size sub-parser self-tests passed')

Size sub-parser self-tests passed


## Step 6.3: Rate sub-parser

A rate paren field contains one or more rate tokens separated by commas.

Commas INSIDE brackets are not separators, e.g. "PAID/3[C]" is one token.

Each token is a distinct rate marking or rate value observed on the postmark.

Token anatomy examples:
  * PAID          -> keyword=PAID, amount=None, bracket=None

  * FREE          -> keyword=FREE, amount=None, bracket=None
 
  * PAID 3        -> keyword=PAID, amount=3, bracket=None
 
  * PAID/3[C]     -> keyword=PAID, amount=3, bracket=C
 
  * 5[C]          -> keyword=None, amount=5, bracket=C
 
  * 25[ms]        -> keyword=None, amount=25, bracket=ms (manuscript rate)
 
  * P.M.Free      -> keyword=PM_FREE, amount=None, bracket=None
 
  * Geo.Fisher P.M.frank -> keyword=PM_FRANK, amount=None, bracket=None
 
  * STEAM         -> keyword=STEAM, amount=None, bracket=None
 
  * DUE 3         -> keyword=DUE, amount=3, bracket=None
 
  * with 24       -> keyword=WITH_ADHESIVE, amount=24, bracket=None
 
  * X             -> keyword=None, amount=None, bracket=None, roman='X'
 
  * negative 5[C] -> keyword=None, amount=5, bracket=C, impression=Negative
  

In [38]:

def split_rate_tokens(field_text):
    """Split rate field on commas, respecting brackets.
    Returns list of raw token strings."""
    tokens = []
    current = []
    depth = 0
    for ch in field_text:
        if ch == '[':
            depth += 1
            current.append(ch)
        elif ch == ']':
            depth -= 1
            current.append(ch)
        elif ch == ',' and depth == 0:
            tokens.append(''.join(current).strip())
            current = []
        else:
            current.append(ch)
    if current:
        tokens.append(''.join(current).strip())
    return [t for t in tokens if t]


RATE_AMOUNT_RE = re.compile(
    r'(\d+(?:[/-]\d+(?:/\d+)?)?)'  # amount: "3", "12-1/2", "3/CENTS"
)

RATE_BRACKET_RE = re.compile(r'\[([^\]]+)\]')

RATE_KEYWORD_RE = re.compile(
    r'\b(PAID|FREE|STEAM|DUE)\b', re.IGNORECASE
)

PM_RE = re.compile(r'P\.?M\.?\s*(Free|frank)', re.IGNORECASE)

NEGATIVE_RE = re.compile(r'^negative\s+', re.IGNORECASE)

ROMAN_RE = re.compile(r'^[IVXLCDM]+$')


def parse_rate_token(tok):
    """Parse a single rate token into structured components."""
    t = tok.strip()

    result = {
        'rate_keyword': None,
        'rate_amount_raw': None,
        'rate_bracket': None,
        'rate_is_manuscript': False,
        'rate_impression': None,
        'rate_raw': t,
    }

    # Check for negative impression prefix
    neg_m = NEGATIVE_RE.match(t)
    if neg_m:
        result['rate_impression'] = 'Negative'
        t = t[neg_m.end():]

    # P.M. notation
    pm_m = PM_RE.search(t)
    if pm_m:
        pm_type = pm_m.group(1).lower()
        if pm_type == 'free':
            result['rate_keyword'] = 'PM_FREE'
        else:
            result['rate_keyword'] = 'PM_FRANK'
        # May have trailing rate: "P.M.Free-Paid 10"
        remainder = t[pm_m.end():].strip().lstrip('-')
        if remainder:
            kw_m = RATE_KEYWORD_RE.search(remainder)
            if kw_m:
                result['rate_keyword'] = 'PM_FREE'  # compound; keep PM_FREE
                amt_after = remainder[kw_m.end():].strip()
                if amt_after:
                    amt_m = RATE_AMOUNT_RE.search(amt_after)
                    if amt_m:
                        result['rate_amount_raw'] = amt_m.group(1)
        return result

    # Bracket: [ms], [C], [F], [box], etc.
    br_m = RATE_BRACKET_RE.search(t)
    if br_m:
        bracket_val = br_m.group(1).strip()
        if bracket_val.lower() == 'ms':
            result['rate_is_manuscript'] = True
        else:
            result['rate_bracket'] = bracket_val

    # Keyword: PAID, FREE, STEAM, DUE
    kw_m = RATE_KEYWORD_RE.search(t)
    if kw_m:
        result['rate_keyword'] = kw_m.group(1).upper()

    # "with NN" = adhesive
    with_m = re.search(r'\bwith\s+(\d+)', t, re.IGNORECASE)
    if with_m:
        result['rate_keyword'] = 'WITH_ADHESIVE'
        result['rate_amount_raw'] = with_m.group(1)
        return result

    # Amount: first numeric sequence not inside a keyword or bracket-only context
    # Strip bracket content and keyword to find rate amount
    t_stripped = RATE_BRACKET_RE.sub('', t)
    t_stripped = RATE_KEYWORD_RE.sub('', t_stripped)
    t_stripped = PM_RE.sub('', t_stripped)
    t_stripped = t_stripped.replace('/', ' ').strip()
    amt_m = RATE_AMOUNT_RE.search(t_stripped)
    if amt_m:
        result['rate_amount_raw'] = amt_m.group(1)

    # Roman numeral check (V, X, etc.) when no other signal
    if (result['rate_keyword'] is None and result['rate_amount_raw'] is None
            and not result['rate_is_manuscript']):
        clean = RATE_BRACKET_RE.sub('', t).strip()
        if ROMAN_RE.match(clean):
            result['rate_amount_raw'] = clean

    return result


def parse_rate_field(text):
    """Decompose a rate-classified paren field into a list of parsed tokens."""
    tokens = split_rate_tokens(text)
    return [parse_rate_token(t) for t in tokens]


# Self-tests
assert split_rate_tokens('PAID/3[C],FREE') == ['PAID/3[C]', 'FREE']
assert split_rate_tokens('5,10,PAID') == ['5', '10', 'PAID']
assert split_rate_tokens('25,12-1/2[ms]Black') == ['25', '12-1/2[ms]Black']

r = parse_rate_token('PAID/3[C]')
assert r['rate_keyword'] == 'PAID'
assert r['rate_amount_raw'] == '3'
assert r['rate_bracket'] == 'C'

r = parse_rate_token('25[ms]')
assert r['rate_amount_raw'] == '25'
assert r['rate_is_manuscript'] is True

r = parse_rate_token('FREE')
assert r['rate_keyword'] == 'FREE'
assert r['rate_amount_raw'] is None

r = parse_rate_token('P.M.Free')
assert r['rate_keyword'] == 'PM_FREE'

r = parse_rate_token('with 24')
assert r['rate_keyword'] == 'WITH_ADHESIVE'
assert r['rate_amount_raw'] == '24'

r = parse_rate_token('negative 5[C]')
assert r['rate_impression'] == 'Negative'
assert r['rate_amount_raw'] == '5'
assert r['rate_bracket'] == 'C'

r = parse_rate_token('STEAM')
assert r['rate_keyword'] == 'STEAM'

print('Rate sub-parser self-tests passed')

Rate sub-parser self-tests passed


In [39]:
# --- Step 6.4: Color sub-parser ---

def parse_color_field(text):
    """Split a color field into individual normalized color names (UPPER)."""
    tokens = [t.strip().upper() for t in text.split(',') if t.strip()]
    return tokens


# Self-tests
assert parse_color_field('Black') == ['BLACK']
assert parse_color_field('Red,Blue,Black') == ['RED', 'BLUE', 'BLACK']
assert parse_color_field('Olive-Yellow') == ['OLIVE-YELLOW']
print('Color sub-parser self-tests passed')

Color sub-parser self-tests passed


In [40]:
# --- Step 6.5: Other-field triage ---
#
# Attempt to reclassify "other" fields that slipped through Step 5.
# Known reclassification patterns:
#   "185-", "186-", "183-51" -> truncated dates (date-like)
#   "DC--", "DLC--", "arc--" -> size with unknown dimension
#   "5,10", "5,V,10", "V,X"  -> rate (bare amounts/roman without keywords)
#   "12-1/2"                  -> rate (fractional cent amount)
#   "Double 50"               -> rate (descriptive)
#   "large 5,10"              -> rate (with qualifier)
#   "fancy shaded 10"         -> rate (with qualifier)
#   "irregular 34"            -> size (with irregular prefix)
#   "30,32"                   -> size (multiple dimensions)
#   "Red,Purple,Blue,Brownish" -> color (unknown color term "Brownish")

TRUNCATED_DATE_RE = re.compile(r'^\d{3}-\d{0,2}$')
SIZE_WITH_DASH_RE = re.compile(
    r'^(?:' + SHAPE_CODE_PAT + r'|arc)[\s\-]*-{1,2}$', re.IGNORECASE
)
BARE_RATE_RE = re.compile(
    r'^(?:(?:large|fancy|shaded|Double|small)\s+)?'
    r'(?:\d+(?:-\d+(?:/\d+)?)?|[IVXLCDM]+)'
    r'(?:\s*,\s*(?:\d+(?:-\d+(?:/\d+)?)?|[IVXLCDM]+))*$'
)
IRREGULAR_SIZE_RE = re.compile(r'^irregular\s+\d', re.IGNORECASE)
MULTI_DIM_RE = re.compile(r'^\d{2,3}\s*,\s*\d{2,3}$')


def triage_other_field(text):
    """Attempt reclassification of an 'other' field.
    Returns (new_type, parsed_result) or ('other', None) if unresolvable."""
    t = text.strip()

    # Truncated date: "185-", "186-", "183-51"
    if TRUNCATED_DATE_RE.match(t):
        # Treat as approximate date range
        prefix = t.split('-')[0]
        suffix = t.split('-')[1] if '-' in t else ''
        if len(prefix) == 3:
            decade_base = int(prefix + '0')
            if suffix and suffix.isdigit():
                year_end = int(prefix + suffix) if len(suffix) == 1 else int('1' + suffix) if len(suffix) == 2 else decade_base + 9
            else:
                year_end = decade_base + 9
            return 'date', {
                'date_month': None, 'date_day': None,
                'date_year_start': decade_base,
                'date_year_end': year_end,
                'date_granularity': 'RANGE',
                'date_is_circa': False,
                'date_raw': t,
                'date_error': 'reclassified from other (truncated date)',
            }

    # Size with unknown dim: "DC--", "DLC--", "arc--"
    if SIZE_WITH_DASH_RE.match(t):
        # Extract shape code
        shape = re.match(r'^([A-Za-z]+)', t).group(1).upper()
        return 'size', {
            'size_shape_code': shape,
            'size_dim1': None, 'size_dim2': None,
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t, 'size_error': None,
        }

    # Irregular size: "irregular 34"
    if IRREGULAR_SIZE_RE.match(t):
        return 'size', parse_size_field(t)

    # Multi-dimension: "30,32"
    if MULTI_DIM_RE.match(t):
        dims = t.split(',')
        return 'size', {
            'size_shape_code': None,
            'size_dim1': float(dims[0].strip()),
            'size_dim2': float(dims[1].strip()),
            'size_dateformat': None, 'size_is_irregular': False,
            'size_qualifier': None, 'size_raw': t, 'size_error': None,
        }

    # Bare rate amounts or roman+amount combos: "5,10", "12-1/2", "V,X", "Double 50"
    if BARE_RATE_RE.match(t):
        return 'rate', parse_rate_field(t)

    # Color with unknown terms (partial match)
    tokens = [tok.strip() for tok in t.split(',') if tok.strip()]
    known_count = sum(1 for tok in tokens if is_color_token(tok))
    if known_count > 0 and known_count >= len(tokens) - 1:
        # At least one unknown term but majority are colors -> reclassify
        return 'color', [t.upper() for t in tokens]

    return 'other', None


# Self-tests
assert triage_other_field('185-')[0] == 'date'
assert triage_other_field('186-')[0] == 'date'
assert triage_other_field('DC--')[0] == 'size'
assert triage_other_field('DLC--')[0] == 'size'
assert triage_other_field('5,10')[0] == 'rate'
assert triage_other_field('12-1/2')[0] == 'rate'
assert triage_other_field('irregular 34')[0] == 'size'
assert triage_other_field('30,32')[0] == 'size'
assert triage_other_field('Red,Purple,Blue,Brownish')[0] == 'color'
assert triage_other_field('Double 50')[0] == 'rate'
print('Other-field triage self-tests passed')

Other-field triage self-tests passed


In [41]:
# --- Step 6.6: Apply sub-parsers to all classified paren fields ---

def subparse_fields(row):
    """Apply the appropriate sub-parser to each paren field based on its type.
    Returns parallel lists: parsed_dates, parsed_sizes, parsed_rates, parsed_colors,
    plus is_manuscript flag and other_fields list."""
    fields = row['paren_fields']
    types = row['paren_field_types']

    parsed_dates = []
    parsed_sizes = []
    parsed_rates = []     # list of lists (each rate field -> list of rate tokens)
    parsed_colors = []    # flat list of color names across all color fields
    is_manuscript = False
    other_fields = []
    reclassified = []     # track reclassification for audit

    for i, (field, ftype) in enumerate(zip(fields, types)):
        if ftype == 'ms':
            is_manuscript = True

        elif ftype == 'date':
            parsed_dates.append(parse_date_field(field))

        elif ftype == 'size':
            parsed_sizes.append(parse_size_field(field))

        elif ftype == 'rate':
            parsed_rates.append(parse_rate_field(field))

        elif ftype == 'color':
            parsed_colors.extend(parse_color_field(field))

        elif ftype == 'other':
            new_type, parsed = triage_other_field(field)
            if new_type != 'other':
                reclassified.append({
                    'position': i, 'original_type': 'other',
                    'new_type': new_type, 'field': field,
                })
                if new_type == 'date':
                    parsed_dates.append(parsed)
                elif new_type == 'size':
                    parsed_sizes.append(parsed)
                elif new_type == 'rate':
                    if isinstance(parsed, list):
                        parsed_rates.append(parsed)
                    else:
                        parsed_rates.append([parsed])
                elif new_type == 'color':
                    parsed_colors.extend(parsed)
            else:
                other_fields.append(field)

    return pd.Series({
        'parsed_dates': parsed_dates,
        'parsed_sizes': parsed_sizes,
        'parsed_rates': parsed_rates,
        'parsed_colors': parsed_colors,
        'is_manuscript': is_manuscript,
        'other_fields': other_fields,
        'reclassified_fields': reclassified,
    })


parsed = listings.apply(subparse_fields, axis=1)
listings = pd.concat([listings, parsed], axis=1)

# Summary counts
print('Step 6: Field-level sub-parsing applied')
print(f'  Listings processed: {len(listings)}')
print(f'  Manuscript entries: {listings["is_manuscript"].sum()}')
print(f'  Entries with dates: {listings["parsed_dates"].apply(len).gt(0).sum()}')
print(f'  Entries with sizes: {listings["parsed_sizes"].apply(len).gt(0).sum()}')
print(f'  Entries with rates: {listings["parsed_rates"].apply(len).gt(0).sum()}')
print(f'  Entries with colors: {listings["parsed_colors"].apply(len).gt(0).sum()}')
total_reclassified = listings['reclassified_fields'].apply(len).sum()
print(f'  Other fields reclassified: {total_reclassified}')
remaining_other = listings['other_fields'].apply(len).sum()
print(f'  Remaining unresolved other fields: {remaining_other}')

Step 6: Field-level sub-parsing applied
  Listings processed: 136
  Manuscript entries: 8
  Entries with dates: 129
  Entries with sizes: 112
  Entries with rates: 53
  Entries with colors: 127
  Other fields reclassified: 5
  Remaining unresolved other fields: 8


In [42]:
# --- Step 6.7: Audit sub-parsing errors ---

# Date parse errors
date_errors = []
for idx, row in listings.iterrows():
    for d in row['parsed_dates']:
        if d.get('date_error') and 'reclassified' not in str(d.get('date_error', '')):
            date_errors.append((idx, d['date_raw'], d['date_error']))

print(f'Date parse errors: {len(date_errors)}')
for idx, raw, err in date_errors[:10]:
    print(f'  [{idx}] {raw!r}: {err}')
print()

# Size parse errors
size_errors = []
for idx, row in listings.iterrows():
    for s in row['parsed_sizes']:
        if s.get('size_error'):
            size_errors.append((idx, s['size_raw'], s['size_error']))

print(f'Size parse errors: {len(size_errors)}')
for idx, raw, err in size_errors[:10]:
    print(f'  [{idx}] {raw!r}: {err}')
print()

# Reclassification audit
reclass_all = []
for idx, row in listings.iterrows():
    for rc in row['reclassified_fields']:
        reclass_all.append((idx, rc['field'], rc['new_type']))

print(f'Reclassified other fields: {len(reclass_all)}')
for idx, field, new_type in reclass_all[:15]:
    print(f'  [{idx}] {field!r} -> {new_type}')
print()

# Remaining unresolved
unresolved = []
for idx, row in listings.iterrows():
    for f in row['other_fields']:
        unresolved.append((idx, f, row['clean_text'][:100]))

print(f'Unresolved other fields: {len(unresolved)}')
for idx, field, raw in unresolved:
    print(f'  [{idx}] {field!r}')
    print(f'    raw: {raw}')
    print()

Date parse errors: 0

Size parse errors: 8
  [65] 'YMDD': unparsed size: 'YMDD'
  [73] 'MDD': unparsed size: 'MDD'
  [74] 'MDD': unparsed size: 'MDD'
  [77] 'DC-25-13': unparsed size: 'DC-25-13'
  [83] 'MDD': unparsed size: 'MDD'
  [85] 'MD': unparsed size: 'MD'
  [86] 'MDD': unparsed size: 'MDD'
  [112] 'dotted C-28.5': unparsed size: 'dotted C-28.5'

Reclassified other fields: 5
  [23] 'Red-brown,Purplish' -> color
  [34] '29,30,31' -> rate
  [39] '29-30' -> rate
  [42] '186-' -> date
  [99] '31,32' -> size

Unresolved other fields: 8
  [28] 'STEAMBOAT'
    raw: Same(STEAMBOAT)See Inland Waterways listing

  [36] '1 over town mark,possibly attached rate'
    raw: Same(1846;1 over town mark,possibly attached rate;Red) 100.00

  [37] 'S,10'
    raw: Same(4mm letters)(1847-49;31;S,10;Red-orange) 20.00

  [46] '1 ct'
    raw: Same/D.C.(1851;31;1 ct;Black)circular rate . . 100.00

  [59] '32,32.5'
    raw: Same(1850;32,32.5;Red,Black) 20.00

  [74] '24mm'
    raw: Same(1868;24mm;MDD;Black

In [43]:
# --- Step 6.8: Parsed value distributions ---

# Date granularity
all_dates = [d for dlist in listings['parsed_dates'] for d in dlist]
if all_dates:
    gran_counts = pd.Series([d['date_granularity'] for d in all_dates]).value_counts()
    print('Date granularity distribution:')
    for g, c in gran_counts.items():
        print(f'  {g}: {c}')
    print()

# Size shape codes
all_sizes = [s for slist in listings['parsed_sizes'] for s in slist]
if all_sizes:
    shape_counts = pd.Series([s['size_shape_code'] or '(bare dimension)' for s in all_sizes]).value_counts()
    print('Size shape code distribution:')
    for s, c in shape_counts.items():
        print(f'  {s}: {c}')
    print()

    # Dateformat on size
    df_counts = pd.Series([s['size_dateformat'] or '(none)' for s in all_sizes]).value_counts()
    print('Size dateformat distribution:')
    for d, c in df_counts.items():
        print(f'  {d}: {c}')
    print()

# Rate keywords
all_rate_tokens = [t for rlist in listings['parsed_rates'] for toks in rlist for t in toks]
if all_rate_tokens:
    kw_counts = pd.Series([t['rate_keyword'] or '(bare amount)' for t in all_rate_tokens]).value_counts()
    print('Rate keyword distribution:')
    for k, c in kw_counts.items():
        print(f'  {k}: {c}')
    print()

    ms_rate = sum(1 for t in all_rate_tokens if t['rate_is_manuscript'])
    print(f'Manuscript rate tokens: {ms_rate}')
    neg_rate = sum(1 for t in all_rate_tokens if t['rate_impression'] == 'Negative')
    print(f'Negative-impression rate tokens: {neg_rate}')
    print()

# Color names
all_colors = [c for clist in listings['parsed_colors'] for c in clist]
if all_colors:
    color_counts = pd.Series(all_colors).value_counts()
    print('Color distribution:')
    for c, n in color_counts.items():
        print(f'  {c}: {n}')
    print()

Date granularity distribution:
  RANGE: 76
  YEAR: 49
  DAY: 3
  DECADE: 1

Size shape code distribution:
  (bare dimension): 104
  DC: 4
  SL: 4
  DO: 2
  O: 2
  NOR: 1

Size dateformat distribution:
  (none): 105
  YMDD: 9
  YD: 2
  MDD: 1

Rate keyword distribution:
  FREE: 27
  (bare amount): 26
  PAID: 25
  DUE: 5
  STEAM: 3

Manuscript rate tokens: 0
Negative-impression rate tokens: 0

Color distribution:
  BLACK: 71
  RED: 32
  RED-BROWN: 20
  RED-ORANGE: 15
  BLUE: 10
  BLACK-BROWN: 7
  BROWN: 5
  ORANGE: 5
  PURPLE: 2
  GREEN: 2
  OLIVE-GREEN: 2
  GRAY-BROWN: 1
  PURPLISH: 1
  OLIVE: 1
  ---: 1
  PINK: 1



## 6.9 Step 6 Summary
 
Step 6 decomposes each typed paren field into atomic components, adding seven
new columns to the `listings` DataFrame:
 
- **parsed_dates** -- list of date dicts (month, day, year_start, year_end, granularity, is_circa)
- **parsed_sizes** -- list of size dicts (shape_code, dim1, dim2, dateformat, is_irregular, qualifier)
- **parsed_rates** -- list of rate token lists, each token with (keyword, amount_raw, bracket, is_manuscript, impression)
- **parsed_colors** -- flat list of individual color names
- **is_manuscript** -- boolean (from Ms field)
- **other_fields** -- list of any unresolved fields
- **reclassified_fields** -- audit trail of other-to-type reclassifications
 
No cross-record logic. No domain entity assembly. That begins in Step 7
(relationship resolution).

In [44]:
# --- Step 6.9: Summary ---

total = len(listings)
print('Step 6: Field-Level Sub-Parsing')
print(f'  Listings processed: {total}')
print()
print(f'  Dates parsed:        {sum(len(d) for d in listings["parsed_dates"])}')
print(f'  Sizes parsed:        {sum(len(s) for s in listings["parsed_sizes"])}')
print(f'  Rate fields parsed:  {sum(len(r) for r in listings["parsed_rates"])}')
print(f'    Rate tokens total: {sum(len(t) for rlist in listings["parsed_rates"] for t in rlist)}')
print(f'  Colors extracted:    {sum(len(c) for c in listings["parsed_colors"])}')
print(f'  Manuscript entries:  {listings["is_manuscript"].sum()}')
print()
print(f'  Reclassified others: {sum(len(r) for r in listings["reclassified_fields"])}')
print(f'  Unresolved others:   {sum(len(o) for o in listings["other_fields"])}')

Step 6: Field-Level Sub-Parsing
  Listings processed: 136

  Dates parsed:        129
  Sizes parsed:        117
  Rate fields parsed:  57
    Rate tokens total: 86
  Colors extracted:    176
  Manuscript entries:  8

  Reclassified others: 5
  Unresolved others:   8


# Step 7: Relationship Resolution

Walks listings in catalog order, resolves `Same`/`(L)`/`(E)` inheritance so
every record has a fully resolved inscription text and town root.

First step with cross-record dependencies.

### Rules

1. **Independent entry** (`head_rel_type` is null): inscription = `head_name_body`;
   town root = everything before the first `/` (or the entire string if no `/`).
   Becomes the active parent for subsequent rel entries.

2. **`Same` with no name body**: inherit parent inscription and town.

3. **`Same` with name body**: different device, same town. Name body should start
   with `/` (state abbreviation variant). Resolved inscription = parent town root +
   child name body. Flagged if name body does not start with `/`.

4. **`(L)` or `(E)`**: later / earlier observation of same device. Inherit parent
   inscription and town.

Parent resolution: walk backward unconditionally to nearest preceding independent
entry. No section-boundary guard. Cross-section inheritance is flagged as a warning.

### Outputs

- **parent_idx** (nullable int) -- DataFrame index of resolved parent. Null for independent entries.
- **resolved_inscription** (str) -- Inscription text on the physical device.
- **resolved_town** (str) -- Town root for PostOffice grouping (pre-`/` text, unstripped).
- **s7_warnings** (list of str) -- Confidence-lowering flags for this record.

### Warning codes

- `orphan_rel` -- rel entry with no preceding independent entry found
- `independent_no_name` -- independent entry has null name body
- `same_name_body_no_slash` -- Same entry has name body not starting with `/`
- `cross_section_parent` -- parent is in a different Default Shape section

In [45]:
# --- Step 7.1: Relationship resolution ---

def extract_town_root(inscription):
    """Town root = everything before the first '/', or whole string if no '/'."""
    if '/' in inscription:
        return inscription.split('/')[0]
    return inscription

def resolve_relationships(listings_df):
    """Walk listings in catalog order, resolve inheritance.
    
    Modifies listings_df in place, adding:
      parent_idx, resolved_inscription, resolved_town, s7_warnings
    """
    n = len(listings_df)
    parent_idx = [None] * n
    resolved_inscription = [None] * n
    resolved_town = [None] * n
    s7_warnings = [[] for _ in range(n)]

    # Track the most recent independent entry by iteration position
    current_parent_pos = None

    for pos in range(n):
        row = listings_df.iloc[pos]
        warnings = []

        if pd.isna(row['head_rel_type']) or row['head_rel_type'] is None:
            # --- Independent entry ---
            inscription = row['head_name_body']
            if inscription is None:
                warnings.append('independent_no_name')
                inscription = ''

            town = extract_town_root(inscription)

            parent_idx[pos] = None
            resolved_inscription[pos] = inscription
            resolved_town[pos] = town
            current_parent_pos = pos

        else:
            # --- Relationship entry ---
            if current_parent_pos is None:
                warnings.append('orphan_rel')
                # Best-effort: use own name body if any
                fallback = row['head_name_body'] or ''
                parent_idx[pos] = None
                resolved_inscription[pos] = fallback
                resolved_town[pos] = extract_town_root(fallback) if fallback else ''
            else:
                parent_idx[pos] = listings_df.index[current_parent_pos]
                p_inscription = resolved_inscription[current_parent_pos]
                p_town = resolved_town[current_parent_pos]

                rel = row['head_rel_type']
                name_body = row['head_name_body']

                if rel == 'Same' and pd.notna(name_body):
                    # Different device, same town: reconstruct inscription
                    if not name_body.startswith('/'):
                        warnings.append('same_name_body_no_slash')
                    resolved_inscription[pos] = p_town + name_body
                    resolved_town[pos] = p_town
                else:
                    # Same device (Same w/o name, (L), (E)): inherit
                    resolved_inscription[pos] = p_inscription
                    resolved_town[pos] = p_town

                # Cross-section check
                parent_row = listings_df.iloc[current_parent_pos]
                if row.get('Default Shape') != parent_row.get('Default Shape'):
                    warnings.append('cross_section_parent')

        s7_warnings[pos] = warnings

    listings_df['parent_idx'] = parent_idx
    listings_df['resolved_inscription'] = resolved_inscription
    listings_df['resolved_town'] = resolved_town
    listings_df['s7_warnings'] = s7_warnings
    return listings_df

listings = resolve_relationships(listings)

print(f'Step 7: Relationship resolution applied to {len(listings)} listings')
print(f'  Independent entries: {listings["parent_idx"].isna().sum()}')
print(f'  Resolved from parent: {listings["parent_idx"].notna().sum()}')
print(f'  Distinct resolved towns: {listings["resolved_town"].nunique()}')

# --- Step 7.1b: Attribute inheritance ---
#
# For child records (those with parent_idx), inherit parsed attributes
# from the parent when the child's own paren body is silent on them.
#
# Rules:
#   is_manuscript : inherit if child has no 'ms' or 'size' in paren_field_types
#   parsed_colors : inherit if child's parsed_colors is empty
#   parsed_sizes  : inherit if child's parsed_sizes is empty
#
# Walk in catalog order so that transitive chains (unlikely but safe)
# see already-inherited parent values.

inherited_ms_count = 0
inherited_color_count = 0
inherited_size_count = 0

for pos in range(len(listings)):
    row = listings.iloc[pos]
    if pd.isna(row['parent_idx']):
        continue

    parent = listings.loc[row['parent_idx']]
    child_types = set(row['paren_field_types'])

    # is_manuscript
    if 'ms' not in child_types and 'size' not in child_types:
        if parent['is_manuscript'] != row['is_manuscript']:
            listings.iat[pos, listings.columns.get_loc('is_manuscript')] = parent['is_manuscript']
            inherited_ms_count += 1

    # parsed_colors
    if not row['parsed_colors']:
        if parent['parsed_colors']:
            listings.iat[pos, listings.columns.get_loc('parsed_colors')] = parent['parsed_colors'].copy()
            inherited_color_count += 1

    # parsed_sizes
    if not row['parsed_sizes']:
        if parent['parsed_sizes']:
            listings.iat[pos, listings.columns.get_loc('parsed_sizes')] = parent['parsed_sizes'].copy()
            inherited_size_count += 1

print()
print('Step 7.1b: Attribute inheritance')
print(f'  is_manuscript inherited:  {inherited_ms_count}')
print(f'  parsed_colors inherited:  {inherited_color_count}')
print(f'  parsed_sizes inherited:   {inherited_size_count}')


Step 7: Relationship resolution applied to 136 listings
  Independent entries: 62
  Resolved from parent: 74
  Distinct resolved towns: 45

Step 7.1b: Attribute inheritance
  is_manuscript inherited:  1
  parsed_colors inherited:  8
  parsed_sizes inherited:   14


In [46]:
# --- Step 7.2: Resolution examples by rel_type ---

for rt in [None, 'Same', '(L)', '(E)']:
    if rt is None:
        subset = listings[listings['head_rel_type'].isna()]
        label = '(independent)'
    else:
        subset = listings[listings['head_rel_type'] == rt]
        label = rt
    print(f'=== rel_type={label} ({len(subset)} entries) ===')
    for _, row in subset.head(6).iterrows():
        parent_info = ''
        if pd.notna(row['parent_idx']):
            p = listings.loc[row['parent_idx']]
            parent_info = f'  parent_head={p["seg_head"]!r}'
        print(f'  seg_head={row["seg_head"]!r}')
        print(f'    -> inscription={row["resolved_inscription"]!r}  town={row["resolved_town"]!r}{parent_info}')
        if row['s7_warnings']:
            print(f'    WARNINGS: {row["s7_warnings"]}')
    print()

# Same-with-name-body examples
same_with_name = listings[
    (listings['head_rel_type'] == 'Same') & listings['head_name_body'].notna()
]
print(f'=== Same with name body ({len(same_with_name)} entries) ===')
for _, row in same_with_name.head(10).iterrows():
    if pd.notna(row['parent_idx']):
        p = listings.loc[row['parent_idx']]
        print(f'  head={row["seg_head"]!r}  name_body={row["head_name_body"]!r}')
        print(f'    parent inscription={p["resolved_inscription"]!r}  parent town={p["resolved_town"]!r}')
        print(f'    -> resolved={row["resolved_inscription"]!r}  town={row["resolved_town"]!r}')
        if row['s7_warnings']:
            print(f'    WARNINGS: {row["s7_warnings"]}')
        print()
    else:
        print(f'  head={row["seg_head"]!r}  name_body={row["head_name_body"]!r}  *** ORPHAN ***')


=== rel_type=(independent) (62 entries) ===
  seg_head='W City(Washington City)(E)'
    -> inscription='W City'  town='W City'
  seg_head='WASH.CITY(period high)'
    -> inscription='WASH.CITY'  town='WASH.CITY'
  seg_head='WASHN.CITY("N"high)'
    -> inscription='WASHN.CITY'  town='WASHN.CITY'
  seg_head='WASHINGTON.CITY(ornament at bottom)'
    -> inscription='WASHINGTON.CITY'  town='WASHINGTON.CITY'
  seg_head='CITY OF WASHINGTON'
    -> inscription='CITY OF WASHINGTON'  town='CITY OF WASHINGTON'
  seg_head='WASHINGTON CITY D.C.(3.75mm letters)'
    -> inscription='WASHINGTON CITY D.C.'  town='WASHINGTON CITY D.C.'

=== rel_type=Same (73 entries) ===
  seg_head='Same(slotted circle)'
    -> inscription='WASH.CITY'  town='WASH.CITY'  parent_head='WASH.CITY(period high)'
  seg_head='Same'
    -> inscription='WASHN.CITY'  town='WASHN.CITY'  parent_head='WASHN.CITY("N"high)'
  seg_head='Same'
    -> inscription='WASHN.CITY'  town='WASHN.CITY'  parent_head='WASHN.CITY("N"high)'
  seg_hea

In [47]:
# --- Step 7.3: Validation ---

# V1: Every listing must have resolved_inscription and resolved_town
missing_inscription = listings['resolved_inscription'].isna() | (listings['resolved_inscription'] == '')
missing_town = listings['resolved_town'].isna() | (listings['resolved_town'] == '')
print(f'V1: Missing resolved_inscription: {missing_inscription.sum()}')
print(f'V1: Missing resolved_town: {missing_town.sum()}')
if missing_inscription.any() or missing_town.any():
    problem = listings[missing_inscription | missing_town]
    for _, row in problem.head(10).iterrows():
        print(f'  [{row["head_rel_type"]}] {row["clean_text"][:100]}')
        print(f'    inscription={row["resolved_inscription"]!r} town={row["resolved_town"]!r}')
print()

# V2: No rel entry should be orphaned
rel_entries = listings[listings['head_rel_type'].notna()]
orphans = rel_entries[rel_entries['parent_idx'].isna()]
print(f'V2: Orphan rel entries: {len(orphans)}')
if len(orphans):
    for _, row in orphans.iterrows():
        print(f'  [{row["head_rel_type"]}] {row["clean_text"][:100]}')
print()

# V3: head_first_of_town should co-occur with independent entry
#     (*(L) and *(E) are the exception: first_of_town + rel indicator)
fot_entries = listings[listings['head_first_of_town']]
fot_rel = fot_entries[fot_entries['head_rel_type'].notna()]
print(f'V3: first_of_town entries: {len(fot_entries)}')
print(f'    of which are rel indicators: {len(fot_rel)}')
if len(fot_rel):
    for _, row in fot_rel.iterrows():
        print(f'  [{row["head_rel_type"]}] {row["clean_text"][:100]}')
print()

# V4: Warning distribution
all_warnings = [w for wlist in listings['s7_warnings'] for w in wlist]
print(f'V4: Total warnings: {len(all_warnings)}')
if all_warnings:
    warn_counts = pd.Series(all_warnings).value_counts()
    for w, c in warn_counts.items():
        print(f'  {w}: {c}')
print()

# V5: Same-with-name-body slash check
same_with_name = listings[
    (listings['head_rel_type'] == 'Same') & listings['head_name_body'].notna()
]
no_slash = same_with_name[~same_with_name['head_name_body'].str.startswith('/')]
print(f'V5: Same-with-name entries: {len(same_with_name)}')
print(f'    starting with /: {len(same_with_name) - len(no_slash)}')
print(f'    NOT starting with / (flagged): {len(no_slash)}')
if len(no_slash):
    for _, row in no_slash.iterrows():
        print(f'  head={row["seg_head"]!r}  name_body={row["head_name_body"]!r}')
        print(f'    resolved={row["resolved_inscription"]!r}')

V1: Missing resolved_inscription: 0
V1: Missing resolved_town: 0

V2: Orphan rel entries: 0

V3: first_of_town entries: 2
    of which are rel indicators: 0

V4: Total warnings: 2
  same_name_body_no_slash: 2

V5: Same-with-name entries: 8
    starting with /: 6
    NOT starting with / (flagged): 2
  head='Same D.C./5'  name_body='D.C./5'
    resolved='WASHINGTON D.C.D.C./5'
  head='Same D.C./3 PAID'  name_body='D.C./3 PAID'
    resolved='GEORGETOWND.C./3 PAID'


In [48]:
# --- Step 7.4: Resolved town distribution ---

town_counts = listings['resolved_town'].value_counts()
print(f'Distinct towns: {len(town_counts)}')
print(f'Towns with most listings:')
for town, count in town_counts.head(20).items():
    print(f'  {town!r}: {count}')
print()

# Single-listing towns
single = (town_counts == 1).sum()
print(f'Single-listing towns: {single} ({single/len(town_counts)*100:.1f}%)')
print()

# Inscription variants per town (shows device diversity)
inscription_per_town = listings.groupby('resolved_town')['resolved_inscription'].nunique()
multi_variant = inscription_per_town[inscription_per_town > 1]
print(f'Towns with multiple inscription variants: {len(multi_variant)}')
for town, n_variants in multi_variant.head(15).items():
    variants = listings[listings['resolved_town'] == town]['resolved_inscription'].unique()
    print(f'  {town!r}: {n_variants} variants')
    for v in variants:
        print(f'    {v!r}')

Distinct towns: 45
Towns with most listings:
  'WASHINGTON D.C.': 22
  'WASHN.CITY': 19
  'WASHINGTON CITY D.C.': 12
  'WASHINGTON': 8
  'GEORGETOWN': 6
  'ALEXA.CA.': 5
  'DEAD LETTER OFFICE': 4
  'WASHINGTON CITY': 4
  'CITY OF WASHINGTON': 4
  'ALEXA.VA.': 3
  'GEOE.TOWN.D.C.': 3
  'ALEXANDRIA': 3
  'ALEX': 3
  'P.ODEPT.': 2
  'GEORGETN.DC.': 2
  'GEORN.CA': 2
  'WASH.CITY': 2
  'ALEXA VA': 2
  'W City': 2
  'ALEX.VA.': 2

Single-listing towns: 24 (53.3%)

Towns with multiple inscription variants: 8
  'DEAD LETTER OFFICE': 3 variants
    'DEAD LETTER OFFICE'
    'DEAD LETTER OFFICE/P.O.DEPT.'
    'DEAD LETTER OFFICE/P.O.DPT'
  'GEORGETN.DC.': 2 variants
    'GEORGETN.DC.'
    'GEORGETN.DC./D.C.'
  'GEORGETOWN': 4 variants
    'GEORGETOWN/DC.'
    'GEORGETOWN/D.C'
    'GEORGETOWND.C./3 PAID'
    'GEORGETOWN/D.C.'
  'P.ODEPT.': 2 variants
    'P.ODEPT./DEAD LETTER OFFICE'
    'P.ODEPT./G.P.O.'
  'WASHINGTON': 2 variants
    'WASHINGTON/D.C.'
    'WASHINGTON/DUE 6'
  'WASHINGTON CITY':

## 7.5 Step 7 Summary

Step 7 resolves cross-record inheritance, adding four new columns to `listings`:

- **parent_idx** -- DataFrame index of the parent entry for rel-indicator rows; null for independent entries
- **resolved_inscription** -- full inscription text on the physical device
- **resolved_town** -- town root for PostOffice grouping (text before first `/`)
- **s7_warnings** -- list of confidence-lowering flags

Step 7.1b propagates parsed attributes from parent to child when the child's own
paren body is silent on the attribute:

- **is_manuscript** -- inherited if child has no `ms` or `size` field type
- **parsed_colors** -- inherited if child's color list is empty
- **parsed_sizes** -- inherited if child's size list is empty

This ensures that relationship-indicator entries (`(L)`, `(E)`, `Same`) carry
the full physical device characteristics of their parent without requiring each
child to repeat them. Inheritance happens before color fan-out and shape
resolution so downstream steps see fully resolved attribute sets.

In [49]:
# --- Step 7.5: Summary ---

total = len(listings)
rel_count = listings['head_rel_type'].notna().sum()
warn_count = sum(len(w) for w in listings['s7_warnings'])

print('Step 7: Relationship Resolution')
print(f'  Listings processed: {total}')
print()
print(f'  Independent entries:  {total - rel_count}')
print(f'  Resolved from parent: {rel_count}')
print(f'    Same (no name):     {((listings["head_rel_type"] == "Same") & listings["head_name_body"].isna()).sum()}')
print(f'    Same (with name):   {((listings["head_rel_type"] == "Same") & listings["head_name_body"].notna()).sum()}')
print(f'    (L):                {(listings["head_rel_type"] == "(L)").sum()}')
print(f'    (E):                {(listings["head_rel_type"] == "(E)").sum()}')
print()
print(f'  Distinct towns:       {listings["resolved_town"].nunique()}')
print(f'  Total warnings:       {warn_count}')
print()

# Inheritance impact
rel_entries = listings[listings['parent_idx'].notna()]
print('  Attribute inheritance impact (rel entries only):')
ms_from_parent = rel_entries[rel_entries['is_manuscript']].shape[0]
print(f'    Manuscript after inheritance: {ms_from_parent}')
has_colors = rel_entries[rel_entries['parsed_colors'].apply(len) > 0].shape[0]
print(f'    With colors after inheritance: {has_colors}')
has_sizes = rel_entries[rel_entries['parsed_sizes'].apply(len) > 0].shape[0]
print(f'    With sizes after inheritance: {has_sizes}')

Step 7: Relationship Resolution
  Listings processed: 136

  Independent entries:  62
  Resolved from parent: 74
    Same (no name):     65
    Same (with name):   8
    (L):                1
    (E):                0

  Distinct towns:       45
  Total warnings:       2

  Attribute inheritance impact (rel entries only):
    Manuscript after inheritance: 1
    With colors after inheritance: 74
    With sizes after inheritance: 73


# Step 8: Postmark Assembly

Transforms the per-listing parsed data from Steps 1-7 into normalized domain
entity DataFrames aligned with the WorldCovers data model. This step handles:

- **Value table construction** -- Color (discovered from data), Shape and Framing (from model seeds)
- **Shape code decomposition** -- Compound ASCC codes (DC, DLC, DLDC, etc.) split into base Shape + MarkFraming rows
- **Color fan-out** -- Multi-color listings exploded into one Postmark per color
- **Postmark record assembly** -- All scalar fields populated, FK references resolved
- **Dependent entity assembly** -- DateObserved, PostmarkValuation, MarkFraming
- **PostOffice normalization** -- Town root dedup, case normalization

Output: Six DataFrames ready for export:
- `postmarks_df` -- one row per Postmark entity
- `date_observed_df` -- one row per date observation
- `postmark_valuation_df` -- one row per valuation tier
- `mark_framing_df` -- one row per framing application
- `post_offices_df` -- one row per normalized post office
- `colors_df`, `shapes_df`, `framings_df` -- value/lookup tables

## 8.1 Value Table Construction

Shape and Framing from model seed values (fixed vocabulary).
Color discovered from all unique color names across listings.

In [50]:
# --- Step 8.1: Value table construction ---

# Shape seeds (from model.md)
SHAPE_SEEDS = [
    'SL', 'BOX', 'O', 'C', 'ARC', 'Octagon',
    'Pictorial', 'Ornamental Mortised', 'Other',
]

shapes_df = pd.DataFrame({
    'shape_id': range(1, len(SHAPE_SEEDS) + 1),
    'name': SHAPE_SEEDS,
})
shape_lookup = dict(zip(shapes_df['name'].str.upper(), shapes_df['shape_id']))

# Framing seeds (from model.md)
FRAMING_SEEDS = [
    'NOR - No Outer Rim', 'SL - Single Line', 'DL - Double Line', 'Dotted',
    'Dashed', 'Cogwheel', 'Fancy', 'Ornate', 'Other',
]

framings_df = pd.DataFrame({
    'framing_id': range(1, len(FRAMING_SEEDS) + 1),
    'name': FRAMING_SEEDS,
})
framing_lookup = dict(zip(framings_df['name'], framings_df['framing_id']))

# Color: discover from data (UPPER normalized)
all_color_names = sorted({
    c for clist in listings['parsed_colors'] for c in clist
})

colors_df = pd.DataFrame({
    'color_id': range(1, len(all_color_names) + 1),
    'name': all_color_names,
})
color_lookup = dict(zip(colors_df['name'], colors_df['color_id']))

# Lettering seeds (from model.md)
LETTERING_SEEDS = [
    'Italic', 'Sans-serif', 'Script', 'Printed', 'Serif',
    'Hollow', 'Thin', 'Block', 'Roman', 'Seriffed', 'Bold',
    'Thick', 'Gothic', 'Other',
]

letterings_df = pd.DataFrame({
    'lettering_id': range(1, len(LETTERING_SEEDS) + 1),
    'name': LETTERING_SEEDS,
})
lettering_lookup = dict(zip(
    letterings_df['name'].str.lower(),
    letterings_df['lettering_id'],
))
# Common plural/variant aliases used in ASCC bracket descriptors
_lettering_aliases = {
    'italics': 'italic',
    'serifs': 'serif',
    'sans serifs': 'sans-serif',
    'sans serif': 'sans-serif',
}
for alias, canonical in _lettering_aliases.items():
    if canonical in lettering_lookup:
        lettering_lookup[alias] = lettering_lookup[canonical]

print(f'Value tables constructed:')
print(f'  Shapes:     {len(shapes_df)} seeds')
print(f'  Framings:   {len(framings_df)} seeds')
print(f'  Colors:     {len(colors_df)} discovered')
print(f'  Letterings: {len(letterings_df)} seeds')
print()
print(f'Colors: {all_color_names}')

Value tables constructed:
  Shapes:     9 seeds
  Framings:   9 seeds
  Colors:     16 discovered
  Letterings: 14 seeds

Colors: ['---', 'BLACK', 'BLACK-BROWN', 'BLUE', 'BROWN', 'GRAY-BROWN', 'GREEN', 'OLIVE', 'OLIVE-GREEN', 'ORANGE', 'PINK', 'PURPLE', 'PURPLISH', 'RED', 'RED-BROWN', 'RED-ORANGE']


## 8.2 Shape Code Decomposition

Compound ASCC codes are decomposed into a base Shape (FK to shapes_df) and
zero or more MarkFraming rows. Each listing's effective shape is resolved by
priority: paren-body shape code > section Default Shape > catalog-wide SL fallback.

Decomposition table:

| Code | Base Shape | MarkFraming rows |
|------|-----------|-----------------|
| SL | SL | (none) |
| C | C | SL - Single Line pos 1 |
| O | O | SL - Single Line pos 1 |
| DC | C | SL - Single Line pos 1, SL - Single Line pos 2 |
| DO | O | SL - Single Line pos 1, SL - Single Line pos 2 |
| DLC | C | DL - Double Line pos 1, SL - Single Line pos 2 |
| DLO | O | DL - Double Line pos 1, SL - Single Line pos 2 |
| DLDC | C | DL - Double Line pos 1, DL - Double Line pos 2 |
| DLDO | O | DL - Double Line pos 1, DL - Double Line pos 2 |
| NOR | C | NOR - No Outer Rim pos 1 |
| BOX | BOX | SL - Single Line pos 1 |
| ARC | ARC | (none) |
| OCTAGON | Octagon | SL - Single Line pos 1 |
| PMK | Other | (none) |

In [51]:
# --- Step 8.2: Shape code decomposition ---

# Decomposition table: ASCC code -> (base_shape_name, [(framing_name, position), ...])
# position=None means the framing applies but ring order is unverified.
SHAPE_DECOMP = {
    'SL':      ('SL',      []),
    'C':       ('C',       [('SL - Single Line', 1)]),
    'O':       ('O',       [('SL - Single Line', 1)]),
    'DC':      ('C',       [('DL - Double Line', 1)]),
    'DO':      ('O',       [('DL - Double Line', 1)]),
    'DLC':     ('C',       [('DL - Double Line', None), ('SL - Single Line', None)]),
    'DLO':     ('O',       [('DL - Double Line', None), ('SL - Single Line', None)]),
    'DLDC':    ('C',       [('DL - Double Line', None), ('DL - Double Line', None)]),
    'DLDO':    ('O',       [('DL - Double Line', None), ('DL - Double Line', None)]),
    'NOR':     ('C',       [('NOR - No Outer Rim', 1)]),
    'BOX':     ('BOX',     [('SL - Single Line', 1)]),
    'ARC':     ('ARC',     []),
    'OCTAGON': ('Octagon', [('SL - Single Line', 1)]),
    'PMK':     ('Other',   []),
}

# Codes where DL prefix creates inner/outer ring ambiguity
DL_AMBIGUOUS_CODES = {'DLC', 'DLO', 'DLDC', 'DLDO'}

CATALOG_FALLBACK_SHAPE = 'SL'


def resolve_effective_shape(row):
    """Determine the effective shape code for a listing.
    Priority: paren-body shape code > Default Shape column > catalog-wide SL.
    Returns (effective_code_upper, source_label)."""

    # 1. Paren-body shape (from parsed_sizes -- use first non-None)
    for s in row['parsed_sizes']:
        if s.get('size_shape_code'):
            return s['size_shape_code'].upper(), 'paren_body'

    # 2. Section-level Default Shape
    default = row.get('Default Shape')
    if pd.notna(default) and str(default).strip():
        # Default Shape column may use full names or codes
        ds = str(default).strip().upper()
        # Map common full names to codes
        name_to_code = {
            'CIRCLE': 'C', 'OVAL': 'O', 'STRAIGHT LINE': 'SL',
            'BOX': 'BOX', 'ARC': 'ARC', 'OCTAGON': 'OCTAGON',
            'DOUBLE CIRCLE': 'DC', 'DOUBLE OVAL': 'DO',
            'DOUBLE LINE CIRCLE': 'DLC', 'DOUBLE LINE OVAL': 'DLO',
            'NO OUTER RIM': 'NOR', 'FANCY': 'C',  # fancy = circle variant
        }
        # Try direct code match first, then name map
        if ds in SHAPE_DECOMP:
            return ds, 'default_shape'
        if ds in name_to_code:
            return name_to_code[ds], 'default_shape'
        # Partial match: check if ds starts with a known code
        for code in sorted(SHAPE_DECOMP.keys(), key=len, reverse=True):
            if ds.startswith(code):
                return code, 'default_shape'
        # Unrecognized default shape -- fall through to catalog default

    # 3. Catalog-wide fallback
    return CATALOG_FALLBACK_SHAPE, 'catalog_fallback'


def decompose_shape(code_upper):
    """Decompose a shape code into (base_shape_name, [(framing_name, position)]).
    Returns (shape_name, framing_list, error_or_None)."""
    if code_upper in SHAPE_DECOMP:
        base, framings = SHAPE_DECOMP[code_upper]
        return base, framings, None
    # Unknown code -- map to Other
    return 'Other', [], f'unknown shape code: {code_upper}'


# Apply to all listings
shape_resolution = listings.apply(
    lambda row: pd.Series(resolve_effective_shape(row),
                          index=['effective_shape_code', 'shape_source']),
    axis=1
)
listings = pd.concat([listings, shape_resolution], axis=1)

# Decompose
decomposed = listings['effective_shape_code'].apply(
    lambda c: pd.Series(decompose_shape(c),
                        index=['base_shape_name', 'shape_framings', 'shape_error'])
)
listings = pd.concat([listings, decomposed], axis=1)

# Resolve shape FK
listings['shape_id'] = listings['base_shape_name'].apply(
    lambda n: shape_lookup.get(n.upper())
)

# Report
print('Shape resolution source distribution:')
for src, count in listings['shape_source'].value_counts().items():
    print(f'  {src}: {count}')
print()

print('Base shape distribution:')
for name, count in listings['base_shape_name'].value_counts().items():
    print(f'  {name}: {count}')
print()

dl_count = listings['effective_shape_code'].isin(DL_AMBIGUOUS_CODES).sum()
print(f'DL-prefix codes (ring order ambiguous): {dl_count}')
print()

errors = listings[listings['shape_error'].notna()]
if len(errors):
    print(f'Shape decomposition errors: {len(errors)}')
    for _, row in errors.head(10).iterrows():
        print(f'  {row["shape_error"]}  raw: {row["clean_text"][:80]}')
else:
    print('No shape decomposition errors.')

Shape resolution source distribution:
  default_shape: 123
  paren_body: 13

Base shape distribution:
  C: 128
  SL: 4
  O: 4

DL-prefix codes (ring order ambiguous): 0

No shape decomposition errors.


## 8.3 Color Fan-Out

Listings with multiple `parsed_colors` are exploded into N Postmark rows, one
per color. All other attributes are duplicated identically. Listings with zero
colors get one row with null color. Listings with one color get one row.

Multi-color fan-out is flagged for the final confidence score since it is
inherently ambiguous whether multiple colors represent the same device in
different ink, or separate observations.

In [52]:
# --- Step 8.3: Color fan-out ---

expanded_rows = []
next_postmark_id = 1

for idx, row in listings.iterrows():
    colors = row['parsed_colors']
    is_multi_color = len(colors) > 1

    if not colors:
        # No color parsed -- single row, null color
        r = {
            'postmark_id': next_postmark_id,
            'source_listing_idx': idx,
            'color_name': None,
            'color_id': None,
            'is_multi_color_fanout': False,
        }
        expanded_rows.append(r)
        next_postmark_id += 1
    else:
        for color_name in colors:
            r = {
                'postmark_id': next_postmark_id,
                'source_listing_idx': idx,
                'color_name': color_name,
                'color_id': color_lookup.get(color_name),
                'is_multi_color_fanout': is_multi_color,
            }
            expanded_rows.append(r)
            next_postmark_id += 1

fanout_df = pd.DataFrame(expanded_rows)

multi_color_listings = listings[listings['parsed_colors'].apply(len) > 1]
total_postmarks = len(fanout_df)
print(f'Color fan-out results:')
print(f'  Input listings: {len(listings)}')
print(f'  Output postmark rows: {total_postmarks}')
print(f'  Net expansion: {total_postmarks - len(listings)} rows')
print(f'  Multi-color source listings: {len(multi_color_listings)}')
print(f'  Rows from multi-color fan-out: {fanout_df["is_multi_color_fanout"].sum()}')
print()

# Distribution of colors-per-listing
color_counts = listings['parsed_colors'].apply(len).value_counts().sort_index()
print('Colors-per-listing distribution:')
for n, count in color_counts.items():
    print(f'  {n} colors: {count} listings')

Color fan-out results:
  Input listings: 136
  Output postmark rows: 191
  Net expansion: 55 rows
  Multi-color source listings: 38
  Rows from multi-color fan-out: 93

Colors-per-listing distribution:
  0 colors: 1 listings
  1 colors: 97 listings
  2 colors: 26 listings
  3 colors: 7 listings
  4 colors: 5 listings


## 8.4 Postmark Record Assembly

Build the main `postmarks_df` with all Postmark model fields. Each row in
`fanout_df` becomes one Postmark record. Scalar fields are pulled from the
source listing; color comes from the fan-out.

In [ ]:
# --- Step 8.4: Postmark record assembly ---

def build_postmark(fan_row):
    """Assemble a single Postmark record from a fan-out row + source listing."""
    src = listings.loc[fan_row['source_listing_idx']]

    is_ms = bool(src['is_manuscript'])

    # Dimensions: first parsed_sizes entry (if any)
    width, height = None, None
    is_irreg = None if is_ms else False
    date_format = None
    if src['parsed_sizes']:
        s = src['parsed_sizes'][0]
        width = s.get('size_dim1')
        height = s.get('size_dim2')
        # Single measurement: use for both width and height (circular/square marking)
        if width is not None and height is None:
            height = width
        if s.get('size_is_irregular'):
            is_irreg = True
        if s.get('size_dateformat'):
            date_format = s['size_dateformat']

    # Shape: null for manuscript, required for handstamped
    shape_id = None if is_ms else src['shape_id']

    # Impression: null for manuscript, default Normal for handstamped
    impression = None if is_ms else 'Normal'

    # Lettering: null for manuscript (per invariant), resolved from annotations otherwise
    lettering_id = None if is_ms else src.get('lettering_id')

    # Parent listing: which listing this postmark inherited from (intermediate for review)
    parent_listing_idx = src.get('parent_idx')
    if pd.notna(parent_listing_idx):
        parent_listing_idx = int(parent_listing_idx)
    else:
        parent_listing_idx = None

    # Images above: catalog page image count (intermediate field for human review)
    images_above = src.get('Images Above')
    if pd.notna(images_above):
        images_above = int(images_above)
    else:
        images_above = None

    return {
        'postmark_id': fan_row['postmark_id'],
        'catalog_text': src['clean_text'],
        'inscription_text': src['resolved_inscription'],
        'is_manuscript': is_ms,
        'shape_id': shape_id,
        'lettering_id': lettering_id,
        'color_id': fan_row['color_id'],
        'width': width,
        'height': height,
        'is_irregular': is_irreg,
        'date_format': date_format,
        'date_type': None,  # Bishop/Franklin/Quaker -- resolved when catalog signals are present
        'impression': impression,
        'post_office_id': None,  # filled in 8.8
        'source_listing_idx': fan_row['source_listing_idx'],
        'color_name': fan_row['color_name'],
        'is_multi_color_fanout': fan_row['is_multi_color_fanout'],
        'images_above': images_above,
        'parent_listing_idx': parent_listing_idx,
    }


postmarks_df = pd.DataFrame(
    [build_postmark(row) for _, row in fanout_df.iterrows()]
)

# --- Resolve parent_listing_idx -> parent_postmark_id ---
# Color-position-aware: match child postmark to parent postmark by color name.
# This ensures that when a parent fans out to (RED, BLUE) and the child
# inherits those colors, each child postmark points to the correct color-
# matched parent postmark rather than always the first.

_pm_by_listing_color = {}
for _, row in postmarks_df.iterrows():
    key = (row['source_listing_idx'], row['color_name'])
    _pm_by_listing_color[key] = row['postmark_id']

# Fallback: first postmark per listing (for cases where colors differ)
_pm_first_by_listing = postmarks_df.groupby('source_listing_idx')['postmark_id'].first()

def resolve_parent_pm(row):
    pidx = row['parent_listing_idx']
    if pd.isna(pidx) or pidx is None:
        return None
    pidx = int(pidx)
    # Try color-matched first
    key = (pidx, row['color_name'])
    if key in _pm_by_listing_color:
        return _pm_by_listing_color[key]
    # Fallback to first postmark of parent
    return _pm_first_by_listing.get(pidx)

postmarks_df['parent_postmark_id'] = postmarks_df.apply(resolve_parent_pm, axis=1)
postmarks_df.drop(columns=['parent_listing_idx'], inplace=True)

print(f'Postmarks assembled: {len(postmarks_df)}')
print(f'  Manuscript: {postmarks_df["is_manuscript"].sum()}')
print(f'  Handstamped: {(~postmarks_df["is_manuscript"]).sum()}')
print(f'  With color: {postmarks_df["color_id"].notna().sum()}')
print(f'  With dimensions: {postmarks_df["width"].notna().sum()}')
print(f'  With images_above: {postmarks_df["images_above"].notna().sum()}')
print(f'  With parent (inherited): {postmarks_df["parent_postmark_id"].notna().sum()}')
print(f'  With date_fmt: {postmarks_df["date_format"].notna().sum()}')
print(f'  Multi-color fan-out rows: {postmarks_df["is_multi_color_fanout"].sum()}')
print()

# Parent linkage: color-matched vs fallback
if postmarks_df['parent_postmark_id'].notna().any():
    children = postmarks_df[postmarks_df['parent_postmark_id'].notna()]
    color_matched = 0
    fallback_used = 0
    for _, row in children.iterrows():
        parent_pm = postmarks_df[postmarks_df['postmark_id'] == row['parent_postmark_id']].iloc[0]
        if row['color_name'] == parent_pm['color_name']:
            color_matched += 1
        else:
            fallback_used += 1
    print(f'Parent-postmark linkage:')
    print(f'  Color-matched: {color_matched}')
    print(f'  Fallback (first of parent): {fallback_used}')
    print()

# Shape distribution on postmarks
shape_dist = postmarks_df['shape_id'].value_counts(dropna=False).sort_index()
print('Shape distribution on postmarks:')
for sid, count in shape_dist.items():
    if pd.isna(sid):
        print(f'  (null -- manuscript): {count}')
    else:
        name = shapes_df.loc[shapes_df['shape_id'] == sid, 'name'].iloc[0]
        print(f'  {name}: {count}')
print()

# Lettering distribution on postmarks
lettering_hits = postmarks_df['lettering_id'].notna().sum()
print(f'Lettering assigned on postmarks: {lettering_hits}')
if lettering_hits:
    for lid, count in postmarks_df['lettering_id'].value_counts(dropna=True).items():
        name = letterings_df.loc[letterings_df['lettering_id'] == lid, 'name'].iloc[0]
        print(f'  {name}: {count}')

Postmarks assembled: 191
  Manuscript: 9
  Handstamped: 182
  With color: 190
  With dimensions: 173
  With images_above: 191
  With parent (inherited): 108
  With date_format: 12
  Multi-color fan-out rows: 93

Parent-postmark linkage:
  Color-matched: 60
  Fallback (first of parent): 48

Shape distribution on postmarks:
  SL: 4
  O: 4
  C: 174
  (null -- manuscript): 9

Lettering assigned on postmarks: 0


## 8.5 DateObserved Assembly

Each parsed date from Step 6 becomes a DateObserved row linked to every Postmark
that was produced from that listing (including fan-out siblings).

Granularity mapping:
- DAY -> granularity=DAY, date = full date
- MONTH -> granularity=MONTH, date = year-month-01
- YEAR -> granularity=YEAR, date = year-01-01
- RANGE and DECADE -> two bookend YEAR rows (start, end) since the model
  uses individual date points, not spans

In [54]:
# --- Step 8.5: DateObserved assembly ---

from datetime import date as dt_date

date_rows = []
next_date_id = 1

for _, pm in postmarks_df.iterrows():
    src = listings.loc[pm['source_listing_idx']]
    for d in src['parsed_dates']:
        gran = d['date_granularity']

        if gran == 'DAY':
            try:
                obs_date = dt_date(d['date_year_start'], d['date_month'], d['date_day'])
            except (ValueError, TypeError):
                obs_date = None
            date_rows.append({
                'date_observed_id': next_date_id,
                'postmark_id': pm['postmark_id'],
                'date': str(obs_date) if obs_date else None,
                'granularity': 'DAY',
                'date_raw': d.get('date_raw'),
                'date_error': d.get('date_error'),
            })
            next_date_id += 1

        elif gran == 'MONTH':
            try:
                obs_date = dt_date(d['date_year_start'], d['date_month'], 1)
            except (ValueError, TypeError):
                obs_date = None
            date_rows.append({
                'date_observed_id': next_date_id,
                'postmark_id': pm['postmark_id'],
                'date': str(obs_date) if obs_date else None,
                'granularity': 'MONTH',
                'date_raw': d.get('date_raw'),
                'date_error': d.get('date_error'),
            })
            next_date_id += 1

        elif gran == 'YEAR':
            try:
                obs_date = dt_date(d['date_year_start'], 1, 1)
            except (ValueError, TypeError):
                obs_date = None
            date_rows.append({
                'date_observed_id': next_date_id,
                'postmark_id': pm['postmark_id'],
                'date': str(obs_date) if obs_date else None,
                'granularity': 'YEAR',
                'date_raw': d.get('date_raw'),
                'date_error': d.get('date_error'),
            })
            next_date_id += 1

        elif gran in ('RANGE', 'DECADE'):
            # Two bookend YEAR rows
            for yr in (d['date_year_start'], d['date_year_end']):
                try:
                    obs_date = dt_date(int(yr), 1, 1)
                except (ValueError, TypeError):
                    obs_date = None
                date_rows.append({
                    'date_observed_id': next_date_id,
                    'postmark_id': pm['postmark_id'],
                    'date': str(obs_date) if obs_date else None,
                    'granularity': 'YEAR',
                    'date_raw': d.get('date_raw'),
                    'date_error': d.get('date_error'),
                })
                next_date_id += 1

date_observed_df = pd.DataFrame(date_rows) if date_rows else pd.DataFrame(
    columns=['date_observed_id', 'postmark_id', 'date', 'granularity',
             'date_raw', 'date_error']
)

print(f'DateObserved rows: {len(date_observed_df)}')
if len(date_observed_df):
    print(f'  Linked to {date_observed_df["postmark_id"].nunique()} postmarks')
    gran_dist = date_observed_df['granularity'].value_counts()
    for g, c in gran_dist.items():
        print(f'  {g}: {c}')
    errors = date_observed_df[date_observed_df['date'].isna()]
    if len(errors):
        print(f'  Date construction errors: {len(errors)}')

DateObserved rows: 301
  Linked to 180 postmarks
  YEAR: 298
  DAY: 3


## 8.6 PostmarkValuation Assembly

Each valuation tier from Step 3 becomes a PostmarkValuation row. Position is
1-based ordinal (position 1 = earliest date period). Multi-color fan-out
siblings share the same valuation (each Postmark gets a copy of all tiers).

In [55]:
# --- Step 8.6: PostmarkValuation assembly ---

val_rows = []
next_val_id = 1

for _, pm in postmarks_df.iterrows():
    src = listings.loc[pm['source_listing_idx']]
    tiers = src['valuation_tiers']

    for pos_0, tier_str in enumerate(tiers):
        amount = None
        if tier_str is not None:
            # Strip commas, parse as float
            try:
                amount = float(tier_str.replace(',', ''))
            except (ValueError, AttributeError):
                amount = None  # unparseable -- leave null

        val_rows.append({
            'valuation_id': next_val_id,
            'postmark_id': pm['postmark_id'],
            'amount': amount,
            'appraisal_position': pos_0 + 1,  # 1-based
            'appraisal_date': None,  # catalog publication date; TBD
        })
        next_val_id += 1

postmark_valuation_df = pd.DataFrame(val_rows) if val_rows else pd.DataFrame(
    columns=['valuation_id', 'postmark_id', 'amount',
             'appraisal_position', 'appraisal_date']
)

print(f'PostmarkValuation rows: {len(postmark_valuation_df)}')
if len(postmark_valuation_df):
    print(f'  Linked to {postmark_valuation_df["postmark_id"].nunique()} postmarks')
    pos_dist = postmark_valuation_df['appraisal_position'].value_counts().sort_index()
    for pos, c in pos_dist.items():
        print(f'  Position {pos}: {c}')
    unpriced = postmark_valuation_df['amount'].isna().sum()
    print(f'  Unpriced (null amount): {unpriced}')

PostmarkValuation rows: 179
  Linked to 179 postmarks
  Position 1: 179
  Unpriced (null amount): 0


## 8.7 MarkFraming Assembly

Compound shape codes decomposed in 8.2 produce MarkFraming rows linking each
Postmark to its Framing(s) with positional ordering (outermost = 1).
Manuscript postmarks produce no MarkFraming rows.

In [56]:
# --- Step 8.7: MarkFraming assembly ---

mf_rows = []
next_mf_id = 1

for _, pm in postmarks_df.iterrows():
    if pm['is_manuscript']:
        continue

    src = listings.loc[pm['source_listing_idx']]
    framings_list = src['shape_framings']

    for framing_name, position in framings_list:
        fid = framing_lookup.get(framing_name)
        if fid is None:
            continue  # should not happen with seed vocabulary

        mf_rows.append({
            'mark_framing_id': next_mf_id,
            'parent_mark_type': 'POSTMARK',
            'parent_mark_id': pm['postmark_id'],
            'framing_id': fid,
            'framing_position': position,
        })
        next_mf_id += 1

mark_framing_df = pd.DataFrame(mf_rows) if mf_rows else pd.DataFrame(
    columns=['mark_framing_id', 'parent_mark_type', 'parent_mark_id',
             'framing_id', 'framing_position']
)

print(f'MarkFraming rows: {len(mark_framing_df)}')
if len(mark_framing_df):
    print(f'  Linked to {mark_framing_df["parent_mark_id"].nunique()} postmarks')
    framing_dist = mark_framing_df['framing_id'].value_counts()
    for fid, c in framing_dist.items():
        fname = framings_df.loc[framings_df['framing_id'] == fid, 'name'].iloc[0]
        print(f'  {fname}: {c}')

MarkFraming rows: 178
  Linked to 178 postmarks
  SL - Single Line: 171
  DL - Double Line: 6
  NOR - No Outer Rim: 1


## 8.8 PostOffice Normalization

Normalize `resolved_town` into canonical PostOffice records. Normalization:
- Strip trailing periods and commas
- Uppercase for grouping (preserve one representative display form)
- Assign PostOffice IDs
- Back-fill `post_office_id` on postmarks_df

In [57]:
# --- Step 8.8: PostOffice normalization ---

import re as _re

def normalize_town(raw_town):
    """Normalize a raw town root for PostOffice grouping."""
    if not raw_town or (isinstance(raw_town, float) and pd.isna(raw_town)):
        return None
    t = str(raw_town).strip()
    # Strip trailing periods, commas, and spaces
    t = _re.sub(r'[.,\s]+$', '', t)
    # Uppercase for consistent output
    t = t.upper()
    return t


# Build normalized town for each listing
listings['normalized_town'] = listings['resolved_town'].apply(normalize_town)

# Group by uppercased form, pick first occurrence as display name
town_groups = {}
for _, row in listings.iterrows():
    nt = row['normalized_town']
    if nt is None:
        continue
    if nt not in town_groups:
        town_groups[nt] = nt  # already UPPER

# Build post_offices_df
po_rows = []
next_po_id = 1
po_lookup = {}  # UPPER key -> post_office_id

for key in sorted(town_groups.keys()):
    po_rows.append({
        'post_office_id': next_po_id,
        'name': town_groups[key],
        'region_id': None,  # Region assignment is a separate concern
    })
    po_lookup[key] = next_po_id
    next_po_id += 1

post_offices_df = pd.DataFrame(po_rows) if po_rows else pd.DataFrame(
    columns=['post_office_id', 'name', 'region_id']
)

# Map listings to PostOffice IDs
listings['post_office_id'] = listings['normalized_town'].apply(
    lambda t: po_lookup.get(t) if t else None
)

# Back-fill post_office_id onto postmarks_df
listing_po = listings['post_office_id']
postmarks_df['post_office_id'] = postmarks_df['source_listing_idx'].map(listing_po)

print(f'PostOffice records: {len(post_offices_df)}')
print(f'  From {listings["resolved_town"].nunique()} raw distinct towns')
print(f'  To {len(post_offices_df)} normalized post offices')
print()

# Show normalization collapses (if any)
raw_to_norm = listings.groupby('normalized_town')['resolved_town'].apply(
    lambda x: list(x.unique())
)
collapsed = raw_to_norm[raw_to_norm.apply(len) > 1]
if len(collapsed):
    print(f'Towns collapsed by normalization: {len(collapsed)}')
    for norm, variants in collapsed.head(20).items():
        print(f'  {norm!r} <- {variants}')
else:
    print('No normalization collapses (all raw towns map 1:1).')
print()

# Verify all postmarks have a post_office_id
missing_po = postmarks_df['post_office_id'].isna().sum()
if missing_po:
    print(f'WARNING: {missing_po} postmarks missing post_office_id')
else:
    print('All postmarks have a post_office_id.')

PostOffice records: 44
  From 45 raw distinct towns
  To 44 normalized post offices

Towns collapsed by normalization: 1
  'ALEX' <- ['ALEX.', 'ALEX']

All postmarks have a post_office_id.


## 8.9 Assembly Confidence

Combine warnings from all steps into a per-postmark confidence assessment.
Warnings reduce confidence from HIGH to MEDIUM or LOW.

In [58]:
# --- Step 8.9: Assembly confidence ---

def compute_assembly_warnings(pm_row):
    """Collect all warnings applicable to a single postmark."""
    src = listings.loc[pm_row['source_listing_idx']]
    warnings = list(src.get('s7_warnings', []))

    # Step 1 confidence
    if src.get('confidence') == 'low':
        warnings.append(f'low_classification_confidence:{src.get("reason")}')

    # Multi-color ambiguity
    if pm_row['is_multi_color_fanout']:
        warnings.append('multi_color_fanout')

    # Shape fallback
    if src.get('shape_source') == 'catalog_fallback':
        warnings.append('shape_from_catalog_fallback')

    # Shape decomposition error
    if pd.notna(src.get('shape_error')):
        warnings.append(f'shape_error:{src["shape_error"]}')

    # DL-prefix ring order ambiguity
    if src.get('effective_shape_code') in DL_AMBIGUOUS_CODES:
        warnings.append('dl_ring_order_ambiguous')

    # Missing dates
    if not src['parsed_dates']:
        warnings.append('no_dates_parsed')

    # Multiple size fields (ambiguous dimensions)
    if len(src['parsed_sizes']) > 1:
        warnings.append(f'multiple_size_fields:{len(src["parsed_sizes"])}')

    # Unresolved other fields
    if src['other_fields']:
        warnings.append(f'unresolved_other_fields:{len(src["other_fields"])}')

    return warnings


postmarks_df['s8_warnings'] = postmarks_df.apply(
    compute_assembly_warnings, axis=1
)

# Confidence level: HIGH if no warnings, MEDIUM if 1-2, LOW if 3+
def confidence_level(warnings):
    n = len(warnings)
    if n == 0:
        return 'HIGH'
    elif n <= 2:
        return 'MEDIUM'
    else:
        return 'LOW'

postmarks_df['assembly_confidence'] = postmarks_df['s8_warnings'].apply(confidence_level)

print('Assembly confidence distribution:')
for level, count in postmarks_df['assembly_confidence'].value_counts().items():
    print(f'  {level}: {count}')
print()

# Warning frequency
all_warnings = [w for wlist in postmarks_df['s8_warnings'] for w in wlist]
if all_warnings:
    warn_counts = pd.Series(all_warnings).value_counts()
    print(f'Warning frequency ({len(all_warnings)} total):')
    for w, c in warn_counts.items():
        print(f'  {w}: {c}')
else:
    print('No warnings -- all postmarks at HIGH confidence.')

Assembly confidence distribution:
  MEDIUM: 111
  HIGH: 80

Warning frequency (125 total):
  multi_color_fanout: 93
  no_dates_parsed: 11
  unresolved_other_fields:1: 11
  multiple_size_fields:2: 5
  same_name_body_no_slash: 3
  low_classification_confidence:anatomy_only: 2


## 8.10 Step 8 Summary

Step 8 transforms the per-listing parsed data into normalized domain-entity
DataFrames. Output tables:

- **postmarks_df** -- one Postmark per color variant per listing
- **date_observed_df** -- date observations linked to postmarks
- **postmark_valuation_df** -- valuation tiers linked to postmarks
- **mark_framing_df** -- framing applications linked to postmarks
- **post_offices_df** -- normalized post office records
- **colors_df, shapes_df, framings_df** -- value/lookup tables

PostOffice normalization collapses raw town roots by stripping trailing
punctuation and grouping by uppercased form.

Assembly confidence combines warnings from all pipeline stages into a
per-postmark HIGH/MEDIUM/LOW rating.

In [59]:
# --- Step 8.10: Summary ---

print('=' * 60)
print('Step 8: Postmark Assembly -- Final Summary')
print('=' * 60)
print()
print(f'Input listings:           {len(listings)}')
print(f'Output postmarks:         {len(postmarks_df)}')
print(f'  (expansion from color fan-out: +{len(postmarks_df) - len(listings)})')
print()
print('Domain entity tables:')
print(f'  postmarks_df:           {len(postmarks_df)} rows')
print(f'  date_observed_df:       {len(date_observed_df)} rows')
print(f'  postmark_valuation_df:  {len(postmark_valuation_df)} rows')
print(f'  mark_framing_df:        {len(mark_framing_df)} rows')
print(f'  post_offices_df:        {len(post_offices_df)} rows')
print()
print('Value/lookup tables:')
print(f'  colors_df:              {len(colors_df)} rows')
print(f'  shapes_df:              {len(shapes_df)} rows')
print(f'  framings_df:            {len(framings_df)} rows')
print()
print('Confidence distribution:')
for level in ['HIGH', 'MEDIUM', 'LOW']:
    count = (postmarks_df['assembly_confidence'] == level).sum()
    pct = count / len(postmarks_df) * 100
    print(f'  {level}: {count} ({pct:.1f}%)')
print()

# FK integrity checks
print('FK integrity checks:')
orphan_colors = postmarks_df[
    postmarks_df['color_id'].notna() &
    ~postmarks_df['color_id'].isin(colors_df['color_id'])
]
print(f'  Postmarks with invalid color_id: {len(orphan_colors)}')

orphan_shapes = postmarks_df[
    postmarks_df['shape_id'].notna() &
    ~postmarks_df['shape_id'].isin(shapes_df['shape_id'])
]
print(f'  Postmarks with invalid shape_id: {len(orphan_shapes)}')

orphan_po = postmarks_df[
    postmarks_df['post_office_id'].notna() &
    ~postmarks_df['post_office_id'].isin(post_offices_df['post_office_id'])
]
print(f'  Postmarks with invalid post_office_id: {len(orphan_po)}')

orphan_dates = date_observed_df[
    ~date_observed_df['postmark_id'].isin(postmarks_df['postmark_id'])
] if len(date_observed_df) else pd.DataFrame()
print(f'  DateObserved with invalid postmark_id: {len(orphan_dates)}')

orphan_vals = postmark_valuation_df[
    ~postmark_valuation_df['postmark_id'].isin(postmarks_df['postmark_id'])
] if len(postmark_valuation_df) else pd.DataFrame()
print(f'  PostmarkValuation with invalid postmark_id: {len(orphan_vals)}')

orphan_mf = mark_framing_df[
    ~mark_framing_df['parent_mark_id'].isin(postmarks_df['postmark_id'])
] if len(mark_framing_df) else pd.DataFrame()
print(f'  MarkFraming with invalid parent_mark_id: {len(orphan_mf)}')

orphan_mf_framing = mark_framing_df[
    ~mark_framing_df['framing_id'].isin(framings_df['framing_id'])
] if len(mark_framing_df) else pd.DataFrame()
print(f'  MarkFraming with invalid framing_id: {len(orphan_mf_framing)}')

Step 8: Postmark Assembly -- Final Summary

Input listings:           136
Output postmarks:         191
  (expansion from color fan-out: +55)

Domain entity tables:
  postmarks_df:           191 rows
  date_observed_df:       301 rows
  postmark_valuation_df:  179 rows
  mark_framing_df:        178 rows
  post_offices_df:        44 rows

Value/lookup tables:
  colors_df:              16 rows
  shapes_df:              9 rows
  framings_df:            9 rows

Confidence distribution:
  HIGH: 80 (41.9%)
  MEDIUM: 111 (58.1%)
  LOW: 0 (0.0%)

FK integrity checks:
  Postmarks with invalid color_id: 0
  Postmarks with invalid shape_id: 0
  Postmarks with invalid post_office_id: 0
  DateObserved with invalid postmark_id: 0
  PostmarkValuation with invalid postmark_id: 0
  MarkFraming with invalid parent_mark_id: 0
  MarkFraming with invalid framing_id: 0


## 8.11 Sample Inspection

Spot-check a few assembled postmarks with their full dependency tree.

In [ ]:
# --- Step 8.11: Sample inspection ---

def inspect_postmark(pm_id):
    """Print full assembled state for a single postmark."""
    pm = postmarks_df[postmarks_df['postmark_id'] == pm_id].iloc[0]
    print(f'=== Postmark {pm_id} ===')
    print(f'  catalog_text:    {pm["catalog_text"][:100]}')
    print(f'  inscription:     {pm["inscription_text"]}')
    print(f'  is_manuscript:   {pm["is_manuscript"]}')

    shape_name = '(null)'
    if pd.notna(pm['shape_id']):
        shape_name = shapes_df.loc[shapes_df['shape_id'] == pm['shape_id'], 'name'].iloc[0]
    print(f'  shape:           {shape_name}')

    color_name = pm.get('color_name', '(null)')
    print(f'  color:           {color_name}')
    print(f'  dimensions:      {pm["width"]}x{pm["height"]}')
    print(f'  date_fmt:     {pm["date_format"]}')
    print(f'  impression:      {pm["impression"]}')
    print(f'  is_irreg:    {pm["is_irregular"]}')

    po_name = '(null)'
    if pd.notna(pm['post_office_id']):
        po_name = post_offices_df.loc[
            post_offices_df['post_office_id'] == pm['post_office_id'], 'name'
        ].iloc[0]
    print(f'  post_office:     {po_name}')

    parent_pm = pm.get('parent_postmark_id')
    print(f'  parent_pm_id:    {parent_pm}')

    print(f'  confidence:      {pm["assembly_confidence"]}')
    if pm['s8_warnings']:
        print(f'  warnings:        {pm["s8_warnings"]}')

    # DateObserved
    dates = date_observed_df[date_observed_df['postmark_id'] == pm_id]
    print(f'  dates ({len(dates)}):')
    for _, d in dates.iterrows():
        print(f'    {d["date"]} ({d["granularity"]})')

    # Valuations
    vals = postmark_valuation_df[postmark_valuation_df['postmark_id'] == pm_id]
    print(f'  valuations ({len(vals)}):')
    for _, v in vals.iterrows():
        print(f'    pos {v["appraisal_position"]}: {v["amount"]}')

    # MarkFraming (filter by POSTMARK type to avoid cross-ID-space hits)
    mfs = mark_framing_df[
        (mark_framing_df['parent_mark_type'] == 'POSTMARK') &
        (mark_framing_df['parent_mark_id'] == pm_id)
    ]
    print(f'  framings ({len(mfs)}):')
    for _, mf in mfs.iterrows():
        fname = framings_df.loc[framings_df['framing_id'] == mf['framing_id'], 'name'].iloc[0]
        print(f'    pos {mf["framing_position"]}: {fname}')
    print()


# Inspect first independent entry
inspect_postmark(1)

# Inspect a multi-color fan-out case (if any)
multi_color = postmarks_df[postmarks_df['is_multi_color_fanout']]
if len(multi_color):
    first_mc_src = multi_color['source_listing_idx'].iloc[0]
    siblings = postmarks_df[postmarks_df['source_listing_idx'] == first_mc_src]
    print(f'--- Multi-color siblings from listing idx {first_mc_src} ---')
    for _, sib in siblings.iterrows():
        inspect_postmark(sib['postmark_id'])

# Inspect a manuscript entry (if any)
ms_entries = postmarks_df[postmarks_df['is_manuscript']]
if len(ms_entries):
    inspect_postmark(ms_entries['postmark_id'].iloc[0])

# Inspect an inherited entry (child with parent_postmark_id)
inherited = postmarks_df[postmarks_df['parent_postmark_id'].notna()]
if len(inherited):
    sample = inherited.iloc[0]
    print(f'--- Inherited postmark (child -> parent) ---')
    inspect_postmark(int(sample['postmark_id']))
    print(f'  Parent postmark:')
    inspect_postmark(int(sample['parent_postmark_id']))

=== Postmark 1 ===
  catalog_text:    W City(Washington City)(E)(Feb.27,1797;Ms;Black) . . . 2000.00
  inscription:     W City
  is_manuscript:   True
  shape:           (null)
  color:           BLACK
  dimensions:      nanxnan
  date_format:     None
  impression:      None
  is_irregular:    None
  post_office:     W CITY
  parent_pm_id:    nan
  confidence:      HIGH
  dates (1):
    1797-02-27 (DAY)
  valuations (1):
    pos 1: 2000.0
  framings (0):

--- Multi-color siblings from listing idx 2 ---
=== Postmark 3 ===
  catalog_text:    WASH.CITY(period high)(1799-1801;26.5;FREE[SL-22x5],PAID;Red,Black-brown) 50.00
  inscription:     WASH.CITY
  is_manuscript:   False
  shape:           C
  color:           RED
  dimensions:      26.5x26.5
  date_format:     None
  impression:      Normal
  is_irregular:    False
  post_office:     WASH.CITY
  parent_pm_id:    nan
  confidence:      MEDIUM
  warnings:        ['multi_color_fanout']
  dates (2):
    1799-01-01 (YEAR)
    1801-01-01 (

# Step 9: Ratemark & Auxmark Assembly

Transforms parsed rate tokens from Step 6 into normalized Ratemark, Auxmark,
and PostmarkRatemark domain entities. Parallels Step 8 (Postmark Assembly).

### Token classification

Each rate token from `parsed_rates` is classified by what domain entities it
produces:

| Pattern | Produces | Examples |
|---------|----------|----------|
| Keyword only | Auxmark (parent=Postmark) | `FREE`, `PAID`, `STEAM` |
| Amount only | Ratemark | `5[C]`, `25[ms]`, `V` |
| Keyword + amount | Auxmark (parent=Ratemark) + Ratemark | `PAID/3[C]`, `DUE 3` |
| PM_FREE / PM_FRANK | Auxmark (parent=Postmark) | `P.M.Free`, `P.M.frank` |
| WITH_ADHESIVE | Skipped (Cover attribute, not a marking) | `with 24` |

### Bracket -> Shape/Framing resolution

Rate token brackets describe the physical form of the marking:
- `[C]` -> Circle, `[box]` -> Box, `[arc]` -> Arc, `[octagon]` -> Octagon
- `[ms]` -> manuscript flag (already consumed by Step 6 rate sub-parser)
- Compound brackets (e.g. `[cogged circle]`) decomposed to shape + framing

### Color inheritance

Ratemarks and Auxmarks inherit color from the parent Postmark. The ASCC color
field describes the entire entry's ink color, not individual markings.

### Output DataFrames

- `ratemarks_df` -- one row per Ratemark entity
- `auxmarks_df` -- one row per Auxmark entity
- `postmark_ratemark_df` -- junction linking Postmarks to Ratemarks
- `mark_framing_df` -- extended with Ratemark/Auxmark framing rows

## 9.1 Rate Amount Parsing

Convert raw amount strings to numeric cents values. Handles ASCC fractional
notation (e.g. `12-1/2` = 12.5) and roman numerals.

In [61]:
# --- Step 9.1: Rate amount parsing ---

ROMAN_VALUES = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}


def roman_to_int(s):
    """Convert a roman numeral string to integer. Returns None on failure."""
    if not s or not all(c in ROMAN_VALUES for c in s.upper()):
        return None
    total = 0
    prev = 0
    for ch in reversed(s.upper()):
        val = ROMAN_VALUES[ch]
        if val < prev:
            total -= val
        else:
            total += val
        prev = val
    return total


FRACTION_RE = re.compile(
    r'^(\d+)?'            # optional whole part
    r'[\-\s]*'            # optional separator
    r'(\d+)/(\d+)$'      # fraction
)


def parse_rate_amount(raw):
    """Parse a rate amount string to a float value in cents.
    Returns (numeric_value, is_roman) or (None, False) on failure.
    """
    if raw is None:
        return None, False

    s = str(raw).strip()
    if not s:
        return None, False

    # Roman numeral check
    if re.match(r'^[IVXLCDM]+$', s):
        val = roman_to_int(s)
        if val is not None:
            return float(val), True

    # Fractional: 12-1/2, 6-1/4, 1/2
    m = FRACTION_RE.match(s)
    if m:
        whole = int(m.group(1)) if m.group(1) else 0
        num = int(m.group(2))
        den = int(m.group(3))
        if den > 0:
            return float(whole) + num / den, False

    # Plain integer or decimal
    try:
        return float(s), False
    except ValueError:
        return None, False


# Self-tests
assert parse_rate_amount('5') == (5.0, False)
assert parse_rate_amount('25') == (25.0, False)
assert parse_rate_amount('12-1/2') == (12.5, False)
assert parse_rate_amount('6-1/4') == (6.25, False)
assert parse_rate_amount('1/2') == (0.5, False)
assert parse_rate_amount('V') == (5.0, True)
assert parse_rate_amount('X') == (10.0, True)
assert parse_rate_amount(None) == (None, False)
print('Rate amount parser self-tests passed')

Rate amount parser self-tests passed


## 9.2 Bracket Shape/Framing Resolution

Map bracket descriptors from rate tokens to Shape and Framing seed values.
Compound bracket content (e.g. `cogged circle`) is decomposed into a base
shape and optional framing modifier.

In [62]:
# --- Step 9.2: Bracket -> Shape/Framing resolution ---

# Direct bracket-to-shape mappings (case-insensitive)
BRACKET_SHAPE_MAP = {
    'c': 'C',
    'o': 'O',
    'box': 'BOX',
    'arc': 'ARC',
    'octagon': 'Octagon',
    'sl': 'SL',
    'rectangle': 'BOX',
    'oval': 'O',
    'circle': 'C',
}

# Framing modifiers that may prefix a shape name
BRACKET_FRAMING_MAP = {
    'cogged': 'Cogwheel',
    'dotted': 'Dotted',
    'double': 'DL - Double Line',
    'fancy': 'Fancy',
    'ornate': 'Ornate',
}

# Pattern to extract optional dimensions from bracket content
BRACKET_DIM_RE = re.compile(r'(\d+\.?\d*)(?:\s*x\s*(\d+\.?\d*))?')


def resolve_bracket(bracket_text):
    """Resolve a bracket descriptor into shape/framing/lettering/dimension components.
    Returns dict with keys: shape_name, framing_name, lettering_name, width, height, qualifier.
    """
    if not bracket_text:
        return {'shape_name': None, 'framing_name': None, 'lettering_name': None,
                'width': None, 'height': None, 'qualifier': None}

    text = bracket_text.strip()
    text_lower = text.lower()
    shape_name = None
    framing_name = None
    lettering_name = None
    width = None
    height = None
    qualifier = None

    # Check for framing modifier prefix
    for prefix, framing in BRACKET_FRAMING_MAP.items():
        if text_lower.startswith(prefix):
            framing_name = framing
            remainder = text_lower[len(prefix):].strip()
            # Check if remainder is a shape name
            if remainder in BRACKET_SHAPE_MAP:
                shape_name = BRACKET_SHAPE_MAP[remainder]
            break

    # Direct shape match (if no framing prefix consumed it)
    if shape_name is None:
        # Try the full text as a shape
        if text_lower in BRACKET_SHAPE_MAP:
            shape_name = BRACKET_SHAPE_MAP[text_lower]
        else:
            # Try each word against the shape map
            for word in text_lower.split():
                if word in BRACKET_SHAPE_MAP:
                    shape_name = BRACKET_SHAPE_MAP[word]
                    break

    # Lettering: check full text and individual words against lettering lookup
    if text_lower in lettering_lookup:
        lettering_name = text_lower
    else:
        for word in re.split(r'[\s,]+', text_lower):
            if word in lettering_lookup:
                lettering_name = word
                break

    # Extract dimensions from bracket content
    dim_m = BRACKET_DIM_RE.search(text)
    if dim_m:
        width = float(dim_m.group(1))
        if dim_m.group(2):
            height = float(dim_m.group(2))

    # Anything not recognized as shape/framing/lettering/dimension is a qualifier
    if shape_name is None and framing_name is None and lettering_name is None and width is None:
        qualifier = text

    return {
        'shape_name': shape_name,
        'framing_name': framing_name,
        'lettering_name': lettering_name,
        'width': width,
        'height': height,
        'qualifier': qualifier,
    }


# Self-tests
assert resolve_bracket('C')['shape_name'] == 'C'
assert resolve_bracket('box')['shape_name'] == 'BOX'
assert resolve_bracket('arc')['shape_name'] == 'ARC'
assert resolve_bracket('octagon')['shape_name'] == 'Octagon'
r = resolve_bracket('cogged circle')
assert r['shape_name'] == 'C'
assert r['framing_name'] == 'Cogwheel'
r = resolve_bracket('octagon 23')
assert r['shape_name'] == 'Octagon'
assert r['width'] == 23.0
r = resolve_bracket('30x23 rectangle')
assert r['shape_name'] == 'BOX'
assert r['width'] == 30.0
assert r['height'] == 23.0
assert resolve_bracket('hdstp rate')['qualifier'] == 'hdstp rate'
assert resolve_bracket(None)['shape_name'] is None
r = resolve_bracket('italics')
assert r['lettering_name'] == 'italics'
assert r['qualifier'] is None
r = resolve_bracket('cross hatched letters')
assert r['lettering_name'] is None  # no seed match
assert r['qualifier'] == 'cross hatched letters'
assert resolve_bracket('C')['lettering_name'] is None
print('Bracket resolver self-tests passed')

Bracket resolver self-tests passed


## 9.3 Token Classification & Entity Emission

Walk each listing's rate tokens and emit Ratemark, Auxmark, PostmarkRatemark,
and MarkFraming rows. Classification logic:

1. Tokens with an amount produce a **Ratemark**.
2. Tokens with a keyword (and no amount) produce an **Auxmark** parented to the Postmark.
3. Tokens with both keyword and amount produce a **Ratemark** plus an **Auxmark**
   parented to that Ratemark.

Ratemarks (and their compound Auxmarks) are created once per fan-out postmark
so that each copy carries the correct color. Standalone Auxmarks (keyword only,
no amount) are likewise created once per fan-out postmark.

In [63]:
# --- Step 9.3: Token classification and entity emission ---

ratemark_rows = []
auxmark_rows = []
pm_rm_rows = []
rate_mf_rows = []

next_rm_id = 1
next_aux_id = 1
next_pm_rm_id = 1
# Continue mark_framing_df IDs from Step 8
next_mf_id = len(mark_framing_df) + 1

# Group postmarks by source listing for fan-out handling
pm_by_listing = postmarks_df.groupby('source_listing_idx')['postmark_id'].apply(list).to_dict()

for listing_idx, row in listings.iterrows():
    postmark_ids = pm_by_listing.get(listing_idx, [])
    if not postmark_ids:
        continue

    # Flatten parsed_rates: list of lists -> flat list of tokens
    all_tokens = [tok for field_toks in row['parsed_rates'] for tok in field_toks]

    # Pre-fetch color for each postmark in this listing
    pm_color_map = {}
    for pm_id in postmark_ids:
        pm_color_map[pm_id] = postmarks_df.loc[
            postmarks_df['postmark_id'] == pm_id, 'color_id'
        ].iloc[0]

    for tok in all_tokens:
        kw = tok['rate_keyword']
        amt_raw = tok['rate_amount_raw']
        bracket = tok['rate_bracket']
        is_ms = tok['rate_is_manuscript']
        impression_override = tok.get('rate_impression')

        # Parse amount
        rate_value, is_roman = parse_rate_amount(amt_raw)
        has_amount = rate_value is not None

        # Resolve bracket -> shape/framing/lettering
        br = resolve_bracket(bracket)
        bracket_shape_id = shape_lookup.get(br['shape_name'].upper()) if br['shape_name'] else None
        bracket_framing_name = br['framing_name']
        bracket_lettering_id = lettering_lookup.get(br['lettering_name']) if br['lettering_name'] else None

        # Determine impression
        if is_ms:
            mark_impression = None
        elif impression_override:
            mark_impression = impression_override
        else:
            mark_impression = 'Normal'

        # ── Tokens with an amount: emit Ratemark (+ compound Auxmark) per postmark ──
        if has_amount:
            for pm_id in postmark_ids:
                pm_color = pm_color_map[pm_id]

                # Inscription text
                if is_roman:
                    rm_inscription = amt_raw
                elif kw and amt_raw:
                    rm_inscription = amt_raw
                else:
                    rm_inscription = amt_raw or ''

                rm_id = next_rm_id
                ratemark_rows.append({
                    'ratemark_id': rm_id,
                    'inscription_text': rm_inscription,
                    'rate_value': rate_value,
                    'is_manuscript': is_ms,
                    'shape_id': None if is_ms else bracket_shape_id,
                    'lettering_id': None if is_ms else bracket_lettering_id,
                    'color_id': pm_color,
                    'width': br['width'] if not is_ms else None,
                    'height': br['height'] if not is_ms else None,
                    'is_irregular': None if is_ms else False,
                    'impression': mark_impression,
                    'source_listing_idx': listing_idx,
                    'rate_raw': tok['rate_raw'],
                    'bracket_qualifier': br.get('qualifier'),
                })
                next_rm_id += 1

                # PostmarkRatemark junction
                pm_rm_rows.append({
                    'postmark_ratemark_id': next_pm_rm_id,
                    'postmark_id': pm_id,
                    'ratemark_id': rm_id,
                    'placement_type': None,
                })
                next_pm_rm_id += 1

                # MarkFraming for ratemark (from bracket framing)
                if bracket_framing_name and not is_ms:
                    fid = framing_lookup.get(bracket_framing_name)
                    if fid:
                        rate_mf_rows.append({
                            'mark_framing_id': next_mf_id,
                            'parent_mark_type': 'RATEMARK',
                            'parent_mark_id': rm_id,
                            'framing_id': fid,
                            'framing_position': 1,
                        })
                        next_mf_id += 1

                # Compound auxmark: keyword + amount -> parent to this ratemark
                if kw:
                    aux_inscription = kw
                    if kw in ('PM_FREE', 'PM_FRANK'):
                        aux_inscription = 'FREE' if kw == 'PM_FREE' else 'FRANK'

                    auxmark_rows.append({
                        'auxmark_id': next_aux_id,
                        'inscription_text': aux_inscription,
                        'parent_mark_type': 'RATEMARK',
                        'parent_mark_id': rm_id,
                        'is_manuscript': False,
                        'shape_id': None if is_ms else bracket_shape_id,
                        'lettering_id': None if is_ms else bracket_lettering_id,
                        'color_id': pm_color,
                        'width': None,
                        'height': None,
                        'is_irregular': None if is_ms else False,
                        'impression': mark_impression,
                        'source_listing_idx': listing_idx,
                    })
                    next_aux_id += 1

        # ── Standalone keyword (no amount): emit Auxmark per postmark ──
        elif kw:
            aux_inscription = kw
            if kw in ('PM_FREE', 'PM_FRANK'):
                aux_inscription = 'FREE' if kw == 'PM_FREE' else 'FRANK'

            for pm_id in postmark_ids:
                pm_color = pm_color_map[pm_id]

                auxmark_rows.append({
                    'auxmark_id': next_aux_id,
                    'inscription_text': aux_inscription,
                    'parent_mark_type': 'POSTMARK',
                    'parent_mark_id': pm_id,
                    'is_manuscript': False,
                    'shape_id': bracket_shape_id,
                    'lettering_id': bracket_lettering_id,
                    'color_id': pm_color,
                    'width': br['width'],
                    'height': br['height'],
                    'is_irregular': False,
                    'impression': mark_impression,
                    'source_listing_idx': listing_idx,
                })

                # MarkFraming for this auxmark (from bracket framing)
                if bracket_framing_name:
                    fid = framing_lookup.get(bracket_framing_name)
                    if fid:
                        rate_mf_rows.append({
                            'mark_framing_id': next_mf_id,
                            'parent_mark_type': 'AUXMARK',
                            'parent_mark_id': next_aux_id,
                            'framing_id': fid,
                            'framing_position': 1,
                        })
                        next_mf_id += 1

                next_aux_id += 1

print(f'Token classification complete')
print(f'  Ratemarks emitted: {len(ratemark_rows)}')
print(f'  Auxmarks emitted: {len(auxmark_rows)}')
print(f'  PostmarkRatemark junctions: {len(pm_rm_rows)}')
print(f'  MarkFraming rows (rate/aux): {len(rate_mf_rows)}')

Token classification complete
  Ratemarks emitted: 61
  Auxmarks emitted: 106
  PostmarkRatemark junctions: 61
  MarkFraming rows (rate/aux): 0


## 9.4 DataFrame Construction

Build DataFrames from the emitted rows and extend the existing `mark_framing_df`
with rate/auxmark framing rows.

In [64]:
# --- Step 9.4: DataFrame construction ---

ratemarks_df = pd.DataFrame(ratemark_rows) if ratemark_rows else pd.DataFrame(
    columns=['ratemark_id', 'inscription_text', 'rate_value', 'is_manuscript',
             'shape_id', 'lettering_id', 'color_id', 'width', 'height',
             'is_irregular', 'impression', 'source_listing_idx', 'rate_raw',
             'bracket_qualifier']
)

auxmarks_df = pd.DataFrame(auxmark_rows) if auxmark_rows else pd.DataFrame(
    columns=['auxmark_id', 'inscription_text', 'parent_mark_type',
             'parent_mark_id', 'is_manuscript', 'shape_id', 'lettering_id',
             'color_id', 'width', 'height', 'is_irregular', 'impression',
             'source_listing_idx']
)

postmark_ratemark_df = pd.DataFrame(pm_rm_rows) if pm_rm_rows else pd.DataFrame(
    columns=['postmark_ratemark_id', 'postmark_id', 'ratemark_id',
             'placement_type']
)

# Extend mark_framing_df with rate/aux framing rows
if rate_mf_rows:
    rate_mf_df = pd.DataFrame(rate_mf_rows)
    mark_framing_df = pd.concat([mark_framing_df, rate_mf_df], ignore_index=True)

print('DataFrames constructed:')
print(f'  ratemarks_df:          {len(ratemarks_df)} rows')
print(f'  auxmarks_df:           {len(auxmarks_df)} rows')
print(f'  postmark_ratemark_df:  {len(postmark_ratemark_df)} rows')
print(f'  mark_framing_df:       {len(mark_framing_df)} rows (was {len(mark_framing_df) - len(rate_mf_rows)} before Step 9)')

DataFrames constructed:
  ratemarks_df:          61 rows
  auxmarks_df:           106 rows
  postmark_ratemark_df:  61 rows
  mark_framing_df:       178 rows (was 178 before Step 9)


## 9.5 Value Distributions

In [65]:
# --- Step 9.5: Value distributions ---

if len(ratemarks_df):
    print('Ratemark distributions:')
    print(f'  Total ratemarks: {len(ratemarks_df)}')
    print(f'  Manuscript ratemarks: {ratemarks_df["is_manuscript"].sum()}')
    print(f'  With shape: {ratemarks_df["shape_id"].notna().sum()}')
    print(f'  With bracket qualifier (unresolved): {ratemarks_df["bracket_qualifier"].notna().sum()}')
    print()

    # Rate value distribution
    rate_vals = ratemarks_df['rate_value'].dropna()
    if len(rate_vals):
        print(f'  Rate value range: {rate_vals.min():.1f} - {rate_vals.max():.1f} cents')
        print(f'  Most common values:')
        for val, count in rate_vals.value_counts().head(10).items():
            print(f'    {val:.1f}: {count}')
        print()

    # Shape distribution on ratemarks
    rm_shape_dist = ratemarks_df['shape_id'].value_counts(dropna=False)
    print('  Shape distribution:')
    for sid, count in rm_shape_dist.items():
        if pd.isna(sid):
            print(f'    (null): {count}')
        else:
            name = shapes_df.loc[shapes_df['shape_id'] == sid, 'name'].iloc[0]
            print(f'    {name}: {count}')
    print()

    # Lettering distribution on ratemarks
    rm_lettering_hits = ratemarks_df['lettering_id'].notna().sum()
    print(f'  Lettering assigned: {rm_lettering_hits}')
    if rm_lettering_hits:
        for lid, count in ratemarks_df['lettering_id'].value_counts(dropna=True).items():
            name = letterings_df.loc[letterings_df['lettering_id'] == lid, 'name'].iloc[0]
            print(f'    {name}: {count}')
    print()

if len(auxmarks_df):
    print('Auxmark distributions:')
    print(f'  Total auxmarks: {len(auxmarks_df)}')
    print(f'  Parented to Postmark: {(auxmarks_df["parent_mark_type"] == "Postmark").sum()}')
    print(f'  Parented to Ratemark: {(auxmarks_df["parent_mark_type"] == "Ratemark").sum()}')
    print()

    # Inscription text distribution
    print('  Inscription text distribution:')
    for text, count in auxmarks_df['inscription_text'].value_counts().items():
        print(f'    {text}: {count}')
    print()

    # Lettering distribution on auxmarks
    aux_lettering_hits = auxmarks_df['lettering_id'].notna().sum()
    print(f'  Lettering assigned: {aux_lettering_hits}')
    if aux_lettering_hits:
        for lid, count in auxmarks_df['lettering_id'].value_counts(dropna=True).items():
            name = letterings_df.loc[letterings_df['lettering_id'] == lid, 'name'].iloc[0]
            print(f'    {name}: {count}')
    print()

if len(postmark_ratemark_df):
    print('PostmarkRatemark junction:')
    print(f'  Total rows: {len(postmark_ratemark_df)}')
    print(f'  Distinct postmarks linked: {postmark_ratemark_df["postmark_id"].nunique()}')
    print(f'  Distinct ratemarks linked: {postmark_ratemark_df["ratemark_id"].nunique()}')
    rms_per_pm = postmark_ratemark_df.groupby('postmark_id').size()
    print(f'  Ratemarks per postmark: min={rms_per_pm.min()}, max={rms_per_pm.max()}, mean={rms_per_pm.mean():.1f}')

Ratemark distributions:
  Total ratemarks: 61
  Manuscript ratemarks: 0
  With shape: 26
  With bracket qualifier (unresolved): 0

  Rate value range: 1.0 - 31.0 cents
  Most common values:
    5.0: 18
    3.0: 13
    10.0: 9
    30.0: 6
    2.0: 4
    29.0: 3
    31.0: 3
    1.0: 2
    20.0: 2
    6.0: 1

  Shape distribution:
    (null): 35
    C: 14
    BOX: 8
    ARC: 4

  Lettering assigned: 0

Auxmark distributions:
  Total auxmarks: 106
  Parented to Postmark: 0
  Parented to Ratemark: 0

  Inscription text distribution:
    PAID: 51
    FREE: 46
    DUE: 5
    STEAM: 4

  Lettering assigned: 0

PostmarkRatemark junction:
  Total rows: 61
  Distinct postmarks linked: 37
  Distinct ratemarks linked: 61
  Ratemarks per postmark: min=1, max=5, mean=1.6


## 9.6 FK Integrity Checks

In [66]:
# --- Step 9.6: FK integrity checks ---

print('Step 9 FK integrity checks:')

# Ratemark -> Shape
if len(ratemarks_df):
    orphan = ratemarks_df[
        ratemarks_df['shape_id'].notna() &
        ~ratemarks_df['shape_id'].isin(shapes_df['shape_id'])
    ]
    print(f'  Ratemarks with invalid shape_id: {len(orphan)}')

# Ratemark -> Color
if len(ratemarks_df):
    orphan = ratemarks_df[
        ratemarks_df['color_id'].notna() &
        ~ratemarks_df['color_id'].isin(colors_df['color_id'])
    ]
    print(f'  Ratemarks with invalid color_id: {len(orphan)}')

# Auxmark -> parent (Postmark or Ratemark)
if len(auxmarks_df):
    pm_auxmarks = auxmarks_df[auxmarks_df['parent_mark_type'] == 'Postmark']
    orphan_pm = pm_auxmarks[~pm_auxmarks['parent_mark_id'].isin(postmarks_df['postmark_id'])]
    print(f'  Auxmarks with invalid Postmark parent: {len(orphan_pm)}')

    rm_auxmarks = auxmarks_df[auxmarks_df['parent_mark_type'] == 'Ratemark']
    if len(ratemarks_df):
        orphan_rm = rm_auxmarks[~rm_auxmarks['parent_mark_id'].isin(ratemarks_df['ratemark_id'])]
    else:
        orphan_rm = rm_auxmarks
    print(f'  Auxmarks with invalid Ratemark parent: {len(orphan_rm)}')

# Auxmark -> Shape
if len(auxmarks_df):
    orphan = auxmarks_df[
        auxmarks_df['shape_id'].notna() &
        ~auxmarks_df['shape_id'].isin(shapes_df['shape_id'])
    ]
    print(f'  Auxmarks with invalid shape_id: {len(orphan)}')

# PostmarkRatemark -> Postmark
if len(postmark_ratemark_df):
    orphan = postmark_ratemark_df[
        ~postmark_ratemark_df['postmark_id'].isin(postmarks_df['postmark_id'])
    ]
    print(f'  PostmarkRatemark with invalid postmark_id: {len(orphan)}')

# PostmarkRatemark -> Ratemark
if len(postmark_ratemark_df) and len(ratemarks_df):
    orphan = postmark_ratemark_df[
        ~postmark_ratemark_df['ratemark_id'].isin(ratemarks_df['ratemark_id'])
    ]
    print(f'  PostmarkRatemark with invalid ratemark_id: {len(orphan)}')

# New MarkFraming rows -> parent Ratemark/Auxmark
new_mf = mark_framing_df[mark_framing_df['parent_mark_type'].isin(['Ratemark', 'Auxmark'])]
if len(new_mf):
    rm_mf = new_mf[new_mf['parent_mark_type'] == 'Ratemark']
    if len(rm_mf) and len(ratemarks_df):
        orphan = rm_mf[~rm_mf['parent_mark_id'].isin(ratemarks_df['ratemark_id'])]
        print(f'  MarkFraming(Ratemark) with invalid parent: {len(orphan)}')

    aux_mf = new_mf[new_mf['parent_mark_type'] == 'Auxmark']
    if len(aux_mf) and len(auxmarks_df):
        orphan = aux_mf[~aux_mf['parent_mark_id'].isin(auxmarks_df['auxmark_id'])]
        print(f'  MarkFraming(Auxmark) with invalid parent: {len(orphan)}')

    # Framing FK
    orphan = new_mf[~new_mf['framing_id'].isin(framings_df['framing_id'])]
    print(f'  MarkFraming(Rate/Aux) with invalid framing_id: {len(orphan)}')

Step 9 FK integrity checks:
  Ratemarks with invalid shape_id: 0
  Ratemarks with invalid color_id: 0
  Auxmarks with invalid Postmark parent: 0
  Auxmarks with invalid Ratemark parent: 0
  Auxmarks with invalid shape_id: 0
  PostmarkRatemark with invalid postmark_id: 0
  PostmarkRatemark with invalid ratemark_id: 0


## 9.7 Step 9 Summary

Step 9 transforms the parsed rate tokens from Step 6 into normalized Ratemark,
Auxmark, and PostmarkRatemark domain entities.

Ratemarks are created once per source listing and shared across color fan-out
postmarks via the PostmarkRatemark junction. Auxmarks parented to Ratemarks
are similarly created once. Standalone keyword auxmarks (e.g. PAID, FREE)
are duplicated per fan-out postmark since they directly reference a Postmark.

Key decisions:
- `WITH_ADHESIVE` tokens are not modeled as markings (they are Cover attributes)
- Bracket descriptors are resolved to Shape/Framing seed values where possible
- Color is inherited from the parent Postmark
- `placementType` on PostmarkRatemark is left null (not determinable from catalog text)

In [67]:
# --- Step 9.7: Summary ---

print('=' * 60)
print('Step 9: Ratemark & Auxmark Assembly -- Final Summary')
print('=' * 60)
print()
print(f'Input: {len(listings)} listings with {sum(len(t) for rlist in listings["parsed_rates"] for t in rlist)} rate tokens')
print()
print('New domain entity tables:')
print(f'  ratemarks_df:          {len(ratemarks_df)} rows')
print(f'  auxmarks_df:           {len(auxmarks_df)} rows')
print(f'  postmark_ratemark_df:  {len(postmark_ratemark_df)} rows')
print()
print('Extended tables:')
print(f'  mark_framing_df:       {len(mark_framing_df)} rows (added {len(rate_mf_rows)} rate/aux framings)')
print()

# Listings coverage
listings_with_rates = listings[listings['parsed_rates'].apply(
    lambda rlist: any(len(toks) > 0 for toks in rlist)
)].index
listings_with_ratemarks = set(ratemarks_df['source_listing_idx']) if len(ratemarks_df) else set()
listings_with_auxmarks = set(auxmarks_df['source_listing_idx']) if len(auxmarks_df) else set()
print(f'Listings with rate tokens: {len(listings_with_rates)}')
print(f'Listings producing ratemarks: {len(listings_with_ratemarks)}')
print(f'Listings producing auxmarks: {len(listings_with_auxmarks)}')
print(f'Listings with no rate/aux output: {len(listings) - len(listings_with_ratemarks | listings_with_auxmarks)}')

Step 9: Ratemark & Auxmark Assembly -- Final Summary

Input: 136 listings with 86 rate tokens

New domain entity tables:
  ratemarks_df:          61 rows
  auxmarks_df:           106 rows
  postmark_ratemark_df:  61 rows

Extended tables:
  mark_framing_df:       178 rows (added 0 rate/aux framings)

Listings with rate tokens: 53
Listings producing ratemarks: 20
Listings producing auxmarks: 49
Listings with no rate/aux output: 87


## 9.8 Implied Cover Assembly

Each Postmark in the catalog was observed on at least one physical Cover.
Since the catalog does not describe individual covers in detail, we generate
one **implied Cover** per Postmark and wire them together with CoverPostmark
junction records.

All Cover-level attributes (coverType, colorId, width, height, isInstitutional,
hasAdhesive) are null/default -- the catalog does not provide these.

CoverPostmark:

- **isBackstamp** -- defaults to False (not determinable from catalog text).

Output DataFrames:

- `covers_df` -- one row per implied Cover
- `cover_postmark_df` -- one row per Cover-Postmark junction

In [68]:
# --- Step 9.8: Implied Cover Assembly ---

# Build one Cover per Postmark, one CoverPostmark per pair
cover_rows = []
cp_rows = []
next_cover_id = 1
next_cp_id = 1

for _, pm in postmarks_df.iterrows():
    src = listings.loc[pm['source_listing_idx']]

    # Institutional Ownership: section-level default from catalog
    inst_val = src.get('Institutional Ownership')
    is_institutional = True if pd.notna(inst_val) and str(inst_val).strip() else None

    cover_rows.append({
        'cover_id': next_cover_id,
        'color_id': None,
        'cover_type': None,
        'has_adhesive': False,
        'width': None,
        'height': None,
        'is_institutional': is_institutional,
        'source_listing_idx': pm['source_listing_idx'],
    })

    cp_rows.append({
        'cover_postmark_id': next_cp_id,
        'cover_id': next_cover_id,
        'postmark_id': pm['postmark_id'],
        'is_backstamp': False,
    })

    next_cover_id += 1
    next_cp_id += 1

covers_df = pd.DataFrame(cover_rows)
cover_postmark_df = pd.DataFrame(cp_rows)

print(f'Implied covers created: {len(covers_df)}')
print(f'  Institutional: {covers_df["is_institutional"].notna().sum()}')
print(f'CoverPostmark junctions: {len(cover_postmark_df)}')

Implied covers created: 191
  Institutional: 0
CoverPostmark junctions: 191


## 9.9 Cover FK Integrity & Summary

In [69]:
# --- Step 9.9: Cover FK integrity checks ---

# Every CoverPostmark.cover_id must reference a valid Cover
orphan_cp_cover = cover_postmark_df[
    ~cover_postmark_df['cover_id'].isin(covers_df['cover_id'])
]
print(f'CoverPostmark with invalid cover_id: {len(orphan_cp_cover)}')

# Every CoverPostmark.postmark_id must reference a valid Postmark
orphan_cp_pm = cover_postmark_df[
    ~cover_postmark_df['postmark_id'].isin(postmarks_df['postmark_id'])
]
print(f'CoverPostmark with invalid postmark_id: {len(orphan_cp_pm)}')

# Every Postmark must be referenced by at least one CoverPostmark (model invariant)
linked_pm_ids = set(cover_postmark_df['postmark_id'])
unlinked_pms = postmarks_df[~postmarks_df['postmark_id'].isin(linked_pm_ids)]
print(f'Postmarks with no CoverPostmark (invariant violation): {len(unlinked_pms)}')

# Cover-Postmark uniqueness check
dup_cp = cover_postmark_df.duplicated(subset=['cover_id', 'postmark_id'], keep=False)
print(f'Duplicate cover_id+postmark_id pairs: {dup_cp.sum()}')

print()
print('Cover assembly summary:')
print(f'  covers_df:          {len(covers_df)} rows')
print(f'  cover_postmark_df:  {len(cover_postmark_df)} rows')
print(f'  1:1 cover-to-postmark: {len(covers_df) == len(cover_postmark_df)}')

CoverPostmark with invalid cover_id: 0
CoverPostmark with invalid postmark_id: 0
Postmarks with no CoverPostmark (invariant violation): 0
Duplicate cover_id+postmark_id pairs: 0

Cover assembly summary:
  covers_df:          191 rows
  cover_postmark_df:  191 rows
  1:1 cover-to-postmark: True


## 9.10 Sample Inspection

Spot-check assembled rate/aux entities and implied covers for representative catalog entries.

In [70]:
# --- Step 9.10: Sample inspection ---

def inspect_ratemark(rm_id):
    """Print full assembled state for a single ratemark."""
    rm = ratemarks_df[ratemarks_df['ratemark_id'] == rm_id].iloc[0]
    print(f'  === Ratemark {rm_id} ===')
    print(f'    inscription:   {rm["inscription_text"]}')
    print(f'    rate_value:    {rm["rate_value"]}')
    print(f'    is_manuscript: {rm["is_manuscript"]}')

    shape_name = '(null)'
    if pd.notna(rm.get('shape_id')):
        shape_name = shapes_df.loc[shapes_df['shape_id'] == rm['shape_id'], 'name'].iloc[0]
    print(f'    shape:         {shape_name}')

    color_name = '(null)'
    if pd.notna(rm.get('color_id')):
        color_name = colors_df.loc[colors_df['color_id'] == rm['color_id'], 'name'].iloc[0]
    print(f'    color:         {color_name}')
    print(f'    impression:    {rm["impression"]}')
    print(f'    raw:           {rm["rate_raw"]}')

    # Auxmarks parented to this ratemark
    child_aux = auxmarks_df[
        (auxmarks_df['parent_mark_type'] == 'RATEMARK') &
        (auxmarks_df['parent_mark_id'] == rm_id)
    ]
    if len(child_aux):
        for _, ax in child_aux.iterrows():
            aux_color = '(null)'
            if pd.notna(ax.get('color_id')):
                aux_color = colors_df.loc[colors_df['color_id'] == ax['color_id'], 'name'].iloc[0]
            print(f'    -> Auxmark {ax["auxmark_id"]}: {ax["inscription_text"]} [{aux_color}]')

    # MarkFraming
    mfs = mark_framing_df[
        (mark_framing_df['parent_mark_type'] == 'RATEMARK') &
        (mark_framing_df['parent_mark_id'] == rm_id)
    ]
    if len(mfs):
        for _, mf in mfs.iterrows():
            fname = framings_df.loc[framings_df['framing_id'] == mf['framing_id'], 'name'].iloc[0]
            print(f'    -> Framing: {fname} (pos {mf["framing_position"]})')
    print()


def inspect_postmark_rates(pm_id):
    """Print all rate/aux associations for a postmark."""
    pm = postmarks_df[postmarks_df['postmark_id'] == pm_id].iloc[0]
    color_name = '(null)'
    if pd.notna(pm.get('color_id')):
        color_name = colors_df.loc[colors_df['color_id'] == pm['color_id'], 'name'].iloc[0]
    print(f'Postmark {pm_id}: {pm["catalog_text"][:100]}')
    print(f'  inscription: {pm["inscription_text"]}')
    print(f'  color:       {color_name}')
    print()

    # Linked ratemarks
    links = postmark_ratemark_df[postmark_ratemark_df['postmark_id'] == pm_id]
    if len(links):
        print(f'  Linked ratemarks ({len(links)}):')
        for _, lnk in links.iterrows():
            inspect_ratemark(lnk['ratemark_id'])
    else:
        print('  No linked ratemarks')

    # Direct auxmarks (parented to this postmark)
    direct_aux = auxmarks_df[
        (auxmarks_df['parent_mark_type'] == 'POSTMARK') &
        (auxmarks_df['parent_mark_id'] == pm_id)
    ]
    if len(direct_aux):
        print(f'  Direct auxmarks ({len(direct_aux)}):')
        for _, ax in direct_aux.iterrows():
            shape_name = '(null)'
            if pd.notna(ax.get('shape_id')):
                shape_name = shapes_df.loc[shapes_df['shape_id'] == ax['shape_id'], 'name'].iloc[0]
            aux_color = '(null)'
            if pd.notna(ax.get('color_id')):
                aux_color = colors_df.loc[colors_df['color_id'] == ax['color_id'], 'name'].iloc[0]
            print(f'    Auxmark {ax["auxmark_id"]}: {ax["inscription_text"]} [{shape_name}] [{aux_color}]')
    print()


# Find postmarks with rate tokens to inspect
# 1. A postmark with compound rate tokens (PAID + amount)
if len(postmark_ratemark_df):
    # Pick a postmark that has both ratemarks and direct auxmarks
    pms_with_rm = set(postmark_ratemark_df['postmark_id'])
    pms_with_aux = set(auxmarks_df.loc[
        auxmarks_df['parent_mark_type'] == 'POSTMARK', 'parent_mark_id'
    ]) if len(auxmarks_df) else set()
    both = pms_with_rm & pms_with_aux
    if both:
        sample_pm = sorted(both)[0]
        print('--- Sample: Postmark with both ratemarks and auxmarks ---')
        inspect_postmark_rates(sample_pm)

    # 2. A postmark with only auxmarks (keyword-only tokens)
    aux_only = pms_with_aux - pms_with_rm
    if aux_only:
        sample_pm = sorted(aux_only)[0]
        print('--- Sample: Postmark with auxmarks only ---')
        inspect_postmark_rates(sample_pm)

    # 3. A ratemark with compound bracket (if any exist)
    if len(ratemarks_df):
        bracket_rms = ratemarks_df[ratemarks_df['bracket_qualifier'].notna()]
        if len(bracket_rms):
            print('--- Sample: Ratemark with unresolved bracket qualifier ---')
            inspect_ratemark(bracket_rms['ratemark_id'].iloc[0])

# 4. Implied cover inspection
print()
print('--- Sample: Implied Cover -> Postmark chain ---')
# Pick a postmark with both ratemarks and auxmarks for a rich example
if len(postmark_ratemark_df):
    pms_with_rm_ = set(postmark_ratemark_df['postmark_id'])
    pms_with_aux_ = set(auxmarks_df.loc[
        auxmarks_df['parent_mark_type'] == 'POSTMARK', 'parent_mark_id'
    ]) if len(auxmarks_df) else set()
    both_ = pms_with_rm_ & pms_with_aux_
    sample_pm_id = sorted(both_)[0] if both_ else postmarks_df['postmark_id'].iloc[0]
else:
    sample_pm_id = postmarks_df['postmark_id'].iloc[0]

cp_link = cover_postmark_df[cover_postmark_df['postmark_id'] == sample_pm_id].iloc[0]
cover = covers_df[covers_df['cover_id'] == cp_link['cover_id']].iloc[0]
pm = postmarks_df[postmarks_df['postmark_id'] == sample_pm_id].iloc[0]

print(f'Cover {cover["cover_id"]}:')
print(f'  cover_type:     {cover["cover_type"]}')
print(f'  -> CoverPostmark {cp_link["cover_postmark_id"]}:')
print(f'     postmark_id:  {cp_link["postmark_id"]}')
print(f'     is_backstamp: {cp_link["is_backstamp"]}')
print(f'  -> Postmark: {pm["catalog_text"][:80]}')

--- Sample: Postmark with both ratemarks and auxmarks ---
Postmark 49: Same(3mm letters)(1845-48;29,30,31;PAID,5,10;Red,Orange,Red-orange) . . . . 20.00
  inscription: WASHINGTON CITY D.C.
  color:       RED

  Linked ratemarks (5):
  === Ratemark 1 ===
    inscription:   29
    rate_value:    29.0
    is_manuscript: False
    shape:         (null)
    color:         RED
    impression:    Normal
    raw:           29

  === Ratemark 4 ===
    inscription:   30
    rate_value:    30.0
    is_manuscript: False
    shape:         (null)
    color:         RED
    impression:    Normal
    raw:           30

  === Ratemark 7 ===
    inscription:   31
    rate_value:    31.0
    is_manuscript: False
    shape:         (null)
    color:         RED
    impression:    Normal
    raw:           31

  === Ratemark 10 ===
    inscription:   5
    rate_value:    5.0
    is_manuscript: False
    shape:         (null)
    color:         RED
    impression:    Normal
    raw:           5

  === Rat

# Step 10: Output

Write all domain entity and seed tables to `./wip/out/` as CSV.
Intermediate/working DataFrames (`field_df`, `fanout_df`, `rate_mf_df`) are excluded.

In [71]:
import os

OUT_DIR = './wip/out/'
os.makedirs(OUT_DIR, exist_ok=True)

domain_tables = {
    # Seed / value tables
    'shapes':              shapes_df,
    'framings':            framings_df,
    'letterings':          letterings_df,
    'colors':              colors_df,

    # Core domain entities
    'postmarks':           postmarks_df,
    'ratemarks':           ratemarks_df,
    'auxmarks':            auxmarks_df,
    'covers':              covers_df,

    # Junction / association tables
    'cover_postmark':      cover_postmark_df,
    'postmark_ratemark':   postmark_ratemark_df,
    'mark_framing':        mark_framing_df,

    # Dependent domain entities
    'date_observed':       date_observed_df,
    'postmark_valuation':  postmark_valuation_df,
    'post_offices':        post_offices_df,
}

# Map source_listing_idx -> source_page for tables that carry provenance
page_lookup = listings['Page']

# Coerce nullable-int columns (IDs, positions) so they don't write as floats
INT_SUFFIXES = ('_id', '_position', '_idx')

for name, frame in domain_tables.items():
    out = frame.copy()
    if 'source_listing_idx' in out.columns:
        out['source_page'] = out['source_listing_idx'].map(page_lookup)
        out.drop(columns=['source_listing_idx'], inplace=True)
    for col in out.columns:
        if col.endswith(INT_SUFFIXES) and out[col].dtype == 'float64':
            out[col] = out[col].astype('Int64')
    path = os.path.join(OUT_DIR, f'{name}.csv')
    out.to_csv(path, index=False)
    print(f'  {name + ".csv":<28s} {len(out):>5d} rows  ->  {path}')

print()
print(f'Wrote {len(domain_tables)} tables to {OUT_DIR}')

  shapes.csv                       9 rows  ->  ./wip/out/shapes.csv
  framings.csv                     9 rows  ->  ./wip/out/framings.csv
  letterings.csv                  14 rows  ->  ./wip/out/letterings.csv
  colors.csv                      16 rows  ->  ./wip/out/colors.csv
  postmarks.csv                  191 rows  ->  ./wip/out/postmarks.csv
  ratemarks.csv                   61 rows  ->  ./wip/out/ratemarks.csv
  auxmarks.csv                   106 rows  ->  ./wip/out/auxmarks.csv
  covers.csv                     191 rows  ->  ./wip/out/covers.csv
  cover_postmark.csv             191 rows  ->  ./wip/out/cover_postmark.csv
  postmark_ratemark.csv           61 rows  ->  ./wip/out/postmark_ratemark.csv
  mark_framing.csv               178 rows  ->  ./wip/out/mark_framing.csv
  date_observed.csv              301 rows  ->  ./wip/out/date_observed.csv
  postmark_valuation.csv         179 rows  ->  ./wip/out/postmark_valuation.csv
  post_offices.csv                44 rows  ->  ./wip/out/p